# End-to-End Semantic-Guided SAM2 Checkpoint

## Purpose

This checkpoint is a **decision experiment** between the completed Group E simple-fusion screening and further pipeline development.

It tests whether the semantic reference-point strategies developed in Groups A–E translate into meaningful improvements in actual SAM2-based object counting.

### Main objectives

- **Validate the A–E direction**
  - Test whether improved semantic reference points lead to improved SAM2 masks and final counts.
  - Compare all guided variants against B0 on the same frozen FSC-147 dev100 subset.

- **Test density-dependent behaviour**
  - Evaluate every system overall and separately on S1–S6.
  - Test whether:
    - F2 retains its expected advantage in sparse scenes;
    - F1 retains its expected recovery advantage in dense scenes.
  - S1–S6 are **evaluation strata only** and are not used for inference-time routing.

- **Measure breadth of improvement**
  - Determine whether gains occur across many images or are dominated by a small number of high-count images.

- **Identify the bottleneck**
  - Distinguish failures in semantic reference-point generation from failures in the reference-point → SAM2 mask → count stage.

- **Guide the next architecture decision**
  - Continue/refine semantic integration if guidance is useful.
  - Investigate GT-free combination/adaptation if F1/F2 show complementary regimes.
  - Reconsider the semantic-to-SAM2 interface or wider architecture if guidance provides little benefit.

---

## Frozen systems

All systems will be evaluated on **all 100 dev images**.

1. **B0-dev100**
   - Unguided SAM2 baseline.

2. **Overall representative**
   - F2 / C2b / D6B-T-native / τ = 0.010

3. **Sparse diagnostic representative**
   - F2 / C2b / D6B-T-native / τ = 0.005

4. **Dense-recovery diagnostic representative**
   - F1 / C1 / D6B9-strict / τ = 0.050

No further C/D/F1/F2 parameter tuning is performed in this checkpoint.

No Marigold/geometric prior is used.

---

## Evaluation

### Absolute counting performance
- MAE
- RMSE
- Mean signed error / counting bias

### Image-normalised performance
- Mean and median relative absolute error
- Exact-count rate
- Percentage of images within:
  - 1%
  - 5%
  - 10%
  - 20% relative count error

### Breadth of improvement relative to B0
For each guided system:
- number/percentage of images improved;
- unchanged;
- worsened;
- paired change in absolute count error.

### Density-stratified evaluation
Report the relevant metrics separately for:
- S1
- S2
- S3
- S4
- S5
- S6

### Pipeline diagnostics
Track, where applicable:
- semantic reference-point count;
- SAM2 masks generated;
- masks retained;
- final predicted count;
- duplicate, merged, missed and non-target-mask failure patterns.

---

## Questions to answer

**Q1 — End-to-end validity**  
Does semantic-guided SAM2 materially improve counting over B0-dev100?

**Q2 — Density dependence**  
Where across S1–S6 does each semantic strategy improve or degrade performance?

**Q3 — Pipeline bottleneck**  
When a guided system fails, is the main limitation the semantic reference points or their conversion into SAM2 masks/counts?

**Q4 — Breadth and robustness**  
Are improvements broad across images, or driven mainly by a small number of high-count images?

**Q5 — Proxy-to-end-to-end consistency**  
Do the improvements observed during the Group E reference-point experiments predict improvements in final SAM2 counting?

---

## Experimental constraints

- Same frozen FSC-147 dev100 images for all systems.
- Same GT annotations and density strata.
- Guided configurations are frozen before the checkpoint.
- No GT information is used during inference.
- No density-based routing.
- No configuration-specific SAM2 tuning.
- No Marigold.
- This is a **development checkpoint**, not the final held-out evaluation.

The results will determine whether Group E should resume with structured semantic integration, be modified based on observed failure modes, or whether the wider architecture should be reconsidered.

1. Setup and project paths
2. Load + validate frozen checkpoint artefacts
   2.1 dev100
   2.2 B0-dev100
   2.3 Group-E shortlist
   2.4 Group-E reference points
   2.5 cross-source 100-image identity validation

3. Build FSC-147 validation dataset
   - validation only
   - test untouched
   - recover canonical GT points

4. Initialise SAM2.1 Hiera-Small
   - SAM2ImagePredictor
   - same model/checkpoint as B1/B0
   - single positive point
   - multimask_output=False
   - retain_all

5. Validate the three frozen prompt sets
   - exactly 100 rows/system
   - pred_count == len(pred_points_xy)
   - image IDs/category match dev100
   - coordinates inside image bounds
   - no density routing

6. Guided SAM2 inference
   - set image once
   - run each semantic point independently
   - capture mask + SAM2 score
   - no GT used here
   - no filtering/tuning

7. Point→mask diagnostics
   - one/zero/multiple GT points per mask
   - duplicate GT coverage
   - missed GT points
   - prompt→mask consistency

8. Counting evaluation
   - overall
   - S1–S6
   - relative-error metrics
   - paired change vs B0

9. Breadth / concentration analysis
   - improved / same / worsened images
   - contribution of S1–S6
   - top-k image contribution

10. Proxy ↔ end-to-end diagnostic comparison
    - Group-E point-level F1/MAE
    - SAM2 mask correspondence
    - B0 comparison

11. Freeze all checkpoint artefacts
12. Final validation and interpretation

# 1. Setup

In [1]:
# 1.0 - mount google drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 1.1 - Define Project / Dataset / Checkpoint Paths
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MSc_Project_ZeroShot_Counting"
)

DATA_ROOT = PROJECT_ROOT / "datasets"
RESULTS_ROOT = PROJECT_ROOT / "results"


# ------------------------------------------------------------
# FSC-147
# ------------------------------------------------------------

FSC147_ROOT = DATA_ROOT / "FSC147"
FSC147_RAW = FSC147_ROOT / "raw"

FSC147_IMAGE_DIR = (
    FSC147_RAW
    / "images_384_VarV2"
)

FSC147_DENSITY_DIR = (
    FSC147_RAW
    / "gt_density_map_adaptive_384_VarV2"
)

FSC147_ANNOTATION_FILE = (
    FSC147_RAW
    / "annotation_FSC147_384.json"
)

FSC147_SPLIT_FILE = (
    FSC147_RAW
    / "Train_Test_Val_FSC_147.json"
)

FSC147_CLASSES_FILE = (
    FSC147_RAW
    / "ImageClasses_FSC147.txt"
)


# ------------------------------------------------------------
# Frozen dev100
# ------------------------------------------------------------

DEV100_CSV = (
    RESULTS_ROOT
    / "B1_experiments"
    / "subsets"
    / "FSC147_val_dev_100.csv"
)


# ------------------------------------------------------------
# Frozen B0-dev100 control
# ------------------------------------------------------------

B0_DEV100_RESULTS_DIR = (
    RESULTS_ROOT
    / "B0_dev100"
)


# ------------------------------------------------------------
# Frozen Group-E semantic integration
# ------------------------------------------------------------

E3_RESULTS_DIR = (
    RESULTS_ROOT
    / "B1_experiments"
    / "Group_E"
    / "E3_semantic_integration"
)

E3_FUSION_GEN_DIR = (
    E3_RESULTS_DIR
    / "fusion_generation"
)

E3_FINAL_DIR = (
    E3_RESULTS_DIR
    / "final"
)


# ------------------------------------------------------------
# End-to-end checkpoint output
# ------------------------------------------------------------

E2E_RESULTS_DIR = (
    RESULTS_ROOT
    / "B1_experiments"
    / "End_to_End_Checkpoint"
)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("PROJECT_ROOT        :", PROJECT_ROOT)
print("RESULTS_ROOT        :", RESULTS_ROOT)

print("\nFSC-147:")
print("FSC147_ROOT         :", FSC147_ROOT)
print("FSC147_RAW          :", FSC147_RAW)

print("\nFrozen inputs:")
print("DEV100_CSV          :", DEV100_CSV)
print("B0_DEV100_RESULTS   :", B0_DEV100_RESULTS_DIR)
print("E3_RESULTS_DIR      :", E3_RESULTS_DIR)

print("\nCheckpoint output:")
print("E2E_RESULTS_DIR     :", E2E_RESULTS_DIR)

PROJECT_ROOT        : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting
RESULTS_ROOT        : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results

FSC-147:
FSC147_ROOT         : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/FSC147
FSC147_RAW          : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/FSC147/raw

Frozen inputs:
DEV100_CSV          : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/subsets/FSC147_val_dev_100.csv
B0_DEV100_RESULTS   : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B0_dev100
E3_RESULTS_DIR      : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Group_E/E3_semantic_integration

Checkpoint output:
E2E_RESULTS_DIR     : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint


In [3]:
# ============================================================
# Step 1.2 - Clone Project Repository
# ============================================================

%cd /content

!rm -rf Surrey-MScProject-Zero-Shot-Object-Counting

!git clone \
    https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git

/content
Cloning into 'Surrey-MScProject-Zero-Shot-Object-Counting'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 47 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 65.52 KiB | 645.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.


In [4]:
# ============================================================
# Step 1.3 - Import Project Dataset Code
# ============================================================

import sys


repo_path = (
    "/content/"
    "Surrey-MScProject-Zero-Shot-Object-Counting"
)

if repo_path not in sys.path:
    sys.path.append(repo_path)


from src.datasets import FSC147Dataset


print("Project imports: PASS")
print("FSC147Dataset available: PASS")

Project imports: PASS
FSC147Dataset available: PASS


In [5]:
# ============================================================
# Step 2.1 - Load and Validate Frozen dev100 + B0 Control
# ============================================================

import json
import pandas as pd


# ------------------------------------------------------------
# Frozen input files
# ------------------------------------------------------------

B0_PER_IMAGE_FILE = (
    B0_DEV100_RESULTS_DIR
    / "B0_dev100_per_image.csv"
)

B0_OVERALL_FILE = (
    B0_DEV100_RESULTS_DIR
    / "B0_dev100_overall_metrics.csv"
)

B0_DENSITY_FILE = (
    B0_DEV100_RESULTS_DIR
    / "B0_dev100_density_metrics.csv"
)

B0_METADATA_FILE = (
    B0_DEV100_RESULTS_DIR
    / "B0_dev100_metadata.json"
)


# ------------------------------------------------------------
# File existence checks
# ------------------------------------------------------------

required_files = [
    DEV100_CSV,
    B0_PER_IMAGE_FILE,
    B0_OVERALL_FILE,
    B0_DENSITY_FILE,
    B0_METADATA_FILE,
]

for path in required_files:
    assert path.exists(), (
        f"Required frozen input not found:\n{path}"
    )


# ------------------------------------------------------------
# Load authoritative dev100
# ------------------------------------------------------------

fsc147_dev_100_df = pd.read_csv(
    DEV100_CSV
)

dev100_required_cols = {
    "dataset_index",
    "image_id",
    "category",
    "gt_count",
    "density_stratum",
}

missing_dev100_cols = (
    dev100_required_cols
    - set(fsc147_dev_100_df.columns)
)

assert not missing_dev100_cols, (
    "dev100 missing required columns: "
    f"{sorted(missing_dev100_cols)}"
)


# ------------------------------------------------------------
# Load frozen B0 control
# ------------------------------------------------------------

b0_per_image_df = pd.read_csv(
    B0_PER_IMAGE_FILE
)

b0_overall_df = pd.read_csv(
    B0_OVERALL_FILE
)

b0_density_df = pd.read_csv(
    B0_DENSITY_FILE
)

with open(
    B0_METADATA_FILE,
    "r"
) as f:
    b0_metadata = json.load(f)


# ------------------------------------------------------------
# Validate dev100 integrity
# ------------------------------------------------------------

assert len(fsc147_dev_100_df) == 100

assert (
    fsc147_dev_100_df["dataset_index"].is_unique
)

assert (
    fsc147_dev_100_df["image_id"].is_unique
)


expected_stratum_sizes = {
    "S1_0_20": 20,
    "S2_20_40": 20,
    "S3_40_60": 20,
    "S4_60_80": 20,
    "S5_80_95": 10,
    "S6_95_100": 10,
}

actual_stratum_sizes = (
    fsc147_dev_100_df[
        "density_stratum"
    ]
    .value_counts()
    .to_dict()
)

assert (
    actual_stratum_sizes
    == expected_stratum_sizes
), (
    "Unexpected dev100 density distribution:\n"
    f"{actual_stratum_sizes}"
)


# ------------------------------------------------------------
# Validate B0 per-image schema
# ------------------------------------------------------------

b0_required_cols = {
    "dataset_index",
    "image_id",
    "category",
    "gt_count",
    "density_stratum",
    "num_raw_masks",
    "num_selected_masks",
    "pred_count",
    "signed_error",
    "abs_error",
    "squared_error",
    "relative_error",
    "error_direction",
}

missing_b0_cols = (
    b0_required_cols
    - set(b0_per_image_df.columns)
)

assert not missing_b0_cols, (
    "B0 per-image table missing columns: "
    f"{sorted(missing_b0_cols)}"
)


# ------------------------------------------------------------
# Validate B0 image universe
# ------------------------------------------------------------

assert len(b0_per_image_df) == 100

assert (
    b0_per_image_df["dataset_index"].is_unique
)

assert (
    b0_per_image_df["image_id"].is_unique
)


# ------------------------------------------------------------
# Exact dev100 ↔ B0 mapping validation
# ------------------------------------------------------------

for col in [
    "dataset_index",
    "image_id",
    "category",
    "gt_count",
    "density_stratum",
]:

    if col in [
        "image_id",
        "category",
        "density_stratum",
    ]:

        lhs = (
            b0_per_image_df[col]
            .astype(str)
            .tolist()
        )

        rhs = (
            fsc147_dev_100_df[col]
            .astype(str)
            .tolist()
        )

    else:

        lhs = (
            b0_per_image_df[col]
            .astype(int)
            .tolist()
        )

        rhs = (
            fsc147_dev_100_df[col]
            .astype(int)
            .tolist()
        )

    assert lhs == rhs, (
        f"dev100 ↔ B0 mismatch in column: {col}"
    )


# ------------------------------------------------------------
# Validate frozen B0 counting rule
# ------------------------------------------------------------

assert (
    b0_per_image_df["num_raw_masks"]
    ==
    b0_per_image_df["num_selected_masks"]
).all()

assert (
    b0_per_image_df["num_selected_masks"]
    ==
    b0_per_image_df["pred_count"]
).all()


# ------------------------------------------------------------
# Validate B0 summary tables
# ------------------------------------------------------------

assert len(b0_overall_df) == 1

assert (
    int(
        b0_overall_df.iloc[0][
            "num_samples"
        ]
    )
    == 100
)

assert len(b0_density_df) == 6

assert (
    b0_density_df["num_images"]
    .astype(int)
    .tolist()
    ==
    [20, 20, 20, 20, 10, 10]
)


# ------------------------------------------------------------
# Validate frozen B0 metadata
# ------------------------------------------------------------

assert b0_metadata["dataset"] == "FSC-147"
assert b0_metadata["split"] == "val"
assert b0_metadata["num_images"] == 100

assert (
    b0_metadata[
        "additional_mask_filtering"
    ]
    is False
)

assert (
    b0_metadata[
        "test_set_accessed"
    ]
    is False
)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("=" * 100)
print("STEP 2.1 - FROZEN DEV100 + B0 CONTROL")
print("=" * 100)

print(
    f"dev100 images              : "
    f"{len(fsc147_dev_100_df)}"
)

print(
    f"B0 per-image rows          : "
    f"{len(b0_per_image_df)}"
)

print(
    "dev100 ↔ B0 mapping        : PASS"
)

print(
    "B0 frozen counting rule    : PASS"
)

print(
    "B0 overall summary         : PASS"
)

print(
    "B0 S1-S6 summary           : PASS"
)

print(
    "Validation split only      : PASS"
)

print(
    "Test set accessed          : NO"
)

print("\nFrozen B0 overall metrics:")

display(
    b0_overall_df.round(4)
)

print("\nFrozen dev100 density distribution:")

display(
    fsc147_dev_100_df[
        "density_stratum"
    ]
    .value_counts()
    .reindex(
        expected_stratum_sizes.keys()
    )
    .rename("num_images")
    .to_frame()
)

print("=" * 100)

STEP 2.1 - FROZEN DEV100 + B0 CONTROL
dev100 images              : 100
B0 per-image rows          : 100
dev100 ↔ B0 mapping        : PASS
B0 frozen counting rule    : PASS
B0 overall summary         : PASS
B0 S1-S6 summary           : PASS
Validation split only      : PASS
Test set accessed          : NO

Frozen B0 overall metrics:


,num_samples,mae,rmse,mean_signed_error,num_undercount,num_exact,num_overcount,mean_relative_abs_error,median_relative_abs_error,exact_count_rate,within_1pct,within_5pct,within_10pct,within_20pct
0,100,80.69,270.2339,-77.47,76,4,20,0.5593,0.6579,4.0,4.0,7.0,13.0,22.0



Frozen dev100 density distribution:


,num_images
density_stratum,
S1_0_20,20
S2_20_40,20
S3_40_60,20
S4_60_80,20
S5_80_95,10
S6_95_100,10


In [6]:
# ============================================================
# Step 2.2 - Load and Validate Frozen Group-E Representatives
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Frozen Group-E input files
# ------------------------------------------------------------

E3_SHORTLIST_FILE = (
    E3_FINAL_DIR
    / "E3_diagnostic_shortlist.csv"
)

E3_FUSION_PER_IMAGE_FILE = (
    E3_FUSION_GEN_DIR
    / "E3_fusion_per_image.csv"
)


# ------------------------------------------------------------
# File existence checks
# ------------------------------------------------------------

for path in [
    E3_SHORTLIST_FILE,
    E3_FUSION_PER_IMAGE_FILE,
]:
    assert path.exists(), (
        f"Required frozen Group-E input not found:\n{path}"
    )


# ------------------------------------------------------------
# Load frozen Group-E artefacts
# ------------------------------------------------------------

e3_shortlist_df = pd.read_csv(
    E3_SHORTLIST_FILE
)

e3_fusion_per_image_df = pd.read_csv(
    E3_FUSION_PER_IMAGE_FILE
)


# ------------------------------------------------------------
# Validate shortlist schema
# ------------------------------------------------------------

shortlist_required_cols = {
    "checkpoint_id",
    "fusion_family",
    "c_variant",
    "d_variant",
    "tau",
}

missing_shortlist_cols = (
    shortlist_required_cols
    - set(e3_shortlist_df.columns)
)

assert not missing_shortlist_cols, (
    "E3 shortlist missing required columns: "
    f"{sorted(missing_shortlist_cols)}"
)


# ------------------------------------------------------------
# Validate fusion-table schema
# ------------------------------------------------------------

fusion_required_cols = {
    "dataset_index",
    "image_id",
    "category",
    "c_variant",
    "d_variant",
    "tau",
    "fusion_family",
    "pred_count",
    "pred_points_xy",
}

missing_fusion_cols = (
    fusion_required_cols
    - set(e3_fusion_per_image_df.columns)
)

assert not missing_fusion_cols, (
    "E3 fusion table missing required columns: "
    f"{sorted(missing_fusion_cols)}"
)


# ------------------------------------------------------------
# Expected frozen diagnostic representatives
# ------------------------------------------------------------

expected_representatives = pd.DataFrame(
    [
        {
            "checkpoint_id": "overall_joint",
            "fusion_family": "F2",
            "c_variant": "C2b",
            "d_variant": "D6B-T-native",
            "tau": 0.010,
        },
        {
            "checkpoint_id": "sparse_joint",
            "fusion_family": "F2",
            "c_variant": "C2b",
            "d_variant": "D6B-T-native",
            "tau": 0.005,
        },
        {
            "checkpoint_id": "dense_recovery",
            "fusion_family": "F1",
            "c_variant": "C1",
            "d_variant": "D6B9-strict",
            "tau": 0.050,
        },
    ]
)


# ------------------------------------------------------------
# Validate frozen shortlist exactly
# ------------------------------------------------------------

assert len(e3_shortlist_df) == 3

assert set(
    e3_shortlist_df["checkpoint_id"]
) == set(
    expected_representatives["checkpoint_id"]
)


for expected in expected_representatives.itertuples(
    index=False
):

    row = e3_shortlist_df[
        e3_shortlist_df["checkpoint_id"]
        == expected.checkpoint_id
    ]

    assert len(row) == 1

    row = row.iloc[0]

    assert (
        row["fusion_family"]
        == expected.fusion_family
    )

    assert (
        row["c_variant"]
        == expected.c_variant
    )

    assert (
        row["d_variant"]
        == expected.d_variant
    )

    assert np.isclose(
        float(row["tau"]),
        float(expected.tau),
    )


# ------------------------------------------------------------
# Extract exactly the three frozen systems
# ------------------------------------------------------------

selected_frames = []

for cfg in expected_representatives.itertuples(
    index=False
):

    mask = (
        (
            e3_fusion_per_image_df[
                "fusion_family"
            ]
            == cfg.fusion_family
        )
        &
        (
            e3_fusion_per_image_df[
                "c_variant"
            ]
            == cfg.c_variant
        )
        &
        (
            e3_fusion_per_image_df[
                "d_variant"
            ]
            == cfg.d_variant
        )
        &
        np.isclose(
            e3_fusion_per_image_df[
                "tau"
            ].astype(float),
            float(cfg.tau),
        )
    )

    system_df = (
        e3_fusion_per_image_df[
            mask
        ]
        .copy()
    )

    assert len(system_df) == 100, (
        f"{cfg.checkpoint_id}: "
        f"expected 100 rows, "
        f"found {len(system_df)}"
    )

    system_df.insert(
        0,
        "checkpoint_id",
        cfg.checkpoint_id,
    )

    selected_frames.append(
        system_df
    )


e3_checkpoint_points_df = pd.concat(
    selected_frames,
    ignore_index=True,
)


# ------------------------------------------------------------
# Validate 3 × 100 checkpoint universe
# ------------------------------------------------------------

assert len(e3_checkpoint_points_df) == 300

checkpoint_counts = (
    e3_checkpoint_points_df[
        "checkpoint_id"
    ]
    .value_counts()
    .reindex(
        [
            "overall_joint",
            "sparse_joint",
            "dense_recovery",
        ]
    )
)

assert (
    checkpoint_counts.tolist()
    ==
    [100, 100, 100]
)


# ------------------------------------------------------------
# Validate each system against authoritative dev100
# ------------------------------------------------------------

for checkpoint_id in [
    "overall_joint",
    "sparse_joint",
    "dense_recovery",
]:

    system_df = (
        e3_checkpoint_points_df[
            e3_checkpoint_points_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
        .copy()
    )

    # Align using authoritative dev100 order
    system_df = (
        fsc147_dev_100_df[
            [
                "dataset_index",
                "image_id",
                "category",
            ]
        ]
        .merge(
            system_df,
            on=[
                "dataset_index",
                "image_id",
                "category",
            ],
            how="left",
            validate="one_to_one",
        )
    )

    assert len(system_df) == 100

    assert (
        system_df["checkpoint_id"]
        .notna()
        .all()
    ), (
        f"{checkpoint_id}: "
        "missing one or more dev100 images"
    )

    assert (
        system_df["dataset_index"]
        .astype(int)
        .tolist()
        ==
        fsc147_dev_100_df[
            "dataset_index"
        ]
        .astype(int)
        .tolist()
    )

    assert (
        system_df["image_id"]
        .astype(str)
        .tolist()
        ==
        fsc147_dev_100_df[
            "image_id"
        ]
        .astype(str)
        .tolist()
    )


# ------------------------------------------------------------
# No inference-time density routing
# ------------------------------------------------------------

# Every representative must cover all 100 images.
assert (
    checkpoint_counts == 100
).all()


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("=" * 110)
print("STEP 2.2 - FROZEN GROUP-E DIAGNOSTIC REPRESENTATIVES")
print("=" * 110)

print(
    f"Full E3 fusion table rows      : "
    f"{len(e3_fusion_per_image_df)}"
)

print(
    f"Frozen shortlist rows          : "
    f"{len(e3_shortlist_df)}"
)

print(
    f"Selected checkpoint rows       : "
    f"{len(e3_checkpoint_points_df)}"
)

print(
    "Expected total                : 3 × 100 = 300"
)

print(
    "dev100 image universe         : PASS"
)

print(
    "No density routing            : PASS"
)

print(
    "Frozen configurations         : PASS"
)

print("\nFrozen diagnostic shortlist:")

display(
    e3_shortlist_df[
        [
            "checkpoint_id",
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",
            "mae",
            "rmse",
            "macro_f1",
        ]
    ].round(4)
)

print("\nRows per checkpoint:")

display(
    checkpoint_counts
    .rename("num_images")
    .to_frame()
)

print("=" * 110)

STEP 2.2 - FROZEN GROUP-E DIAGNOSTIC REPRESENTATIVES
Full E3 fusion table rows      : 22500
Frozen shortlist rows          : 3
Selected checkpoint rows       : 300
Expected total                : 3 × 100 = 300
dev100 image universe         : PASS
No density routing            : PASS
Frozen configurations         : PASS

Frozen diagnostic shortlist:


,checkpoint_id,fusion_family,c_variant,d_variant,tau,mae,rmse,macro_f1
0,overall_joint,F2,C2b,D6B-T-native,0.010,88.17,262.7308,0.3182
1,sparse_joint,F2,C2b,D6B-T-native,0.005,86.42,264.0067,0.3135
2,dense_recovery,F1,C1,D6B9-strict,0.050,86.43,259.0033,0.3562



Rows per checkpoint:


,num_images
checkpoint_id,
overall_joint,100
sparse_joint,100
dense_recovery,100


In [7]:
# ============================================================
# Step 2.3 - Parse and Validate Frozen Semantic Reference Points
# ============================================================

import ast
import numpy as np


# ------------------------------------------------------------
# Point-list parser
# ------------------------------------------------------------

def parse_point_list(value):

    if isinstance(value, str):
        value = ast.literal_eval(value)

    if isinstance(value, list):
        points = value

    elif value is None:
        points = []

    elif (
        isinstance(value, float)
        and np.isnan(value)
    ):
        points = []

    else:
        raise TypeError(
            "Unexpected pred_points_xy type: "
            f"{type(value)}"
        )

    arr = np.asarray(
        points,
        dtype=np.float32,
    )

    if arr.size == 0:
        return np.empty(
            (0, 2),
            dtype=np.float32,
        )

    return arr.reshape(-1, 2)


# ------------------------------------------------------------
# Parse the 300 frozen reference-point lists
# ------------------------------------------------------------

e3_checkpoint_points_df[
    "pred_points_parsed"
] = (
    e3_checkpoint_points_df[
        "pred_points_xy"
    ]
    .apply(parse_point_list)
)


# ------------------------------------------------------------
# Count parsed reference points
# ------------------------------------------------------------

e3_checkpoint_points_df[
    "num_reference_points"
] = (
    e3_checkpoint_points_df[
        "pred_points_parsed"
    ]
    .apply(len)
    .astype(int)
)


# ------------------------------------------------------------
# Point-count consistency
# ------------------------------------------------------------

assert np.array_equal(
    e3_checkpoint_points_df[
        "num_reference_points"
    ].to_numpy(),
    e3_checkpoint_points_df[
        "pred_count"
    ]
    .astype(int)
    .to_numpy(),
), (
    "Frozen Group-E pred_count does not match "
    "the number of parsed reference points."
)


# ------------------------------------------------------------
# Validate point-array structure
# ------------------------------------------------------------

for row in (
    e3_checkpoint_points_df
    .itertuples(index=False)
):

    points = row.pred_points_parsed

    assert isinstance(
        points,
        np.ndarray,
    )

    assert points.ndim == 2

    assert points.shape[1] == 2

    assert np.isfinite(
        points
    ).all(), (
        f"Non-finite point detected: "
        f"{row.checkpoint_id}, "
        f"{row.image_id}"
    )


# ------------------------------------------------------------
# Summary by frozen system
# ------------------------------------------------------------

reference_point_summary_df = (
    e3_checkpoint_points_df
    .groupby(
        "checkpoint_id",
        observed=True,
    )
    .agg(
        num_images=(
            "image_id",
            "size",
        ),
        total_reference_points=(
            "num_reference_points",
            "sum",
        ),
        mean_points_per_image=(
            "num_reference_points",
            "mean",
        ),
        median_points_per_image=(
            "num_reference_points",
            "median",
        ),
        min_points=(
            "num_reference_points",
            "min",
        ),
        max_points=(
            "num_reference_points",
            "max",
        ),
        zero_point_images=(
            "num_reference_points",
            lambda s: int(
                (s == 0).sum()
            ),
        ),
    )
    .reindex(
        [
            "overall_joint",
            "sparse_joint",
            "dense_recovery",
        ]
    )
    .reset_index()
)


# ------------------------------------------------------------
# Final sanity checks
# ------------------------------------------------------------

assert (
    reference_point_summary_df[
        "num_images"
    ].tolist()
    ==
    [100, 100, 100]
)

assert (
    e3_checkpoint_points_df[
        "num_reference_points"
    ]
    .sum()
    ==
    reference_point_summary_df[
        "total_reference_points"
    ]
    .sum()
)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("=" * 110)
print("STEP 2.3 - FROZEN SEMANTIC REFERENCE-POINT VALIDATION")
print("=" * 110)

print(
    f"Checkpoint rows                : "
    f"{len(e3_checkpoint_points_df)}"
)

print(
    "pred_count == parsed point count: PASS"
)

print(
    "Point-array shape (N, 2)        : PASS"
)

print(
    "Finite coordinates              : PASS"
)

print("\nReference-point summary:")

display(
    reference_point_summary_df.round(4)
)

print("\nFirst 3 parsed checkpoint rows:")

display(
    e3_checkpoint_points_df[
        [
            "checkpoint_id",
            "dataset_index",
            "image_id",
            "category",
            "fusion_family",
            "c_variant",
            "d_variant",
            "tau",
            "pred_count",
            "num_reference_points",
        ]
    ]
    .head(3)
)

print("=" * 110)

STEP 2.3 - FROZEN SEMANTIC REFERENCE-POINT VALIDATION
Checkpoint rows                : 300
pred_count == parsed point count: PASS
Point-array shape (N, 2)        : PASS
Finite coordinates              : PASS

Reference-point summary:


,checkpoint_id,num_images,total_reference_points,mean_points_per_image,median_points_per_image,min_points,max_points,zero_point_images
0,overall_joint,100,5108,51.08,53.0,0,116,1
1,sparse_joint,100,4527,45.27,45.5,0,106,1
2,dense_recovery,100,5744,57.44,50.5,8,185,0



First 3 parsed checkpoint rows:


,checkpoint_id,dataset_index,image_id,category,fusion_family,c_variant,d_variant,tau,pred_count,num_reference_points
0,overall_joint,9,215.jpg,grapes,F2,C2b,D6B-T-native,0.01,40,40
1,overall_joint,14,220.jpg,grapes,F2,C2b,D6B-T-native,0.01,75,75
2,overall_joint,20,228.jpg,grapes,F2,C2b,D6B-T-native,0.01,79,79


In [9]:
# ============================================================
# Step 3.1 - Build FSC-147 Validation Dataset + Canonical GT
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Build FSC-147 validation dataset
# ------------------------------------------------------------

fsc147_val_dataset = FSC147Dataset(
    root=FSC147_RAW,
    split="val",
    load_images=True,
)


# ------------------------------------------------------------
# Canonical FSC-147 GT-point helper
# ------------------------------------------------------------

def get_gt_points(sample, category):
    """
    Return FSC-147 GT points as Nx2 continuous image
    coordinates [x, y].

    Important:
    These coordinates are preserved exactly as supplied by
    the dataset. They are NOT clipped to discrete pixel bounds.
    """

    points = sample[
        "categories"
    ][category]["points"]

    return np.asarray(
        points,
        dtype=float,
    ).reshape(-1, 2)


# ------------------------------------------------------------
# Recover and validate canonical dev100 metadata + GT points
# ------------------------------------------------------------

gt_records = []
gt_boundary_records = []


for row in fsc147_dev_100_df.itertuples(
    index=False
):

    dataset_index = int(
        row.dataset_index
    )

    assert (
        0
        <= dataset_index
        < len(fsc147_val_dataset)
    )

    sample = fsc147_val_dataset[
        dataset_index
    ]


    # --------------------------------------------------------
    # Validate image identity
    # --------------------------------------------------------

    assert (
        str(sample["image_id"])
        == str(row.image_id)
    ), (
        f"Image mismatch at "
        f"dataset_index={dataset_index}: "
        f"{sample['image_id']} != {row.image_id}"
    )


    # --------------------------------------------------------
    # Validate target category
    # --------------------------------------------------------

    assert (
        row.category
        in sample["categories"]
    ), (
        f"Category '{row.category}' "
        f"not found for {row.image_id}"
    )


    # --------------------------------------------------------
    # Recover canonical GT points
    # --------------------------------------------------------

    gt_points = get_gt_points(
        sample,
        row.category,
    )

    assert gt_points.ndim == 2
    assert gt_points.shape[1] == 2

    assert np.isfinite(
        gt_points
    ).all()


    # --------------------------------------------------------
    # Validate GT count
    # --------------------------------------------------------

    gt_count = int(
        sample[
            "categories"
        ][row.category]["count"]
    )

    assert (
        gt_count
        == int(row.gt_count)
    ), (
        f"GT count mismatch for "
        f"{row.image_id}: "
        f"dataset={gt_count}, "
        f"dev100={row.gt_count}"
    )

    assert (
        len(gt_points)
        == gt_count
    ), (
        f"GT point-count mismatch for "
        f"{row.image_id}: "
        f"points={len(gt_points)}, "
        f"count={gt_count}"
    )


    # --------------------------------------------------------
    # Recover image dimensions
    # --------------------------------------------------------

    image = sample["image"]

    image_np = np.asarray(
        image
    )

    assert image_np.ndim == 3

    height, width = (
        image_np.shape[:2]
    )


    # --------------------------------------------------------
    # Diagnose GT boundary coordinates
    #
    # FSC-147 contains a small number of annotation points
    # that lie exactly on, or approximately one pixel outside,
    # the strict discrete-pixel domain.
    #
    # Preserve the original continuous coordinates.
    # Do NOT clip them here.
    # --------------------------------------------------------

    if len(gt_points) > 0:

        strict_in_bounds = (
            (gt_points[:, 0] >= 0)
            & (gt_points[:, 0] < width)
            & (gt_points[:, 1] >= 0)
            & (gt_points[:, 1] < height)
        )

        bad_indices = np.where(
            ~strict_in_bounds
        )[0]

        for point_idx in bad_indices:

            x = float(
                gt_points[
                    point_idx,
                    0
                ]
            )

            y = float(
                gt_points[
                    point_idx,
                    1
                ]
            )

            gt_boundary_records.append(
                {
                    "dataset_index":
                        dataset_index,

                    "image_id":
                        str(row.image_id),

                    "category":
                        str(row.category),

                    "point_index":
                        int(point_idx),

                    "x":
                        x,

                    "y":
                        y,

                    "width":
                        int(width),

                    "height":
                        int(height),
                }
            )


        # Known FSC-147 boundary-coordinate behaviour:
        # allow at most ~1 pixel outside the discrete image.
        #
        # This catches genuinely corrupted coordinates while
        # retaining the previously identified annotation cases.

        tolerance_px = 1.0001

        assert (
            (gt_points[:, 0] >= -tolerance_px)
            &
            (
                gt_points[:, 0]
                <= width + tolerance_px
            )
            &
            (gt_points[:, 1] >= -tolerance_px)
            &
            (
                gt_points[:, 1]
                <= height + tolerance_px
            )
        ).all(), (
            "Unexpected GT coordinate more than "
            f"{tolerance_px:.4f} px outside image: "
            f"{row.image_id}"
        )


    # --------------------------------------------------------
    # Store canonical dev100 record
    # --------------------------------------------------------

    gt_records.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                str(row.image_id),

            "category":
                str(row.category),

            "density_stratum":
                str(row.density_stratum),

            "width":
                int(width),

            "height":
                int(height),

            "gt_count":
                gt_count,

            # Preserve original continuous coordinates.
            "gt_points_xy":
                gt_points,
        }
    )


dev100_gt_df = pd.DataFrame(
    gt_records
)

gt_boundary_df = pd.DataFrame(
    gt_boundary_records
)


# ------------------------------------------------------------
# Final integrity checks
# ------------------------------------------------------------

assert len(
    dev100_gt_df
) == 100

assert (
    dev100_gt_df[
        "dataset_index"
    ].is_unique
)

assert (
    dev100_gt_df[
        "image_id"
    ].is_unique
)

assert (
    dev100_gt_df[
        "gt_count"
    ].sum()
    == 10045
)

assert (
    dev100_gt_df[
        "gt_points_xy"
    ].apply(len)
    ==
    dev100_gt_df[
        "gt_count"
    ]
).all()


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("=" * 110)
print("STEP 3.1 - FSC-147 VALIDATION DATASET + CANONICAL GT")
print("=" * 110)

print(
    "Validation dataset images :",
    len(fsc147_val_dataset),
)

print(
    "Frozen dev100 images      :",
    len(dev100_gt_df),
)

print(
    "Image/category mapping     : PASS"
)

print(
    "GT count mapping           : PASS"
)

print(
    "GT point-count consistency : PASS"
)

print(
    "Total GT objects           :",
    int(
        dev100_gt_df[
            "gt_count"
        ].sum()
    ),
)

print(
    "Strict-boundary GT points  :",
    len(gt_boundary_df),
)

print(
    "Boundary tolerance check   : PASS"
)

print(
    "Canonical GT modified      : NO"
)

print(
    "Test set accessed          : NO"
)


print("\nFirst 5 canonical dev100 rows:")

display(
    dev100_gt_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "width",
            "height",
            "gt_count",
        ]
    ]
    .head()
)


if len(gt_boundary_df) > 0:

    print(
        "\nGT points outside the strict "
        "discrete-pixel domain:"
    )

    display(
        gt_boundary_df
    )


print("=" * 110)

STEP 3.1 - FSC-147 VALIDATION DATASET + CANONICAL GT
Validation dataset images : 1286
Frozen dev100 images      : 100
Image/category mapping     : PASS
GT count mapping           : PASS
GT point-count consistency : PASS
Total GT objects           : 10045
Strict-boundary GT points  : 12
Boundary tolerance check   : PASS
Canonical GT modified      : NO
Test set accessed          : NO

First 5 canonical dev100 rows:


,dataset_index,image_id,category,density_stratum,width,height,gt_count
0,576,3873.jpg,shallots,S1_0_20,384,384,7
1,322,1964.jpg,bottle caps,S1_0_20,540,384,8
2,153,774.jpg,books,S1_0_20,575,384,8
3,360,2304.jpg,camels,S1_0_20,613,384,9
4,514,3573.jpg,chicken wings,S1_0_20,408,384,9



GT points outside the strict discrete-pixel domain:


,dataset_index,image_id,category,point_index,x,y,width,height
0,394,2840.jpg,chairs,0,578.000000,384.0,578,384
1,394,2840.jpg,chairs,1,578.000000,384.0,578,384
2,394,2840.jpg,chairs,2,578.000000,384.0,578,384
3,394,2840.jpg,chairs,3,578.000000,384.0,578,384
4,394,2840.jpg,chairs,4,578.000000,384.0,578,384
5,394,2840.jpg,chairs,5,578.000000,384.0,578,384
6,394,2840.jpg,chairs,6,578.000000,384.0,578,384
7,394,2840.jpg,chairs,7,578.000000,384.0,578,384
8,394,2840.jpg,chairs,8,578.000000,384.0,578,384
9,394,2840.jpg,chairs,9,578.000000,384.0,578,384


In [10]:
# ============================================================
# Step 3.2 - Attach Canonical GT + Validate Reference-Point Bounds
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Canonical dev100 metadata to attach
# ------------------------------------------------------------

canonical_cols = [
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "width",
    "height",
    "gt_count",
    "gt_points_xy",
]

canonical_dev100_df = (
    dev100_gt_df[
        canonical_cols
    ]
    .copy()
)


# ------------------------------------------------------------
# Attach canonical image / GT information
# ------------------------------------------------------------

checkpoint_input_df = (
    e3_checkpoint_points_df
    .merge(
        canonical_dev100_df,
        on=[
            "dataset_index",
            "image_id",
            "category",
        ],
        how="left",
        validate="many_to_one",
    )
)


# ------------------------------------------------------------
# Basic merge integrity
# ------------------------------------------------------------

assert len(
    checkpoint_input_df
) == 300

assert (
    checkpoint_input_df[
        "density_stratum"
    ].notna().all()
)

assert (
    checkpoint_input_df[
        "width"
    ].notna().all()
)

assert (
    checkpoint_input_df[
        "height"
    ].notna().all()
)

assert (
    checkpoint_input_df[
        "gt_count"
    ].notna().all()
)

assert (
    checkpoint_input_df[
        "gt_points_xy"
    ].notna().all()
)


# ------------------------------------------------------------
# Validate checkpoint image universe
# ------------------------------------------------------------

for checkpoint_id in [
    "overall_joint",
    "sparse_joint",
    "dense_recovery",
]:

    system_df = (
        checkpoint_input_df[
            checkpoint_input_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
        .copy()
    )

    assert len(
        system_df
    ) == 100

    assert (
        system_df[
            "dataset_index"
        ].nunique()
        == 100
    )

    assert (
        system_df[
            "image_id"
        ].nunique()
        == 100
    )


# ------------------------------------------------------------
# Validate GT attachment
# ------------------------------------------------------------

assert (
    checkpoint_input_df[
        "gt_points_xy"
    ]
    .apply(len)
    ==
    checkpoint_input_df[
        "gt_count"
    ]
    .astype(int)
).all()


# ------------------------------------------------------------
# Validate every frozen semantic reference point
# against its actual image dimensions
# ------------------------------------------------------------

reference_boundary_records = []

total_reference_points = 0


for row in checkpoint_input_df.itertuples(
    index=False
):

    points = np.asarray(
        row.pred_points_parsed,
        dtype=float,
    ).reshape(-1, 2)

    width = int(
        row.width
    )

    height = int(
        row.height
    )

    total_reference_points += len(
        points
    )


    # --------------------------------------------------------
    # Shape / numerical validity
    # --------------------------------------------------------

    assert points.ndim == 2
    assert points.shape[1] == 2

    assert np.isfinite(
        points
    ).all()


    # --------------------------------------------------------
    # Strict image-bound validation
    #
    # Unlike the small number of canonical FSC-147 GT
    # boundary annotations, the frozen semantic reference
    # points are inference prompts and should fall inside
    # the actual image domain.
    # --------------------------------------------------------

    if len(points) > 0:

        strict_in_bounds = (
            (points[:, 0] >= 0)
            & (points[:, 0] < width)
            & (points[:, 1] >= 0)
            & (points[:, 1] < height)
        )

        bad_indices = np.where(
            ~strict_in_bounds
        )[0]


        for point_idx in bad_indices:

            x = float(
                points[
                    point_idx,
                    0
                ]
            )

            y = float(
                points[
                    point_idx,
                    1
                ]
            )

            reference_boundary_records.append(
                {
                    "checkpoint_id":
                        row.checkpoint_id,

                    "dataset_index":
                        int(
                            row.dataset_index
                        ),

                    "image_id":
                        str(
                            row.image_id
                        ),

                    "category":
                        str(
                            row.category
                        ),

                    "point_index":
                        int(
                            point_idx
                        ),

                    "x":
                        x,

                    "y":
                        y,

                    "width":
                        width,

                    "height":
                        height,
                }
            )


reference_boundary_df = pd.DataFrame(
    reference_boundary_records
)


# ------------------------------------------------------------
# Frozen prompt-count integrity
# ------------------------------------------------------------

expected_total_reference_points = int(
    reference_point_summary_df[
        "total_reference_points"
    ].sum()
)

assert (
    total_reference_points
    ==
    expected_total_reference_points
)

assert (
    total_reference_points
    == 15379
)


# ------------------------------------------------------------
# Inference-prompt boundary integrity
# ------------------------------------------------------------

assert len(
    reference_boundary_df
) == 0, (
    "At least one frozen semantic reference point "
    "lies outside its image boundary."
)


# ------------------------------------------------------------
# Summary by checkpoint
# ------------------------------------------------------------

checkpoint_input_summary_df = (
    checkpoint_input_df
    .groupby(
        "checkpoint_id",
        observed=True,
    )
    .agg(
        num_images=(
            "image_id",
            "size",
        ),
        total_reference_points=(
            "num_reference_points",
            "sum",
        ),
        total_gt_objects=(
            "gt_count",
            "sum",
        ),
        zero_point_images=(
            "num_reference_points",
            lambda s: int(
                (s == 0).sum()
            ),
        ),
    )
    .reindex(
        [
            "overall_joint",
            "sparse_joint",
            "dense_recovery",
        ]
    )
    .reset_index()
)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("=" * 110)
print("STEP 3.2 - CHECKPOINT INPUT + REFERENCE-POINT BOUND VALIDATION")
print("=" * 110)

print(
    "Checkpoint rows             :",
    len(checkpoint_input_df),
)

print(
    "Systems × images            : 3 × 100"
)

print(
    "Canonical GT attachment     : PASS"
)

print(
    "GT point-count consistency  : PASS"
)

print(
    "Total reference points      :",
    total_reference_points,
)

print(
    "Expected reference points   : 15379"
)

print(
    "Reference-point bounds      : PASS"
)

print(
    "Out-of-bound ref. points    :",
    len(reference_boundary_df),
)

print(
    "Density routing             : NO"
)

print(
    "Test set accessed           : NO"
)


print("\nCheckpoint input summary:")

display(
    checkpoint_input_summary_df
)


print("\nFirst 5 checkpoint input rows:")

display(
    checkpoint_input_df[
        [
            "checkpoint_id",
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "width",
            "height",
            "gt_count",
            "num_reference_points",
        ]
    ]
    .head()
)

print("=" * 110)

STEP 3.2 - CHECKPOINT INPUT + REFERENCE-POINT BOUND VALIDATION
Checkpoint rows             : 300
Systems × images            : 3 × 100
Canonical GT attachment     : PASS
GT point-count consistency  : PASS
Total reference points      : 15379
Expected reference points   : 15379
Reference-point bounds      : PASS
Out-of-bound ref. points    : 0
Density routing             : NO
Test set accessed           : NO

Checkpoint input summary:


,checkpoint_id,num_images,total_reference_points,total_gt_objects,zero_point_images
0,overall_joint,100,5108,10045,1
1,sparse_joint,100,4527,10045,1
2,dense_recovery,100,5744,10045,0



First 5 checkpoint input rows:


,checkpoint_id,dataset_index,image_id,category,density_stratum,width,height,gt_count,num_reference_points
0,overall_joint,9,215.jpg,grapes,S6_95_100,408,384,259,40
1,overall_joint,14,220.jpg,grapes,S4_60_80,576,384,59,75
2,overall_joint,20,228.jpg,grapes,S4_60_80,631,384,44,79
3,overall_joint,31,241.jpg,grapes,S3_40_60,588,384,35,44
4,overall_joint,40,252.jpg,grapes,S3_40_60,512,384,22,31


In [11]:
# ============================================================
# Step 4.1 - Initialise Frozen-Comparable Guided SAM2
# ============================================================

import hashlib
import subprocess
import sys
import torch
from pathlib import Path


# ------------------------------------------------------------
# Frozen guided-SAM2 protocol
# ------------------------------------------------------------

SAM2_PROMPT_MODE = "single_positive"
SAM2_MULTIMASK_OUTPUT = False
SAM2_INSTANCE_SELECTION = "retain_all"


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    torch.cuda.get_device_name(0)
    if device.type == "cuda"
    else "CPU",
)


# ------------------------------------------------------------
# Recover exact SAM2 provenance from frozen B0
# ------------------------------------------------------------

assert (
    "code_provenance"
    in b0_metadata
)

assert (
    "sam2_repository"
    in b0_metadata["code_provenance"]
)

assert (
    "sam2_checkpoint"
    in b0_metadata["code_provenance"]
)


B0_SAM2_COMMIT = (
    b0_metadata[
        "code_provenance"
    ][
        "sam2_repository"
    ][
        "commit_sha"
    ]
)

B0_SAM2_CHECKPOINT_SHA256 = (
    b0_metadata[
        "code_provenance"
    ][
        "sam2_checkpoint"
    ][
        "sha256"
    ]
)


print(
    "\nFrozen B0 SAM2 commit:",
    B0_SAM2_COMMIT,
)

print(
    "Frozen B0 checkpoint SHA256:",
    B0_SAM2_CHECKPOINT_SHA256,
)


# ------------------------------------------------------------
# Install official SAM2 repository
#
# Use the same working installation pattern used in Group E.
# ------------------------------------------------------------

SAM2_ROOT = Path(
    "/content/sam2"
)

%cd /content

!rm -rf /content/sam2

!git clone -q \
    https://github.com/facebookresearch/sam2.git \
    /content/sam2


# ------------------------------------------------------------
# Check repository commit BEFORE installation
# ------------------------------------------------------------

sam2_actual_commit = (
    subprocess.check_output(
        [
            "git",
            "-C",
            str(SAM2_ROOT),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    )
    .strip()
)


print(
    "\nCurrent SAM2 commit:",
    sam2_actual_commit,
)


assert (
    sam2_actual_commit
    == B0_SAM2_COMMIT
), (
    "SAM2 source version differs from frozen B0.\n"
    f"B0 commit : {B0_SAM2_COMMIT}\n"
    f"Current   : {sam2_actual_commit}\n\n"
    "STOP before inference so the B0 comparison "
    "remains apples-to-apples."
)


# ------------------------------------------------------------
# Install exact currently verified checkout
# ------------------------------------------------------------

%cd /content/sam2

!pip install -q -e .


# ------------------------------------------------------------
# Prepare SAM2.1 Hiera-Small checkpoint
# ------------------------------------------------------------

SAM2_CHECKPOINT_DIR = (
    SAM2_ROOT
    / "checkpoints"
)

SAM2_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SAM2_CHECKPOINT = (
    SAM2_CHECKPOINT_DIR
    / "sam2.1_hiera_small.pt"
)


if not SAM2_CHECKPOINT.exists():

    print(
        "\nDownloading SAM2.1 "
        "Hiera-Small checkpoint..."
    )

    !wget -q \
        -O {SAM2_CHECKPOINT} \
        https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt


assert SAM2_CHECKPOINT.exists()


# ------------------------------------------------------------
# Verify checkpoint SHA-256 against frozen B0
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


sam2_checkpoint_sha256 = (
    sha256_file(
        SAM2_CHECKPOINT
    )
)


assert (
    sam2_checkpoint_sha256
    == B0_SAM2_CHECKPOINT_SHA256
), (
    "SAM2 checkpoint differs from frozen B0.\n"
    f"B0 SHA256 : "
    f"{B0_SAM2_CHECKPOINT_SHA256}\n"
    f"Current    : "
    f"{sam2_checkpoint_sha256}"
)


# ------------------------------------------------------------
# Build SAM2.1 Hiera-Small image predictor
# ------------------------------------------------------------

if str(SAM2_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(SAM2_ROOT),
    )


from sam2.build_sam import build_sam2

from sam2.sam2_image_predictor import (
    SAM2ImagePredictor,
)


SAM2_MODEL_CFG = (
    "configs/sam2.1/"
    "sam2.1_hiera_s.yaml"
)


sam2_model = build_sam2(
    SAM2_MODEL_CFG,
    str(SAM2_CHECKPOINT),
    device=device,
)

sam2_model.eval()


sam2_predictor = (
    SAM2ImagePredictor(
        sam2_model
    )
)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("STEP 4.1 - GUIDED SAM2 INITIALISATION")
print("=" * 110)

print(
    "SAM2 source matches B0     : PASS"
)

print(
    "SAM2 checkpoint matches B0 : PASS"
)

print(
    "Model                     : SAM2.1 Hiera-Small"
)

print(
    "Model config              :",
    SAM2_MODEL_CFG,
)

print(
    "Predictor                 :",
    sam2_predictor.__class__.__name__,
)

print(
    "Prompt mode               :",
    SAM2_PROMPT_MODE,
)

print(
    "Multimask output          :",
    SAM2_MULTIMASK_OUTPUT,
)

print(
    "Instance selection        :",
    SAM2_INSTANCE_SELECTION,
)

print(
    "Device                    :",
    device,
)

print("=" * 110)

Device: Tesla T4

Frozen B0 SAM2 commit: 2b90b9f5ceec907a1c18123530e92e794ad901a4
Frozen B0 checkpoint SHA256: 6d1aa6f30de5c92224f8172114de081d104bbd23dd9dc5c58996f0cad5dc4d38
/content

Current SAM2 commit: 2b90b9f5ceec907a1c18123530e92e794ad901a4
/content/sam2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 15.2 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done


STEP 4.1 - GUIDED SAM2 INITIALISATION
SAM2 source matches B0     : PASS
SAM2 checkpoint matches B0 : PASS
Model                     : SAM2.1 Hiera-Small
Model config              : configs/

In [12]:
# ============================================================
# Step 4.2 - Sanity-Check Single-Point Guided SAM2 Interface
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Select one frozen checkpoint image with >= 3 reference points
# ------------------------------------------------------------

sanity_row = (
    checkpoint_input_df[
        (
            checkpoint_input_df["checkpoint_id"]
            == "overall_joint"
        )
        &
        (
            checkpoint_input_df["num_reference_points"]
            >= 3
        )
    ]
    .iloc[0]
)


dataset_index = int(
    sanity_row["dataset_index"]
)

image_id = str(
    sanity_row["image_id"]
)

category = str(
    sanity_row["category"]
)

reference_points = np.asarray(
    sanity_row["pred_points_parsed"],
    dtype=np.float32,
).reshape(-1, 2)


# ------------------------------------------------------------
# Load the corresponding FSC-147 image
# ------------------------------------------------------------

sample = fsc147_val_dataset[
    dataset_index
]

assert (
    str(sample["image_id"])
    == image_id
)

assert (
    category
    in sample["categories"]
)


image_np = np.asarray(
    sample["image"]
)


# SAM2ImagePredictor expects H x W x 3 image input.
assert image_np.ndim == 3
assert image_np.shape[2] == 3

height, width = (
    image_np.shape[:2]
)

assert (
    width
    == int(sanity_row["width"])
)

assert (
    height
    == int(sanity_row["height"])
)


# ------------------------------------------------------------
# Set image ONCE
# ------------------------------------------------------------

sam2_predictor.set_image(
    image_np
)


# ------------------------------------------------------------
# Run first 3 reference points independently
# ------------------------------------------------------------

sanity_records = []


for point_id, point_xy in enumerate(
    reference_points[:3]
):

    x = float(
        point_xy[0]
    )

    y = float(
        point_xy[1]
    )


    # One independent positive point prompt
    point_coords = np.asarray(
        [[x, y]],
        dtype=np.float32,
    )

    point_labels = np.asarray(
        [1],
        dtype=np.int32,
    )


    masks, scores, logits = (
        sam2_predictor.predict(
            point_coords=point_coords,
            point_labels=point_labels,
            multimask_output=SAM2_MULTIMASK_OUTPUT,
        )
    )


    # --------------------------------------------------------
    # Validate SAM2 output structure
    # --------------------------------------------------------

    assert isinstance(
        masks,
        np.ndarray,
    )

    assert masks.ndim == 3, (
        f"Unexpected mask shape: {masks.shape}"
    )

    assert masks.shape[0] == 1, (
        "Expected exactly one mask for "
        "multimask_output=False."
    )

    assert masks.shape[1:] == (
        height,
        width,
    ), (
        f"Mask/image size mismatch: "
        f"{masks.shape[1:]} vs "
        f"{(height, width)}"
    )


    assert len(scores) == 1

    assert np.isfinite(
        scores
    ).all()

    assert np.isfinite(
        logits
    ).all()


    # --------------------------------------------------------
    # Retain the single returned mask
    # --------------------------------------------------------

    mask = masks[0].astype(
        bool
    )

    sam2_score = float(
        scores[0]
    )

    mask_area_px = int(
        mask.sum()
    )


    # --------------------------------------------------------
    # Diagnostic:
    # does the positive prompt pixel lie inside its mask?
    #
    # Use nearest valid discrete pixel for this check only.
    # The original continuous point remains unchanged.
    # --------------------------------------------------------

    px = int(
        np.clip(
            np.rint(x),
            0,
            width - 1,
        )
    )

    py = int(
        np.clip(
            np.rint(y),
            0,
            height - 1,
        )
    )

    prompt_inside_mask = bool(
        mask[
            py,
            px
        ]
    )


    sanity_records.append(
        {
            "point_id":
                point_id,

            "x":
                x,

            "y":
                y,

            "pixel_x":
                px,

            "pixel_y":
                py,

            "num_masks":
                int(
                    masks.shape[0]
                ),

            "mask_height":
                int(
                    masks.shape[1]
                ),

            "mask_width":
                int(
                    masks.shape[2]
                ),

            "sam2_score":
                sam2_score,

            "mask_area_px":
                mask_area_px,

            "prompt_inside_mask":
                prompt_inside_mask,
        }
    )


sam2_sanity_df = pd.DataFrame(
    sanity_records
)


# ------------------------------------------------------------
# Protocol-level sanity assertions
# ------------------------------------------------------------

assert len(
    sam2_sanity_df
) == 3

assert (
    sam2_sanity_df[
        "num_masks"
    ]
    == 1
).all()

assert (
    sam2_sanity_df[
        "mask_height"
    ]
    == height
).all()

assert (
    sam2_sanity_df[
        "mask_width"
    ]
    == width
).all()

assert (
    sam2_sanity_df[
        "mask_area_px"
    ]
    > 0
).all()


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("=" * 110)
print("STEP 4.2 - SINGLE-POINT GUIDED SAM2 SANITY CHECK")
print("=" * 110)

print(
    "Checkpoint      :",
    sanity_row["checkpoint_id"],
)

print(
    "Image           :",
    image_id,
)

print(
    "Dataset index   :",
    dataset_index,
)

print(
    "Category        :",
    category,
)

print(
    "Image size      :",
    f"{width} × {height}",
)

print(
    "Points tested   : 3"
)

print(
    "Prompt mode     :",
    SAM2_PROMPT_MODE,
)

print(
    "Multimask       :",
    SAM2_MULTIMASK_OUTPUT,
)

print(
    "Returned masks  : 1 per point — PASS"
)

print(
    "Mask dimensions : source image size — PASS"
)

print(
    "Finite scores   : PASS"
)

print(
    "Non-empty masks : PASS"
)

print("\nPer-point sanity diagnostics:")

display(
    sam2_sanity_df.round(4)
)

print(
    "\nPositive prompt pixel inside returned mask:"
)

print(
    f"{int(sam2_sanity_df['prompt_inside_mask'].sum())}"
    f" / {len(sam2_sanity_df)}"
)

print("=" * 110)

/usr/local/lib/python3.13/dist-packages/torchvision/transforms/functional.py:154: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  img = torch.from_numpy(pic.transpose((2, 0, 1))).contiguous()


STEP 4.2 - SINGLE-POINT GUIDED SAM2 SANITY CHECK
Checkpoint      : overall_joint
Image           : 215.jpg
Dataset index   : 9
Category        : grapes
Image size      : 408 × 384
Points tested   : 3
Prompt mode     : single_positive
Multimask       : False
Returned masks  : 1 per point — PASS
Mask dimensions : source image size — PASS
Finite scores   : PASS
Non-empty masks : PASS

Per-point sanity diagnostics:


,point_id,x,y,pixel_x,pixel_y,num_masks,mask_height,mask_width,sam2_score,mask_area_px,prompt_inside_mask
0,0,384.8182,33.6000,385,34,1,384,408,0.9016,1847,True
1,1,368.0239,35.6391,368,36,1,384,408,0.9096,1914,True
2,2,371.4947,57.2449,371,57,1,384,408,0.7448,1156,True



Positive prompt pixel inside returned mask:
3 / 3


In [14]:
# ============================================================
# Step 5.1 - Define Guided Inference + Mask-Cache Protocol
# ============================================================

import json
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# 1. End-to-end checkpoint output directories
# ------------------------------------------------------------

E2E_TABLES_DIR = (
    E2E_RESULTS_DIR
    / "tables"
)

E2E_MASK_CACHE_DIR = (
    E2E_RESULTS_DIR
    / "guided_mask_cache"
)

E2E_METADATA_DIR = (
    E2E_RESULTS_DIR
    / "metadata"
)


for path in [
    E2E_RESULTS_DIR,
    E2E_TABLES_DIR,
    E2E_MASK_CACHE_DIR,
    E2E_METADATA_DIR,
]:

    path.mkdir(
        parents=True,
        exist_ok=True,
    )


CHECKPOINT_IDS = [
    "overall_joint",
    "sparse_joint",
    "dense_recovery",
]


for checkpoint_id in CHECKPOINT_IDS:

    (
        E2E_MASK_CACHE_DIR
        / checkpoint_id
    ).mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# 2. Freeze inference protocol
# ------------------------------------------------------------

guided_inference_protocol = {

    "dataset":
        "FSC-147",

    "split":
        "val",

    "subset":
        "frozen dev100",

    "num_images":
        100,

    "checkpoint_ids":
        CHECKPOINT_IDS,

    "num_checkpoint_systems":
        3,

    "total_reference_points":
        int(
            checkpoint_input_df[
                "num_reference_points"
            ].sum()
        ),

    "sam2_model":
        "SAM2.1 Hiera-Small",

    "sam2_model_config":
        SAM2_MODEL_CFG,

    "sam2_commit":
        B0_SAM2_COMMIT,

    "sam2_checkpoint_sha256":
        B0_SAM2_CHECKPOINT_SHA256,

    "prompt_mode":
        SAM2_PROMPT_MODE,

    "multimask_output":
        SAM2_MULTIMASK_OUTPUT,

    "instance_selection":
        SAM2_INSTANCE_SELECTION,

    "prompt_label":
        1,

    "prompt_execution":
        (
            "Each frozen semantic reference point "
            "is submitted independently as one "
            "positive SAM2 point prompt."
        ),

    "image_embedding_policy":
        (
            "SAM2 set_image() once per image; "
            "all prompts for the three frozen systems "
            "reuse that image embedding."
        ),

    "mask_filtering":
        False,

    "mask_deduplication":
        False,

    "score_thresholding":
        False,

    "counting_rule":
        (
            "One retained SAM2 mask per frozen "
            "semantic reference point."
        ),

    "gt_used_during_inference":
        False,

    "density_routing":
        False,

    "test_set_accessed":
        False,

    "mask_cache_schema_version":
        1,
}


# ------------------------------------------------------------
# 3. Lossless binary-mask packing helpers
# ------------------------------------------------------------

MASK_PACK_BITORDER = "little"


def pack_binary_masks(mask_stack):
    """
    Losslessly bit-pack a boolean mask stack.

    Input
    -----
    mask_stack : ndarray
        Shape (N, H, W), dtype bool.
        N may be zero.

    Returns
    -------
    packed : ndarray
        Shape (N, ceil(H*W/8)), dtype uint8.
    height : int
    width  : int
    """

    mask_stack = np.asarray(
        mask_stack,
        dtype=bool,
    )

    assert mask_stack.ndim == 3

    num_masks, height, width = (
        mask_stack.shape
    )

    num_pixels = (
        int(height)
        * int(width)
    )

    bytes_per_mask = (
        num_pixels + 7
    ) // 8


    # Explicitly support zero-mask images.
    if num_masks == 0:

        packed = np.empty(
            (
                0,
                bytes_per_mask,
            ),
            dtype=np.uint8,
        )

        return (
            packed,
            int(height),
            int(width),
        )


    flat = mask_stack.reshape(
        num_masks,
        num_pixels,
    )

    packed = np.packbits(
        flat,
        axis=1,
        bitorder=MASK_PACK_BITORDER,
    )

    return (
        packed,
        int(height),
        int(width),
    )


def unpack_binary_masks(
    packed,
    height,
    width,
):
    """
    Losslessly restore packed masks to (N, H, W).

    Supports N = 0.
    """

    packed = np.asarray(
        packed,
        dtype=np.uint8,
    )

    if packed.ndim == 1:

        packed = packed.reshape(
            1,
            -1,
        )


    assert packed.ndim == 2

    num_masks = int(
        packed.shape[0]
    )

    height = int(
        height
    )

    width = int(
        width
    )

    num_pixels = (
        height * width
    )


    if num_masks == 0:

        return np.empty(
            (
                0,
                height,
                width,
            ),
            dtype=bool,
        )


    bits = np.unpackbits(
        packed,
        axis=1,
        bitorder=MASK_PACK_BITORDER,
    )

    bits = bits[
        :,
        :num_pixels
    ]

    return (
        bits
        .reshape(
            num_masks,
            height,
            width,
        )
        .astype(bool)
    )


# ------------------------------------------------------------
# 4. Cache-file naming helper
# ------------------------------------------------------------

def guided_mask_cache_path(
    checkpoint_id,
    dataset_index,
    image_id,
):

    assert (
        checkpoint_id
        in CHECKPOINT_IDS
    )

    image_stem = Path(
        str(image_id)
    ).stem

    filename = (
        f"{int(dataset_index):04d}"
        f"_{image_stem}.npz"
    )

    return (
        E2E_MASK_CACHE_DIR
        / checkpoint_id
        / filename
    )


# ------------------------------------------------------------
# 5. Frozen cache contents
# ------------------------------------------------------------

MASK_CACHE_FIELDS = [
    "checkpoint_id",
    "dataset_index",
    "image_id",
    "packed_masks",
    "prompt_points_xy",
    "sam2_scores",
    "mask_area_px",
    "prompt_inside_mask",
    "height",
    "width",
]


# ------------------------------------------------------------
# 6. Estimate mask-cache storage
# ------------------------------------------------------------

raw_bool_bytes = 0
packed_bytes = 0


for row in checkpoint_input_df.itertuples(
    index=False
):

    n = int(
        row.num_reference_points
    )

    height = int(
        row.height
    )

    width = int(
        row.width
    )

    num_pixels = (
        height * width
    )


    # NumPy bool ~ 1 byte/pixel.
    raw_bool_bytes += (
        n * num_pixels
    )


    bytes_per_packed_mask = (
        num_pixels + 7
    ) // 8

    packed_bytes += (
        n
        * bytes_per_packed_mask
    )


raw_bool_mib = (
    raw_bool_bytes
    / (1024 ** 2)
)

packed_mib = (
    packed_bytes
    / (1024 ** 2)
)


# ------------------------------------------------------------
# 7. Verify pack/unpack is exactly lossless
# ------------------------------------------------------------

# Non-empty case
_test_masks = np.zeros(
    (3, 5, 7),
    dtype=bool,
)

_test_masks[
    0,
    1:4,
    2:5
] = True

_test_masks[
    1,
    :,
    0
] = True

_test_masks[
    2,
    4,
    6
] = True


(
    _test_packed,
    _test_h,
    _test_w,
) = pack_binary_masks(
    _test_masks
)

_test_restored = (
    unpack_binary_masks(
        _test_packed,
        _test_h,
        _test_w,
    )
)

assert np.array_equal(
    _test_masks,
    _test_restored,
)


# Zero-mask case
_test_empty_masks = np.empty(
    (0, 5, 7),
    dtype=bool,
)

(
    _test_empty_packed,
    _test_empty_h,
    _test_empty_w,
) = pack_binary_masks(
    _test_empty_masks
)

_test_empty_restored = (
    unpack_binary_masks(
        _test_empty_packed,
        _test_empty_h,
        _test_empty_w,
    )
)

assert (
    _test_empty_restored.shape
    == (0, 5, 7)
)

assert np.array_equal(
    _test_empty_masks,
    _test_empty_restored,
)


# ------------------------------------------------------------
# 8. Protocol integrity checks
# ------------------------------------------------------------

assert (
    guided_inference_protocol[
        "total_reference_points"
    ]
    == 15379
)

assert (
    guided_inference_protocol[
        "prompt_mode"
    ]
    == "single_positive"
)

assert (
    guided_inference_protocol[
        "multimask_output"
    ]
    is False
)

assert (
    guided_inference_protocol[
        "instance_selection"
    ]
    == "retain_all"
)

assert (
    guided_inference_protocol[
        "mask_filtering"
    ]
    is False
)

assert (
    guided_inference_protocol[
        "mask_deduplication"
    ]
    is False
)

assert (
    guided_inference_protocol[
        "gt_used_during_inference"
    ]
    is False
)

assert (
    guided_inference_protocol[
        "density_routing"
    ]
    is False
)


# ------------------------------------------------------------
# 9. Confirmation
# ------------------------------------------------------------

print("=" * 110)
print("STEP 5.1 - GUIDED INFERENCE + MASK-CACHE PROTOCOL")
print("=" * 110)

print(
    "Checkpoint systems        :",
    len(CHECKPOINT_IDS),
)

print(
    "Images per system         : 100"
)

print(
    "Total semantic prompts    :",
    guided_inference_protocol[
        "total_reference_points"
    ],
)

print(
    "Prompt mode               :",
    SAM2_PROMPT_MODE,
)

print(
    "Multimask output          :",
    SAM2_MULTIMASK_OUTPUT,
)

print(
    "Instance selection        :",
    SAM2_INSTANCE_SELECTION,
)

print(
    "Mask filtering            : NO"
)

print(
    "Mask deduplication        : NO"
)

print(
    "Density routing           : NO"
)

print(
    "GT used during inference  : NO"
)

print(
    "Test set accessed         : NO"
)

print(
    "\nRaw boolean-mask size     : "
    f"{raw_bool_mib:.2f} MiB"
)

print(
    "Bit-packed size estimate  : "
    f"{packed_mib:.2f} MiB"
)

print(
    "Additional .npz compression: YES"
)

print(
    "\nNon-empty mask round-trip : PASS"
)

print(
    "Zero-mask round-trip      : PASS"
)

print(
    "\nMask-cache directory      :",
    E2E_MASK_CACHE_DIR,
)

print(
    "Tables directory          :",
    E2E_TABLES_DIR,
)

print(
    "Metadata directory        :",
    E2E_METADATA_DIR,
)

print("=" * 110)

STEP 5.1 - GUIDED INFERENCE + MASK-CACHE PROTOCOL
Checkpoint systems        : 3
Images per system         : 100
Total semantic prompts    : 15379
Prompt mode               : single_positive
Multimask output          : False
Instance selection        : retain_all
Mask filtering            : NO
Mask deduplication        : NO
Density routing           : NO
GT used during inference  : NO
Test set accessed         : NO

Raw boolean-mask size     : 3158.87 MiB
Bit-packed size estimate  : 394.86 MiB
Additional .npz compression: YES

Non-empty mask round-trip : PASS
Zero-mask round-trip      : PASS

Mask-cache directory      : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/guided_mask_cache
Tables directory          : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables
Metadata directory        : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoin

In [15]:
# ============================================================
# Step 5.2 - Full Resumable Guided SAM2 Inference
# ============================================================

import json
import os
import time

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm


# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

GUIDED_INFERENCE_TABLE_FILE = (
    E2E_TABLES_DIR
    / "guided_inference_per_image.csv"
)

GUIDED_PROTOCOL_FILE = (
    E2E_METADATA_DIR
    / "guided_inference_protocol.json"
)

GUIDED_RUN_METADATA_FILE = (
    E2E_METADATA_DIR
    / "guided_inference_run_metadata.json"
)


# ------------------------------------------------------------
# Save frozen inference protocol before inference begins
# ------------------------------------------------------------

with open(
    GUIDED_PROTOCOL_FILE,
    "w",
) as f:

    json.dump(
        guided_inference_protocol,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# Helper: validate and summarise one saved cache
# ------------------------------------------------------------

def validate_and_summarise_cache(
    cache_path,
    row,
    cache_status,
    prompt_inference_seconds,
):

    with np.load(
        cache_path,
        allow_pickle=False,
    ) as data:

        # ----------------------------------------------------
        # Required cache fields
        # ----------------------------------------------------

        missing_fields = (
            set(MASK_CACHE_FIELDS)
            - set(data.files)
        )

        assert not missing_fields, (
            f"Cache missing fields: "
            f"{sorted(missing_fields)}\n"
            f"{cache_path}"
        )


        # ----------------------------------------------------
        # Identity
        # ----------------------------------------------------

        cached_checkpoint_id = str(
            data["checkpoint_id"].item()
        )

        cached_dataset_index = int(
            data["dataset_index"].item()
        )

        cached_image_id = str(
            data["image_id"].item()
        )

        assert (
            cached_checkpoint_id
            == str(row.checkpoint_id)
        )

        assert (
            cached_dataset_index
            == int(row.dataset_index)
        )

        assert (
            cached_image_id
            == str(row.image_id)
        )


        # ----------------------------------------------------
        # Dimensions
        # ----------------------------------------------------

        height = int(
            data["height"].item()
        )

        width = int(
            data["width"].item()
        )

        assert (
            height
            == int(row.height)
        )

        assert (
            width
            == int(row.width)
        )


        # ----------------------------------------------------
        # Prompt / mask arrays
        # ----------------------------------------------------

        packed_masks = np.asarray(
            data["packed_masks"],
            dtype=np.uint8,
        )

        prompt_points_xy = np.asarray(
            data["prompt_points_xy"],
            dtype=np.float32,
        ).reshape(-1, 2)

        sam2_scores = np.asarray(
            data["sam2_scores"],
            dtype=np.float32,
        ).reshape(-1)

        mask_area_px = np.asarray(
            data["mask_area_px"],
            dtype=np.int64,
        ).reshape(-1)

        prompt_inside_mask = np.asarray(
            data["prompt_inside_mask"],
            dtype=bool,
        ).reshape(-1)


        num_masks = int(
            packed_masks.shape[0]
        )

        expected_num_masks = int(
            row.num_reference_points
        )


        assert (
            num_masks
            == expected_num_masks
        )

        assert (
            len(prompt_points_xy)
            == num_masks
        )

        assert (
            len(sam2_scores)
            == num_masks
        )

        assert (
            len(mask_area_px)
            == num_masks
        )

        assert (
            len(prompt_inside_mask)
            == num_masks
        )


        # ----------------------------------------------------
        # Frozen prompts must match upstream Group-E prompts
        # ----------------------------------------------------

        expected_points = np.asarray(
            row.pred_points_parsed,
            dtype=np.float32,
        ).reshape(-1, 2)

        assert np.array_equal(
            prompt_points_xy,
            expected_points,
        )


        # ----------------------------------------------------
        # Packed-mask shape
        # ----------------------------------------------------

        expected_bytes_per_mask = (
            (
                height * width
            )
            + 7
        ) // 8

        assert (
            packed_masks.ndim == 2
        )

        assert (
            packed_masks.shape[1]
            == expected_bytes_per_mask
        )


        # ----------------------------------------------------
        # Numerical integrity
        # ----------------------------------------------------

        assert np.isfinite(
            prompt_points_xy
        ).all()

        assert np.isfinite(
            sam2_scores
        ).all()

        assert (
            mask_area_px >= 0
        ).all()


        # ----------------------------------------------------
        # Per-image diagnostic summaries
        # ----------------------------------------------------

        if num_masks > 0:

            mean_score = float(
                sam2_scores.mean()
            )

            median_score = float(
                np.median(
                    sam2_scores
                )
            )

            mean_mask_area_px = float(
                mask_area_px.mean()
            )

            median_mask_area_px = float(
                np.median(
                    mask_area_px
                )
            )

            prompt_inside_rate = float(
                prompt_inside_mask.mean()
            )

            num_empty_masks = int(
                (
                    mask_area_px == 0
                ).sum()
            )

        else:

            mean_score = np.nan
            median_score = np.nan

            mean_mask_area_px = np.nan
            median_mask_area_px = np.nan

            prompt_inside_rate = np.nan

            num_empty_masks = 0


    return {

        "checkpoint_id":
            str(row.checkpoint_id),

        "dataset_index":
            int(row.dataset_index),

        "image_id":
            str(row.image_id),

        "category":
            str(row.category),

        "density_stratum":
            str(row.density_stratum),

        "width":
            int(row.width),

        "height":
            int(row.height),

        "gt_count":
            int(row.gt_count),

        "num_reference_points":
            int(row.num_reference_points),

        "num_retained_masks":
            int(num_masks),

        # Under retain_all, the end-to-end count
        # is the number of retained SAM2 masks.
        "pred_count":
            int(num_masks),

        "mean_sam2_score":
            mean_score,

        "median_sam2_score":
            median_score,

        "mean_mask_area_px":
            mean_mask_area_px,

        "median_mask_area_px":
            median_mask_area_px,

        "num_empty_masks":
            int(num_empty_masks),

        "prompt_inside_mask_rate":
            prompt_inside_rate,

        "prompt_inference_seconds":
            float(
                prompt_inference_seconds
            ),

        "cache_status":
            str(cache_status),

        "cache_path":
            str(cache_path),
    }


# ------------------------------------------------------------
# Helper: save cache atomically
#
# Write to temporary file first, then rename.
# An interrupted write therefore does not leave a seemingly
# complete .npz cache behind.
# ------------------------------------------------------------

def save_guided_cache_atomic(
    cache_path,
    checkpoint_id,
    dataset_index,
    image_id,
    packed_masks,
    prompt_points_xy,
    sam2_scores,
    mask_area_px,
    prompt_inside_mask,
    height,
    width,
):

    temp_path = (
        cache_path.parent
        / (
            cache_path.name
            + ".tmp"
        )
    )

    if temp_path.exists():
        temp_path.unlink()


    with open(
        temp_path,
        "wb",
    ) as f:

        np.savez_compressed(

            f,

            checkpoint_id=
                np.asarray(
                    str(checkpoint_id)
                ),

            dataset_index=
                np.asarray(
                    int(dataset_index),
                    dtype=np.int64,
                ),

            image_id=
                np.asarray(
                    str(image_id)
                ),

            packed_masks=
                np.asarray(
                    packed_masks,
                    dtype=np.uint8,
                ),

            prompt_points_xy=
                np.asarray(
                    prompt_points_xy,
                    dtype=np.float32,
                ).reshape(-1, 2),

            sam2_scores=
                np.asarray(
                    sam2_scores,
                    dtype=np.float32,
                ).reshape(-1),

            mask_area_px=
                np.asarray(
                    mask_area_px,
                    dtype=np.int64,
                ).reshape(-1),

            prompt_inside_mask=
                np.asarray(
                    prompt_inside_mask,
                    dtype=bool,
                ).reshape(-1),

            height=
                np.asarray(
                    int(height),
                    dtype=np.int64,
                ),

            width=
                np.asarray(
                    int(width),
                    dtype=np.int64,
                ),
        )


    os.replace(
        temp_path,
        cache_path,
    )


# ------------------------------------------------------------
# Build fast lookup:
#
# (dataset_index, checkpoint_id) -> frozen checkpoint row
# ------------------------------------------------------------

checkpoint_row_lookup = {}


for row in checkpoint_input_df.itertuples(
    index=False
):

    key = (
        int(row.dataset_index),
        str(row.checkpoint_id),
    )

    assert (
        key
        not in checkpoint_row_lookup
    )

    checkpoint_row_lookup[
        key
    ] = row


assert (
    len(checkpoint_row_lookup)
    == 300
)


# ------------------------------------------------------------
# Full inference
# ------------------------------------------------------------

inference_records = []

num_generated_caches = 0
num_reused_caches = 0

num_generated_prompts = 0
num_reused_prompts = 0

run_start_time = time.perf_counter()


print("=" * 110)
print("STEP 5.2 - FULL GUIDED SAM2 INFERENCE")
print("=" * 110)

print(
    "Images                  : 100"
)

print(
    "Systems per image       : 3"
)

print(
    "Total frozen prompts    : 15379"
)

print(
    "Resumable cache         : YES"
)

print(
    "GT used by SAM2         : NO"
)

print(
    "Mask filtering          : NO"
)

print(
    "Mask deduplication      : NO"
)

print(
    "\nStarting / resuming inference..."
)


# ------------------------------------------------------------
# Iterate authoritative dev100 once.
#
# Important:
# set_image() is performed at most once for each image,
# and the three systems reuse that image embedding.
# ------------------------------------------------------------

for image_number, dev_row in enumerate(

    tqdm(
        dev100_gt_df.itertuples(
            index=False
        ),
        total=len(
            dev100_gt_df
        ),
        desc="Guided SAM2 dev100",
    ),

    start=1,
):

    dataset_index = int(
        dev_row.dataset_index
    )

    image_id = str(
        dev_row.image_id
    )


    # --------------------------------------------------------
    # Retrieve the three frozen system rows
    # --------------------------------------------------------

    system_rows = {

        checkpoint_id:
            checkpoint_row_lookup[
                (
                    dataset_index,
                    checkpoint_id,
                )
            ]

        for checkpoint_id
        in CHECKPOINT_IDS
    }


    # --------------------------------------------------------
    # Determine which system/image caches already exist
    # --------------------------------------------------------

    cache_paths = {

        checkpoint_id:
            guided_mask_cache_path(
                checkpoint_id=
                    checkpoint_id,

                dataset_index=
                    dataset_index,

                image_id=
                    image_id,
            )

        for checkpoint_id
        in CHECKPOINT_IDS
    }


    missing_checkpoint_ids = [

        checkpoint_id

        for checkpoint_id
        in CHECKPOINT_IDS

        if not cache_paths[
            checkpoint_id
        ].exists()
    ]


    # --------------------------------------------------------
    # If any cache is missing, load image + set SAM2 embedding
    # ONCE for this image.
    # --------------------------------------------------------

    if missing_checkpoint_ids:

        sample = (
            fsc147_val_dataset[
                dataset_index
            ]
        )

        assert (
            str(
                sample["image_id"]
            )
            == image_id
        )


        # Writable copy avoids the torchvision warning seen
        # during the Step 4.2 sanity check.
        image_np = np.array(
            sample["image"],
            copy=True,
        )

        assert image_np.ndim == 3
        assert image_np.shape[2] == 3

        height, width = (
            image_np.shape[:2]
        )

        assert (
            height
            == int(
                dev_row.height
            )
        )

        assert (
            width
            == int(
                dev_row.width
            )
        )


        # ----------------------------------------------------
        # Compute image embedding once.
        # ----------------------------------------------------

        with torch.inference_mode():

            sam2_predictor.set_image(
                image_np
            )


            # ------------------------------------------------
            # Run only missing system/image caches
            # ------------------------------------------------

            for checkpoint_id in (
                missing_checkpoint_ids
            ):

                row = system_rows[
                    checkpoint_id
                ]

                points = np.asarray(
                    row.pred_points_parsed,
                    dtype=np.float32,
                ).reshape(-1, 2)


                assert (
                    len(points)
                    == int(
                        row.num_reference_points
                    )
                )


                masks_list = []
                scores_list = []
                areas_list = []
                prompt_inside_list = []


                prompt_start_time = (
                    time.perf_counter()
                )


                # --------------------------------------------
                # Each semantic point is an INDEPENDENT
                # positive SAM2 prompt.
                # --------------------------------------------

                for point_xy in points:

                    x = float(
                        point_xy[0]
                    )

                    y = float(
                        point_xy[1]
                    )


                    point_coords = np.asarray(
                        [[x, y]],
                        dtype=np.float32,
                    )

                    point_labels = np.asarray(
                        [1],
                        dtype=np.int32,
                    )


                    masks, scores, _ = (
                        sam2_predictor.predict(

                            point_coords=
                                point_coords,

                            point_labels=
                                point_labels,

                            multimask_output=
                                SAM2_MULTIMASK_OUTPUT,
                        )
                    )


                    # ----------------------------------------
                    # Frozen protocol:
                    # exactly one returned mask per prompt.
                    # ----------------------------------------

                    assert (
                        masks.ndim
                        == 3
                    )

                    assert (
                        masks.shape[0]
                        == 1
                    )

                    assert (
                        masks.shape[1:]
                        == (
                            height,
                            width,
                        )
                    )

                    assert (
                        len(scores)
                        == 1
                    )

                    assert np.isfinite(
                        scores
                    ).all()


                    # ----------------------------------------
                    # Retain mask exactly as returned.
                    #
                    # No score threshold.
                    # No mask-area threshold.
                    # No deduplication.
                    # ----------------------------------------

                    mask = np.asarray(
                        masks[0],
                        dtype=bool,
                    )

                    score = float(
                        scores[0]
                    )

                    mask_area_px = int(
                        mask.sum()
                    )


                    # Nearest valid pixel is used only for
                    # diagnostic prompt-in-mask lookup.
                    px = int(
                        np.clip(
                            np.rint(x),
                            0,
                            width - 1,
                        )
                    )

                    py = int(
                        np.clip(
                            np.rint(y),
                            0,
                            height - 1,
                        )
                    )

                    prompt_inside = bool(
                        mask[
                            py,
                            px
                        ]
                    )


                    masks_list.append(
                        mask
                    )

                    scores_list.append(
                        score
                    )

                    areas_list.append(
                        mask_area_px
                    )

                    prompt_inside_list.append(
                        prompt_inside
                    )


                prompt_inference_seconds = (
                    time.perf_counter()
                    - prompt_start_time
                )


                # --------------------------------------------
                # Build mask stack
                #
                # Explicitly support zero-point F2 images.
                # --------------------------------------------

                if len(
                    masks_list
                ) > 0:

                    mask_stack = np.stack(
                        masks_list,
                        axis=0,
                    )

                else:

                    mask_stack = np.empty(
                        (
                            0,
                            height,
                            width,
                        ),
                        dtype=bool,
                    )


                assert (
                    mask_stack.shape[0]
                    == len(points)
                )


                # --------------------------------------------
                # Lossless bit packing
                # --------------------------------------------

                (
                    packed_masks,
                    packed_height,
                    packed_width,
                ) = pack_binary_masks(
                    mask_stack
                )


                assert (
                    packed_height
                    == height
                )

                assert (
                    packed_width
                    == width
                )


                # --------------------------------------------
                # Incremental cache save
                # --------------------------------------------

                cache_path = (
                    cache_paths[
                        checkpoint_id
                    ]
                )


                save_guided_cache_atomic(

                    cache_path=
                        cache_path,

                    checkpoint_id=
                        checkpoint_id,

                    dataset_index=
                        dataset_index,

                    image_id=
                        image_id,

                    packed_masks=
                        packed_masks,

                    prompt_points_xy=
                        points,

                    sam2_scores=
                        scores_list,

                    mask_area_px=
                        areas_list,

                    prompt_inside_mask=
                        prompt_inside_list,

                    height=
                        height,

                    width=
                        width,
                )


                assert (
                    cache_path.exists()
                )


                num_generated_caches += 1

                num_generated_prompts += (
                    len(points)
                )


    # --------------------------------------------------------
    # Validate / summarise all three caches for this image.
    #
    # This also handles caches from an earlier interrupted run.
    # --------------------------------------------------------

    for checkpoint_id in CHECKPOINT_IDS:

        row = system_rows[
            checkpoint_id
        ]

        cache_path = (
            cache_paths[
                checkpoint_id
            ]
        )

        assert (
            cache_path.exists()
        )


        if (
            checkpoint_id
            in missing_checkpoint_ids
        ):

            cache_status = (
                "generated"
            )

            # Retrieve elapsed prompt time from the generated
            # cache's current-run bookkeeping.
            #
            # For simplicity, prompt time is reconstructed as
            # NaN here if multiple systems were generated.
            # It is not used for scientific evaluation.
            prompt_seconds_for_record = (
                np.nan
            )

        else:

            cache_status = (
                "reused"
            )

            prompt_seconds_for_record = (
                np.nan
            )

            num_reused_caches += 1

            num_reused_prompts += int(
                row.num_reference_points
            )


        record = (
            validate_and_summarise_cache(

                cache_path=
                    cache_path,

                row=
                    row,

                cache_status=
                    cache_status,

                prompt_inference_seconds=
                    prompt_seconds_for_record,
            )
        )


        inference_records.append(
            record
        )


    # --------------------------------------------------------
    # Incremental summary-table save after every image
    # --------------------------------------------------------

    current_inference_df = pd.DataFrame(
        inference_records
    )

    current_inference_df.to_csv(
        GUIDED_INFERENCE_TABLE_FILE,
        index=False,
    )


# ------------------------------------------------------------
# Full-run elapsed time
# ------------------------------------------------------------

run_elapsed_seconds = (
    time.perf_counter()
    - run_start_time
)


# ------------------------------------------------------------
# Final authoritative guided-inference table
# ------------------------------------------------------------

guided_inference_df = pd.DataFrame(
    inference_records
)


checkpoint_order = {
    checkpoint_id: i
    for i, checkpoint_id
    in enumerate(
        CHECKPOINT_IDS
    )
}


guided_inference_df[
    "_checkpoint_order"
] = (
    guided_inference_df[
        "checkpoint_id"
    ]
    .map(
        checkpoint_order
    )
)


guided_inference_df = (
    guided_inference_df
    .sort_values(
        [
            "_checkpoint_order",
            "dataset_index",
        ]
    )
    .drop(
        columns=[
            "_checkpoint_order"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Final integrity checks
# ------------------------------------------------------------

assert (
    len(
        guided_inference_df
    )
    == 300
)


for checkpoint_id in CHECKPOINT_IDS:

    system_df = (
        guided_inference_df[
            guided_inference_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
    )

    assert (
        len(system_df)
        == 100
    )


# One retained SAM2 mask per semantic point.
assert (
    guided_inference_df[
        "num_retained_masks"
    ]
    ==
    guided_inference_df[
        "num_reference_points"
    ]
).all()


assert (
    guided_inference_df[
        "pred_count"
    ]
    ==
    guided_inference_df[
        "num_retained_masks"
    ]
).all()


assert (
    guided_inference_df[
        "num_retained_masks"
    ].sum()
    == 15379
)


# Every expected cache must exist.
for row in guided_inference_df.itertuples(
    index=False
):

    assert Path(
        row.cache_path
    ).exists()


# ------------------------------------------------------------
# Save final table
# ------------------------------------------------------------

guided_inference_df.to_csv(
    GUIDED_INFERENCE_TABLE_FILE,
    index=False,
)


# ------------------------------------------------------------
# Save run metadata
# ------------------------------------------------------------

guided_run_metadata = {

    "num_images":
        100,

    "num_systems":
        3,

    "num_checkpoint_rows":
        300,

    "total_masks":
        int(
            guided_inference_df[
                "num_retained_masks"
            ].sum()
        ),

    "generated_caches_this_run":
        int(
            num_generated_caches
        ),

    "reused_caches_this_run":
        int(
            num_reused_caches
        ),

    "generated_prompts_this_run":
        int(
            num_generated_prompts
        ),

    "reused_prompts_this_run":
        int(
            num_reused_prompts
        ),

    "wall_time_seconds_this_run":
        float(
            run_elapsed_seconds
        ),

    "mask_cache_directory":
        str(
            E2E_MASK_CACHE_DIR
        ),

    "per_image_table":
        str(
            GUIDED_INFERENCE_TABLE_FILE
        ),

    "test_set_accessed":
        False,
}


with open(
    GUIDED_RUN_METADATA_FILE,
    "w",
) as f:

    json.dump(
        guided_run_metadata,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("\n" + "=" * 110)
print("STEP 5.2 - FULL GUIDED SAM2 INFERENCE COMPLETE")
print("=" * 110)

print(
    "Checkpoint rows          :",
    len(
        guided_inference_df
    ),
)

print(
    "Expected                 : 300"
)

print(
    "Total retained masks     :",
    int(
        guided_inference_df[
            "num_retained_masks"
        ].sum()
    ),
)

print(
    "Expected                 : 15379"
)

print(
    "Generated caches         :",
    num_generated_caches,
)

print(
    "Reused caches            :",
    num_reused_caches,
)

print(
    "Generated prompts        :",
    num_generated_prompts,
)

print(
    "Reused prompts           :",
    num_reused_prompts,
)

print(
    "Current-run wall time    : "
    f"{run_elapsed_seconds / 60:.2f} min"
)

print(
    "Mask filtering           : NO"
)

print(
    "Mask deduplication       : NO"
)

print(
    "GT used during inference : NO"
)

print(
    "Test set accessed        : NO"
)


print("\nPer-system inference summary:")

display(
    guided_inference_df
    .groupby(
        "checkpoint_id",
        observed=True,
    )
    .agg(
        num_images=(
            "image_id",
            "size",
        ),

        total_reference_points=(
            "num_reference_points",
            "sum",
        ),

        total_retained_masks=(
            "num_retained_masks",
            "sum",
        ),

        mean_sam2_score=(
            "mean_sam2_score",
            "mean",
        ),

        mean_prompt_inside_rate=(
            "prompt_inside_mask_rate",
            "mean",
        ),

        total_empty_masks=(
            "num_empty_masks",
            "sum",
        ),
    )
    .reindex(
        CHECKPOINT_IDS
    )
    .round(4)
)

print(
    "\nSaved per-image table:",
    GUIDED_INFERENCE_TABLE_FILE,
)

print(
    "Mask cache:",
    E2E_MASK_CACHE_DIR,
)

print("=" * 110)

STEP 5.2 - FULL GUIDED SAM2 INFERENCE
Images                  : 100
Systems per image       : 3
Total frozen prompts    : 15379
Resumable cache         : YES
GT used by SAM2         : NO
Mask filtering          : NO
Mask deduplication      : NO

Starting / resuming inference...


Guided SAM2 dev100:   0%|          | 0/100 [00:00<?, ?it/s]


STEP 5.2 - FULL GUIDED SAM2 INFERENCE COMPLETE
Checkpoint rows          : 300
Expected                 : 300
Total retained masks     : 15379
Expected                 : 15379
Generated caches         : 300
Reused caches            : 0
Generated prompts        : 15379
Reused prompts           : 0
Current-run wall time    : 2.39 min
Mask filtering           : NO
Mask deduplication       : NO
GT used during inference : NO
Test set accessed        : NO

Per-system inference summary:


,num_images,total_reference_points,total_retained_masks,mean_sam2_score,mean_prompt_inside_rate,total_empty_masks
checkpoint_id,,,,,,
overall_joint,100,5108,5108,0.8144,0.9782,0
sparse_joint,100,4527,4527,0.8159,0.9807,0
dense_recovery,100,5744,5744,0.8166,0.9793,0



Saved per-image table: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/guided_inference_per_image.csv
Mask cache: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/guided_mask_cache


In [16]:
# ============================================================
# Step 5.3 - Validate Frozen Guided-Mask Cache
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm


# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

CACHE_VALIDATION_SUMMARY_FILE = (
    E2E_TABLES_DIR
    / "guided_cache_validation_summary.csv"
)

CACHE_VALIDATION_METADATA_FILE = (
    E2E_METADATA_DIR
    / "guided_cache_validation_metadata.json"
)


# ------------------------------------------------------------
# 1. Rebuild authoritative frozen-source lookup
# ------------------------------------------------------------

source_lookup = {}

for row in checkpoint_input_df.itertuples(
    index=False
):

    key = (
        str(row.checkpoint_id),
        int(row.dataset_index),
    )

    assert key not in source_lookup

    source_lookup[key] = row


assert len(source_lookup) == 300


# ------------------------------------------------------------
# 2. Expected cache paths
# ------------------------------------------------------------

expected_cache_paths = set()

for row in checkpoint_input_df.itertuples(
    index=False
):

    cache_path = guided_mask_cache_path(
        checkpoint_id=
            str(row.checkpoint_id),

        dataset_index=
            int(row.dataset_index),

        image_id=
            str(row.image_id),
    )

    expected_cache_paths.add(
        cache_path.resolve()
    )


assert (
    len(expected_cache_paths)
    == 300
)


# ------------------------------------------------------------
# 3. Actual cache files on disk
# ------------------------------------------------------------

actual_cache_paths = {
    path.resolve()

    for path
    in E2E_MASK_CACHE_DIR.rglob(
        "*.npz"
    )
}


missing_cache_paths = (
    expected_cache_paths
    - actual_cache_paths
)

unexpected_cache_paths = (
    actual_cache_paths
    - expected_cache_paths
)


assert not missing_cache_paths, (
    "Missing expected cache files:\n"
    + "\n".join(
        str(p)
        for p in sorted(
            missing_cache_paths
        )
    )
)

assert not unexpected_cache_paths, (
    "Unexpected cache files found:\n"
    + "\n".join(
        str(p)
        for p in sorted(
            unexpected_cache_paths
        )
    )
)

assert (
    len(actual_cache_paths)
    == 300
)


# ------------------------------------------------------------
# 4. Validate every cache
# ------------------------------------------------------------

validation_records = []

total_masks = 0
total_empty_masks = 0
total_prompt_inside = 0

all_scores = []

total_cache_bytes = 0


for cache_path in tqdm(
    sorted(
        actual_cache_paths
    ),
    total=300,
    desc="Validating guided caches",
):

    total_cache_bytes += (
        cache_path.stat().st_size
    )


    with np.load(
        cache_path,
        allow_pickle=False,
    ) as data:

        # ----------------------------------------------------
        # Required schema
        # ----------------------------------------------------

        missing_fields = (
            set(MASK_CACHE_FIELDS)
            - set(data.files)
        )

        assert not missing_fields, (
            f"Missing cache fields "
            f"{sorted(missing_fields)} "
            f"in {cache_path}"
        )


        # ----------------------------------------------------
        # Identity
        # ----------------------------------------------------

        checkpoint_id = str(
            data[
                "checkpoint_id"
            ].item()
        )

        dataset_index = int(
            data[
                "dataset_index"
            ].item()
        )

        image_id = str(
            data[
                "image_id"
            ].item()
        )


        source_key = (
            checkpoint_id,
            dataset_index,
        )

        assert (
            source_key
            in source_lookup
        )

        source_row = (
            source_lookup[
                source_key
            ]
        )


        assert (
            image_id
            == str(
                source_row.image_id
            )
        )


        # ----------------------------------------------------
        # Dimensions
        # ----------------------------------------------------

        height = int(
            data[
                "height"
            ].item()
        )

        width = int(
            data[
                "width"
            ].item()
        )


        assert (
            height
            == int(
                source_row.height
            )
        )

        assert (
            width
            == int(
                source_row.width
            )
        )


        # ----------------------------------------------------
        # Cached arrays
        # ----------------------------------------------------

        packed_masks = np.asarray(
            data[
                "packed_masks"
            ],
            dtype=np.uint8,
        )

        prompt_points_xy = np.asarray(
            data[
                "prompt_points_xy"
            ],
            dtype=np.float32,
        ).reshape(-1, 2)

        sam2_scores = np.asarray(
            data[
                "sam2_scores"
            ],
            dtype=np.float32,
        ).reshape(-1)

        cached_mask_area_px = np.asarray(
            data[
                "mask_area_px"
            ],
            dtype=np.int64,
        ).reshape(-1)

        cached_prompt_inside = np.asarray(
            data[
                "prompt_inside_mask"
            ],
            dtype=bool,
        ).reshape(-1)


        num_masks = int(
            packed_masks.shape[0]
        )


        # ----------------------------------------------------
        # Frozen prompt identity
        # ----------------------------------------------------

        source_points = np.asarray(
            source_row.pred_points_parsed,
            dtype=np.float32,
        ).reshape(-1, 2)


        assert np.array_equal(
            prompt_points_xy,
            source_points,
        ), (
            "Cached prompts differ from frozen "
            "Group-E prompts:\n"
            f"{checkpoint_id}, "
            f"{image_id}"
        )


        assert (
            num_masks
            == int(
                source_row.num_reference_points
            )
        )

        assert (
            len(prompt_points_xy)
            == num_masks
        )

        assert (
            len(sam2_scores)
            == num_masks
        )

        assert (
            len(cached_mask_area_px)
            == num_masks
        )

        assert (
            len(cached_prompt_inside)
            == num_masks
        )


        # ----------------------------------------------------
        # Restore masks losslessly
        # ----------------------------------------------------

        restored_masks = (
            unpack_binary_masks(
                packed_masks,
                height,
                width,
            )
        )


        assert (
            restored_masks.shape
            ==
            (
                num_masks,
                height,
                width,
            )
        )


        # ----------------------------------------------------
        # Repack and require byte-for-byte identity
        # ----------------------------------------------------

        (
            repacked_masks,
            repacked_height,
            repacked_width,
        ) = pack_binary_masks(
            restored_masks
        )


        assert (
            repacked_height
            == height
        )

        assert (
            repacked_width
            == width
        )

        assert np.array_equal(
            repacked_masks,
            packed_masks,
        ), (
            "Mask pack/unpack/repack mismatch:\n"
            f"{checkpoint_id}, "
            f"{image_id}"
        )


        # ----------------------------------------------------
        # Recompute mask areas
        # ----------------------------------------------------

        if num_masks > 0:

            recomputed_mask_area_px = (
                restored_masks
                .reshape(
                    num_masks,
                    -1,
                )
                .sum(
                    axis=1
                )
                .astype(
                    np.int64
                )
            )

        else:

            recomputed_mask_area_px = (
                np.empty(
                    0,
                    dtype=np.int64,
                )
            )


        assert np.array_equal(
            recomputed_mask_area_px,
            cached_mask_area_px,
        ), (
            "Cached mask areas do not match "
            "restored masks:\n"
            f"{checkpoint_id}, "
            f"{image_id}"
        )


        # ----------------------------------------------------
        # Recompute prompt-inside-mask flags
        # ----------------------------------------------------

        recomputed_prompt_inside = []


        for mask_id, (
            x,
            y,
        ) in enumerate(
            prompt_points_xy
        ):

            px = int(
                np.clip(
                    np.rint(x),
                    0,
                    width - 1,
                )
            )

            py = int(
                np.clip(
                    np.rint(y),
                    0,
                    height - 1,
                )
            )


            recomputed_prompt_inside.append(
                bool(
                    restored_masks[
                        mask_id,
                        py,
                        px,
                    ]
                )
            )


        recomputed_prompt_inside = (
            np.asarray(
                recomputed_prompt_inside,
                dtype=bool,
            )
        )


        assert np.array_equal(
            recomputed_prompt_inside,
            cached_prompt_inside,
        ), (
            "Cached prompt-inside-mask flags "
            "do not match restored masks:\n"
            f"{checkpoint_id}, "
            f"{image_id}"
        )


        # ----------------------------------------------------
        # Numerical checks
        # ----------------------------------------------------

        assert np.isfinite(
            sam2_scores
        ).all()

        assert (
            cached_mask_area_px
            >= 0
        ).all()


        num_empty_masks = int(
            (
                cached_mask_area_px
                == 0
            ).sum()
        )

        num_prompt_inside = int(
            cached_prompt_inside.sum()
        )


        total_masks += (
            num_masks
        )

        total_empty_masks += (
            num_empty_masks
        )

        total_prompt_inside += (
            num_prompt_inside
        )


        if num_masks > 0:

            all_scores.append(
                sam2_scores
            )


        # ----------------------------------------------------
        # Per-cache validation record
        # ----------------------------------------------------

        validation_records.append(
            {
                "checkpoint_id":
                    checkpoint_id,

                "dataset_index":
                    dataset_index,

                "image_id":
                    image_id,

                "num_masks":
                    num_masks,

                "num_empty_masks":
                    num_empty_masks,

                "num_prompt_inside":
                    num_prompt_inside,

                "prompt_inside_rate":
                    (
                        float(
                            num_prompt_inside
                            / num_masks
                        )
                        if num_masks > 0
                        else np.nan
                    ),

                "mean_sam2_score":
                    (
                        float(
                            sam2_scores.mean()
                        )
                        if num_masks > 0
                        else np.nan
                    ),

                "mean_mask_area_px":
                    (
                        float(
                            cached_mask_area_px.mean()
                        )
                        if num_masks > 0
                        else np.nan
                    ),

                "cache_size_bytes":
                    int(
                        cache_path.stat().st_size
                    ),

                "round_trip_pass":
                    True,
            }
        )


# ------------------------------------------------------------
# 5. Build validation dataframe
# ------------------------------------------------------------

guided_cache_validation_df = (
    pd.DataFrame(
        validation_records
    )
)


assert (
    len(
        guided_cache_validation_df
    )
    == 300
)


# ------------------------------------------------------------
# 6. True pooled prompt-level statistics
# ------------------------------------------------------------

if all_scores:

    pooled_scores = np.concatenate(
        all_scores
    )

else:

    pooled_scores = np.empty(
        0,
        dtype=np.float32,
    )


assert (
    len(
        pooled_scores
    )
    == 15379
)


pooled_mean_sam2_score = float(
    pooled_scores.mean()
)

pooled_median_sam2_score = float(
    np.median(
        pooled_scores
    )
)

pooled_prompt_inside_rate = float(
    total_prompt_inside
    / total_masks
)


# ------------------------------------------------------------
# 7. Strong global checks
# ------------------------------------------------------------

assert (
    total_masks
    == 15379
)

assert (
    total_empty_masks
    == 0
)

assert (
    guided_cache_validation_df[
        "round_trip_pass"
    ].all()
)


# ------------------------------------------------------------
# 8. Per-system pooled statistics
# ------------------------------------------------------------

system_summary_records = []


for checkpoint_id in CHECKPOINT_IDS:

    system_rows = (
        checkpoint_input_df[
            checkpoint_input_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
    )


    system_scores = []

    system_num_masks = 0
    system_prompt_inside = 0
    system_empty_masks = 0
    system_cache_bytes = 0


    for row in system_rows.itertuples(
        index=False
    ):

        cache_path = guided_mask_cache_path(
            checkpoint_id=
                checkpoint_id,

            dataset_index=
                int(
                    row.dataset_index
                ),

            image_id=
                str(
                    row.image_id
                ),
        )


        with np.load(
            cache_path,
            allow_pickle=False,
        ) as data:

            scores = np.asarray(
                data[
                    "sam2_scores"
                ],
                dtype=np.float32,
            ).reshape(-1)

            areas = np.asarray(
                data[
                    "mask_area_px"
                ],
                dtype=np.int64,
            ).reshape(-1)

            inside = np.asarray(
                data[
                    "prompt_inside_mask"
                ],
                dtype=bool,
            ).reshape(-1)


        system_num_masks += (
            len(scores)
        )

        system_prompt_inside += int(
            inside.sum()
        )

        system_empty_masks += int(
            (
                areas == 0
            ).sum()
        )

        system_cache_bytes += int(
            cache_path.stat().st_size
        )


        if len(scores) > 0:

            system_scores.append(
                scores
            )


    if system_scores:

        system_scores = (
            np.concatenate(
                system_scores
            )
        )

    else:

        system_scores = (
            np.empty(
                0,
                dtype=np.float32,
            )
        )


    system_summary_records.append(
        {
            "checkpoint_id":
                checkpoint_id,

            "num_images":
                100,

            "num_masks":
                int(
                    system_num_masks
                ),

            "mean_sam2_score":
                (
                    float(
                        system_scores.mean()
                    )
                    if len(
                        system_scores
                    ) > 0
                    else np.nan
                ),

            "median_sam2_score":
                (
                    float(
                        np.median(
                            system_scores
                        )
                    )
                    if len(
                        system_scores
                    ) > 0
                    else np.nan
                ),

            "prompt_inside_rate":
                (
                    float(
                        system_prompt_inside
                        / system_num_masks
                    )
                    if system_num_masks > 0
                    else np.nan
                ),

            "num_empty_masks":
                int(
                    system_empty_masks
                ),

            "cache_size_mib":
                float(
                    system_cache_bytes
                    / (1024 ** 2)
                ),
        }
    )


guided_cache_system_summary_df = (
    pd.DataFrame(
        system_summary_records
    )
)


# ------------------------------------------------------------
# 9. Actual cache size
# ------------------------------------------------------------

total_cache_mib = float(
    total_cache_bytes
    / (1024 ** 2)
)

total_cache_gib = float(
    total_cache_bytes
    / (1024 ** 3)
)


# ------------------------------------------------------------
# 10. Save validation artefacts
# ------------------------------------------------------------

guided_cache_system_summary_df.to_csv(
    CACHE_VALIDATION_SUMMARY_FILE,
    index=False,
)


cache_validation_metadata = {

    "num_expected_caches":
        300,

    "num_actual_caches":
        int(
            len(
                actual_cache_paths
            )
        ),

    "total_masks":
        int(
            total_masks
        ),

    "total_empty_masks":
        int(
            total_empty_masks
        ),

    "pooled_mean_sam2_score":
        pooled_mean_sam2_score,

    "pooled_median_sam2_score":
        pooled_median_sam2_score,

    "pooled_prompt_inside_rate":
        pooled_prompt_inside_rate,

    "total_cache_bytes":
        int(
            total_cache_bytes
        ),

    "total_cache_mib":
        total_cache_mib,

    "mask_pack_round_trip":
        True,

    "cached_prompts_match_frozen_group_e":
        True,

    "cached_mask_areas_recomputed":
        True,

    "cached_prompt_inside_flags_recomputed":
        True,

    "test_set_accessed":
        False,
}


with open(
    CACHE_VALIDATION_METADATA_FILE,
    "w",
) as f:

    json.dump(
        cache_validation_metadata,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 11. Confirmation
# ------------------------------------------------------------

print("=" * 110)
print("STEP 5.3 - GUIDED MASK-CACHE INTEGRITY VALIDATION")
print("=" * 110)

print(
    "Expected caches             : 300"
)

print(
    "Actual caches               :",
    len(
        actual_cache_paths
    ),
)

print(
    "Missing caches              :",
    len(
        missing_cache_paths
    ),
)

print(
    "Unexpected caches           :",
    len(
        unexpected_cache_paths
    ),
)

print(
    "Total restored masks        :",
    total_masks,
)

print(
    "Expected masks              : 15379"
)

print(
    "Empty masks                 :",
    total_empty_masks,
)

print(
    "Mask unpack/repack          : PASS"
)

print(
    "Frozen prompt identity      : PASS"
)

print(
    "Mask-area recomputation     : PASS"
)

print(
    "Prompt-inside recomputation : PASS"
)

print(
    "\nTrue pooled SAM2 score:"
)

print(
    f"  Mean                      : "
    f"{pooled_mean_sam2_score:.4f}"
)

print(
    f"  Median                    : "
    f"{pooled_median_sam2_score:.4f}"
)

print(
    "\nTrue pooled prompt-inside rate:"
)

print(
    f"  {pooled_prompt_inside_rate:.4%}"
)

print(
    "\nActual compressed cache size:"
)

print(
    f"  {total_cache_mib:.2f} MiB "
    f"({total_cache_gib:.3f} GiB)"
)


print(
    "\nPer-system pooled validation summary:"
)

display(
    guided_cache_system_summary_df
    .round(4)
)


print(
    "\nSaved validation summary:",
    CACHE_VALIDATION_SUMMARY_FILE,
)

print(
    "Saved validation metadata:",
    CACHE_VALIDATION_METADATA_FILE,
)

print(
    "\nTest set accessed           : NO"
)

print("=" * 110)

Validating guided caches:   0%|          | 0/300 [00:00<?, ?it/s]

STEP 5.3 - GUIDED MASK-CACHE INTEGRITY VALIDATION
Expected caches             : 300
Actual caches               : 300
Missing caches              : 0
Unexpected caches           : 0
Total restored masks        : 15379
Expected masks              : 15379
Empty masks                 : 0
Mask unpack/repack          : PASS
Frozen prompt identity      : PASS
Mask-area recomputation     : PASS
Prompt-inside recomputation : PASS

True pooled SAM2 score:
  Mean                      : 0.8224
  Median                    : 0.8897

True pooled prompt-inside rate:
  98.4849%

Actual compressed cache size:
  6.32 MiB (0.006 GiB)

Per-system pooled validation summary:


,checkpoint_id,num_images,num_masks,mean_sam2_score,median_sam2_score,prompt_inside_rate,num_empty_masks,cache_size_mib
0,overall_joint,100,5108,0.8219,0.8907,0.9832,0,1.8761
1,sparse_joint,100,4527,0.8218,0.8883,0.9854,0,1.6428
2,dense_recovery,100,5744,0.8232,0.8903,0.9859,0,2.8021



Saved validation summary: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/guided_cache_validation_summary.csv
Saved validation metadata: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/metadata/guided_cache_validation_metadata.json

Test set accessed           : NO


In [17]:
# ============================================================
# Step 6.1 - Compute Mask <-> GT-Point Correspondence
# ============================================================

import json
import numpy as np
import pandas as pd

from tqdm.auto import tqdm


# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

MASK_GT_CORRESPONDENCE_FILE = (
    E2E_TABLES_DIR
    / "guided_mask_gt_correspondence.csv"
)

GT_MASK_COVERAGE_FILE = (
    E2E_TABLES_DIR
    / "guided_gt_mask_coverage.csv"
)

MASK_DIAGNOSTICS_PER_IMAGE_FILE = (
    E2E_TABLES_DIR
    / "guided_mask_diagnostics_per_image.csv"
)

MASK_DIAGNOSTICS_BY_SYSTEM_FILE = (
    E2E_TABLES_DIR
    / "guided_mask_diagnostics_by_system.csv"
)

MASK_CORRESPONDENCE_PROTOCOL_FILE = (
    E2E_METADATA_DIR
    / "guided_mask_correspondence_protocol.json"
)


# ------------------------------------------------------------
# 1. Freeze correspondence protocol
# ------------------------------------------------------------

mask_correspondence_protocol = {

    "evaluation_unit":
        "FSC-147 target GT points",

    "gt_coordinate_policy":
        (
            "Canonical continuous FSC-147 GT coordinates "
            "are preserved unchanged."
        ),

    "discrete_mask_lookup":
        (
            "For mask membership only, each GT point is "
            "mapped to the nearest valid image pixel using "
            "np.rint() followed by clipping to image bounds."
        ),

    "zero_gt_mask":
        (
            "A retained SAM2 mask containing zero target "
            "GT points."
        ),

    "one_gt_mask":
        (
            "A retained SAM2 mask containing exactly one "
            "target GT point."
        ),

    "merged_gt_mask":
        (
            "A retained SAM2 mask containing two or more "
            "target GT points."
        ),

    "missed_gt":
        (
            "A target GT point covered by zero retained "
            "SAM2 masks."
        ),

    "single_covered_gt":
        (
            "A target GT point covered by exactly one "
            "retained SAM2 mask."
        ),

    "duplicate_covered_gt":
        (
            "A target GT point covered by two or more "
            "retained SAM2 masks."
        ),

    "strict_one_to_one":
        (
            "A mask contains exactly one GT point and that "
            "GT point is covered by exactly one mask."
        ),

    "boundary_gt_handling":
        (
            "Known FSC-147 strict-boundary GT points are "
            "retained in the primary analysis and explicitly "
            "flagged for later sensitivity analysis."
        ),

    "gt_used_during_inference":
        False,

    "gt_used_here_for_evaluation_only":
        True,

    "test_set_accessed":
        False,
}


with open(
    MASK_CORRESPONDENCE_PROTOCOL_FILE,
    "w",
) as f:

    json.dump(
        mask_correspondence_protocol,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 2. Known strict-boundary GT-point lookup
#
# Key:
#   (dataset_index, gt_point_index)
# ------------------------------------------------------------

boundary_gt_keys = set()


if len(gt_boundary_df) > 0:

    for row in gt_boundary_df.itertuples(
        index=False
    ):

        boundary_gt_keys.add(
            (
                int(row.dataset_index),
                int(row.point_index),
            )
        )


assert (
    len(boundary_gt_keys)
    == 12
)


# ------------------------------------------------------------
# 3. Output records
# ------------------------------------------------------------

mask_records = []
gt_records = []
image_summary_records = []


# ------------------------------------------------------------
# 4. Process all 300 frozen system-image cases
# ------------------------------------------------------------

for row in tqdm(
    checkpoint_input_df.itertuples(
        index=False
    ),
    total=len(
        checkpoint_input_df
    ),
    desc="Mask-GT correspondence",
):

    checkpoint_id = str(
        row.checkpoint_id
    )

    dataset_index = int(
        row.dataset_index
    )

    image_id = str(
        row.image_id
    )

    category = str(
        row.category
    )

    density_stratum = str(
        row.density_stratum
    )

    height = int(
        row.height
    )

    width = int(
        row.width
    )


    # --------------------------------------------------------
    # Canonical GT points
    # --------------------------------------------------------

    gt_points = np.asarray(
        row.gt_points_xy,
        dtype=np.float64,
    ).reshape(-1, 2)

    gt_count = int(
        row.gt_count
    )


    assert (
        len(gt_points)
        == gt_count
    )


    # --------------------------------------------------------
    # Map continuous GT coordinates to nearest valid pixel
    # for discrete mask membership only.
    # --------------------------------------------------------

    gt_pixel_x = np.clip(
        np.rint(
            gt_points[:, 0]
        ),
        0,
        width - 1,
    ).astype(
        np.int64
    )

    gt_pixel_y = np.clip(
        np.rint(
            gt_points[:, 1]
        ),
        0,
        height - 1,
    ).astype(
        np.int64
    )


    gt_boundary_flags = np.asarray(
        [
            (
                dataset_index,
                gt_index,
            )
            in boundary_gt_keys

            for gt_index
            in range(
                gt_count
            )
        ],
        dtype=bool,
    )


    # --------------------------------------------------------
    # Load frozen guided SAM2 cache
    # --------------------------------------------------------

    cache_path = guided_mask_cache_path(
        checkpoint_id=
            checkpoint_id,

        dataset_index=
            dataset_index,

        image_id=
            image_id,
    )


    assert cache_path.exists()


    with np.load(
        cache_path,
        allow_pickle=False,
    ) as data:

        packed_masks = np.asarray(
            data[
                "packed_masks"
            ],
            dtype=np.uint8,
        )

        prompt_points_xy = np.asarray(
            data[
                "prompt_points_xy"
            ],
            dtype=np.float32,
        ).reshape(-1, 2)

        sam2_scores = np.asarray(
            data[
                "sam2_scores"
            ],
            dtype=np.float32,
        ).reshape(-1)

        mask_area_px = np.asarray(
            data[
                "mask_area_px"
            ],
            dtype=np.int64,
        ).reshape(-1)

        prompt_inside_mask = np.asarray(
            data[
                "prompt_inside_mask"
            ],
            dtype=bool,
        ).reshape(-1)


    # --------------------------------------------------------
    # Restore masks
    # --------------------------------------------------------

    restored_masks = (
        unpack_binary_masks(
            packed_masks,
            height,
            width,
        )
    )


    num_masks = int(
        restored_masks.shape[0]
    )


    assert (
        num_masks
        == int(
            row.num_reference_points
        )
    )

    assert (
        len(prompt_points_xy)
        == num_masks
    )

    assert (
        len(sam2_scores)
        == num_masks
    )

    assert (
        len(mask_area_px)
        == num_masks
    )

    assert (
        len(prompt_inside_mask)
        == num_masks
    )


    # --------------------------------------------------------
    # Mask x GT membership matrix
    #
    # Shape:
    #   num_masks x gt_count
    #
    # membership[m, g] == True means:
    # GT point g lies inside SAM2 mask m.
    # --------------------------------------------------------

    membership = (
        restored_masks[
            :,
            gt_pixel_y,
            gt_pixel_x,
        ]
    )


    assert (
        membership.shape
        ==
        (
            num_masks,
            gt_count,
        )
    )


    # --------------------------------------------------------
    # Number of GT points contained by each mask
    # --------------------------------------------------------

    gt_per_mask = (
        membership
        .sum(
            axis=1
        )
        .astype(
            np.int64
        )
    )


    # --------------------------------------------------------
    # Number of masks covering each GT point
    # --------------------------------------------------------

    masks_per_gt = (
        membership
        .sum(
            axis=0
        )
        .astype(
            np.int64
        )
    )


    # --------------------------------------------------------
    # Strict one-to-one correspondence
    #
    # Mask contains exactly one GT AND
    # that GT is contained by exactly one mask.
    # --------------------------------------------------------

    strict_one_to_one_mask = np.zeros(
        num_masks,
        dtype=bool,
    )

    strict_one_to_one_gt = np.zeros(
        gt_count,
        dtype=bool,
    )


    one_gt_mask_indices = np.where(
        gt_per_mask == 1
    )[0]


    for mask_index in one_gt_mask_indices:

        gt_indices = np.flatnonzero(
            membership[
                mask_index
            ]
        )

        assert (
            len(gt_indices)
            == 1
        )

        gt_index = int(
            gt_indices[0]
        )


        if (
            masks_per_gt[
                gt_index
            ]
            == 1
        ):

            strict_one_to_one_mask[
                mask_index
            ] = True

            strict_one_to_one_gt[
                gt_index
            ] = True


    # --------------------------------------------------------
    # Boundary-GT contribution per mask
    # --------------------------------------------------------

    if gt_boundary_flags.any():

        boundary_gt_per_mask = (
            membership[
                :,
                gt_boundary_flags
            ]
            .sum(
                axis=1
            )
            .astype(
                np.int64
            )
        )

    else:

        boundary_gt_per_mask = np.zeros(
            num_masks,
            dtype=np.int64,
        )


    # --------------------------------------------------------
    # Per-mask correspondence records
    # --------------------------------------------------------

    for mask_index in range(
        num_masks
    ):

        num_gt_in_mask = int(
            gt_per_mask[
                mask_index
            ]
        )


        if num_gt_in_mask == 0:

            mask_gt_class = (
                "zero_gt"
            )

        elif num_gt_in_mask == 1:

            mask_gt_class = (
                "one_gt"
            )

        else:

            mask_gt_class = (
                "merged_gt"
            )


        contained_gt_indices = (
            np.flatnonzero(
                membership[
                    mask_index
                ]
            )
            .astype(int)
            .tolist()
        )


        mask_records.append(
            {
                "checkpoint_id":
                    checkpoint_id,

                "dataset_index":
                    dataset_index,

                "image_id":
                    image_id,

                "category":
                    category,

                "density_stratum":
                    density_stratum,

                "mask_index":
                    int(
                        mask_index
                    ),

                "prompt_x":
                    float(
                        prompt_points_xy[
                            mask_index,
                            0,
                        ]
                    ),

                "prompt_y":
                    float(
                        prompt_points_xy[
                            mask_index,
                            1,
                        ]
                    ),

                "sam2_score":
                    float(
                        sam2_scores[
                            mask_index
                        ]
                    ),

                "mask_area_px":
                    int(
                        mask_area_px[
                            mask_index
                        ]
                    ),

                "prompt_inside_mask":
                    bool(
                        prompt_inside_mask[
                            mask_index
                        ]
                    ),

                "num_gt_points_in_mask":
                    num_gt_in_mask,

                "mask_gt_class":
                    mask_gt_class,

                "strict_one_to_one":
                    bool(
                        strict_one_to_one_mask[
                            mask_index
                        ]
                    ),

                "contains_boundary_gt":
                    bool(
                        boundary_gt_per_mask[
                            mask_index
                        ]
                        > 0
                    ),

                "num_boundary_gt_points_in_mask":
                    int(
                        boundary_gt_per_mask[
                            mask_index
                        ]
                    ),

                "gt_indices":
                    json.dumps(
                        contained_gt_indices
                    ),
            }
        )


    # --------------------------------------------------------
    # Per-GT coverage records
    # --------------------------------------------------------

    for gt_index in range(
        gt_count
    ):

        num_covering_masks = int(
            masks_per_gt[
                gt_index
            ]
        )


        if num_covering_masks == 0:

            gt_coverage_class = (
                "missed"
            )

        elif num_covering_masks == 1:

            gt_coverage_class = (
                "single_covered"
            )

        else:

            gt_coverage_class = (
                "duplicate_covered"
            )


        covering_mask_indices = (
            np.flatnonzero(
                membership[
                    :,
                    gt_index
                ]
            )
            .astype(int)
            .tolist()
        )


        gt_records.append(
            {
                "checkpoint_id":
                    checkpoint_id,

                "dataset_index":
                    dataset_index,

                "image_id":
                    image_id,

                "category":
                    category,

                "density_stratum":
                    density_stratum,

                "gt_index":
                    int(
                        gt_index
                    ),

                "gt_x":
                    float(
                        gt_points[
                            gt_index,
                            0
                        ]
                    ),

                "gt_y":
                    float(
                        gt_points[
                            gt_index,
                            1
                        ]
                    ),

                "pixel_x":
                    int(
                        gt_pixel_x[
                            gt_index
                        ]
                    ),

                "pixel_y":
                    int(
                        gt_pixel_y[
                            gt_index
                        ]
                    ),

                "is_boundary_gt":
                    bool(
                        gt_boundary_flags[
                            gt_index
                        ]
                    ),

                "num_covering_masks":
                    num_covering_masks,

                "gt_coverage_class":
                    gt_coverage_class,

                "strict_one_to_one":
                    bool(
                        strict_one_to_one_gt[
                            gt_index
                        ]
                    ),

                "mask_indices":
                    json.dumps(
                        covering_mask_indices
                    ),
            }
        )


    # --------------------------------------------------------
    # Per-image summary
    # --------------------------------------------------------

    num_zero_gt_masks = int(
        (
            gt_per_mask == 0
        ).sum()
    )

    num_one_gt_masks = int(
        (
            gt_per_mask == 1
        ).sum()
    )

    num_merged_gt_masks = int(
        (
            gt_per_mask >= 2
        ).sum()
    )

    num_strict_one_to_one_masks = int(
        strict_one_to_one_mask.sum()
    )


    num_missed_gt = int(
        (
            masks_per_gt == 0
        ).sum()
    )

    num_single_covered_gt = int(
        (
            masks_per_gt == 1
        ).sum()
    )

    num_duplicate_covered_gt = int(
        (
            masks_per_gt >= 2
        ).sum()
    )

    num_strict_one_to_one_gt = int(
        strict_one_to_one_gt.sum()
    )


    num_boundary_affected_masks = int(
        (
            boundary_gt_per_mask
            > 0
        ).sum()
    )


    image_summary_records.append(
        {
            "checkpoint_id":
                checkpoint_id,

            "dataset_index":
                dataset_index,

            "image_id":
                image_id,

            "category":
                category,

            "density_stratum":
                density_stratum,

            "gt_count":
                gt_count,

            "num_masks":
                num_masks,

            "zero_gt_masks":
                num_zero_gt_masks,

            "one_gt_masks":
                num_one_gt_masks,

            "merged_gt_masks":
                num_merged_gt_masks,

            "strict_one_to_one_masks":
                num_strict_one_to_one_masks,

            "supported_mask_rate":
                (
                    float(
                        (
                            num_one_gt_masks
                            +
                            num_merged_gt_masks
                        )
                        / num_masks
                    )
                    if num_masks > 0
                    else np.nan
                ),

            "one_gt_mask_rate":
                (
                    float(
                        num_one_gt_masks
                        / num_masks
                    )
                    if num_masks > 0
                    else np.nan
                ),

            "merged_mask_rate":
                (
                    float(
                        num_merged_gt_masks
                        / num_masks
                    )
                    if num_masks > 0
                    else np.nan
                ),

            "missed_gt":
                num_missed_gt,

            "single_covered_gt":
                num_single_covered_gt,

            "duplicate_covered_gt":
                num_duplicate_covered_gt,

            "covered_gt":
                int(
                    gt_count
                    - num_missed_gt
                ),

            "gt_coverage_rate":
                float(
                    (
                        gt_count
                        - num_missed_gt
                    )
                    / gt_count
                ),

            "duplicate_gt_rate":
                float(
                    num_duplicate_covered_gt
                    / gt_count
                ),

            "strict_one_to_one_gt":
                num_strict_one_to_one_gt,

            "strict_one_to_one_gt_rate":
                float(
                    num_strict_one_to_one_gt
                    / gt_count
                ),

            "boundary_gt_points":
                int(
                    gt_boundary_flags.sum()
                ),

            "boundary_affected_masks":
                num_boundary_affected_masks,
        }
    )


# ------------------------------------------------------------
# 5. Build output tables
# ------------------------------------------------------------

mask_gt_correspondence_df = (
    pd.DataFrame(
        mask_records
    )
)

gt_mask_coverage_df = (
    pd.DataFrame(
        gt_records
    )
)

mask_diagnostics_per_image_df = (
    pd.DataFrame(
        image_summary_records
    )
)


# ------------------------------------------------------------
# 6. Strong integrity checks
# ------------------------------------------------------------

assert (
    len(
        mask_gt_correspondence_df
    )
    == 15379
)

assert (
    len(
        gt_mask_coverage_df
    )
    == (
        10045
        * 3
    )
)

assert (
    len(
        mask_diagnostics_per_image_df
    )
    == 300
)


# Every mask belongs to exactly one class.
assert (
    (
        mask_diagnostics_per_image_df[
            "zero_gt_masks"
        ]
        +
        mask_diagnostics_per_image_df[
            "one_gt_masks"
        ]
        +
        mask_diagnostics_per_image_df[
            "merged_gt_masks"
        ]
    )
    ==
    mask_diagnostics_per_image_df[
        "num_masks"
    ]
).all()


# Every GT point belongs to exactly one coverage class.
assert (
    (
        mask_diagnostics_per_image_df[
            "missed_gt"
        ]
        +
        mask_diagnostics_per_image_df[
            "single_covered_gt"
        ]
        +
        mask_diagnostics_per_image_df[
            "duplicate_covered_gt"
        ]
    )
    ==
    mask_diagnostics_per_image_df[
        "gt_count"
    ]
).all()


# Each system must still contain all canonical GT objects.
for checkpoint_id in CHECKPOINT_IDS:

    system_df = (
        mask_diagnostics_per_image_df[
            mask_diagnostics_per_image_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
    )

    assert len(
        system_df
    ) == 100

    assert (
        int(
            system_df[
                "gt_count"
            ].sum()
        )
        == 10045
    )


# Strict one-to-one correspondence is symmetric:
# number of such masks == number of such GT points.
for checkpoint_id in CHECKPOINT_IDS:

    system_mask_count = int(
        mask_gt_correspondence_df[
            (
                mask_gt_correspondence_df[
                    "checkpoint_id"
                ]
                == checkpoint_id
            )
            &
            (
                mask_gt_correspondence_df[
                    "strict_one_to_one"
                ]
            )
        ].shape[0]
    )

    system_gt_count = int(
        gt_mask_coverage_df[
            (
                gt_mask_coverage_df[
                    "checkpoint_id"
                ]
                == checkpoint_id
            )
            &
            (
                gt_mask_coverage_df[
                    "strict_one_to_one"
                ]
            )
        ].shape[0]
    )

    assert (
        system_mask_count
        == system_gt_count
    )


# ------------------------------------------------------------
# 7. Aggregate true pooled diagnostics by system
# ------------------------------------------------------------

system_summary_records = []


for checkpoint_id in CHECKPOINT_IDS:

    mask_df = (
        mask_gt_correspondence_df[
            mask_gt_correspondence_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
    )

    gt_df = (
        gt_mask_coverage_df[
            gt_mask_coverage_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
    )


    total_masks = len(
        mask_df
    )

    total_gt = len(
        gt_df
    )


    zero_gt_masks = int(
        (
            mask_df[
                "mask_gt_class"
            ]
            == "zero_gt"
        ).sum()
    )

    one_gt_masks = int(
        (
            mask_df[
                "mask_gt_class"
            ]
            == "one_gt"
        ).sum()
    )

    merged_gt_masks = int(
        (
            mask_df[
                "mask_gt_class"
            ]
            == "merged_gt"
        ).sum()
    )

    strict_masks = int(
        mask_df[
            "strict_one_to_one"
        ].sum()
    )


    missed_gt = int(
        (
            gt_df[
                "gt_coverage_class"
            ]
            == "missed"
        ).sum()
    )

    single_covered_gt = int(
        (
            gt_df[
                "gt_coverage_class"
            ]
            == "single_covered"
        ).sum()
    )

    duplicate_covered_gt = int(
        (
            gt_df[
                "gt_coverage_class"
            ]
            == "duplicate_covered"
        ).sum()
    )

    strict_gt = int(
        gt_df[
            "strict_one_to_one"
        ].sum()
    )


    assert (
        strict_masks
        == strict_gt
    )


    system_summary_records.append(
        {
            "checkpoint_id":
                checkpoint_id,

            "num_images":
                100,

            "total_masks":
                int(
                    total_masks
                ),

            "zero_gt_masks":
                zero_gt_masks,

            "one_gt_masks":
                one_gt_masks,

            "merged_gt_masks":
                merged_gt_masks,

            "supported_mask_rate":
                float(
                    (
                        one_gt_masks
                        +
                        merged_gt_masks
                    )
                    / total_masks
                ),

            "one_gt_mask_rate":
                float(
                    one_gt_masks
                    / total_masks
                ),

            "merged_mask_rate":
                float(
                    merged_gt_masks
                    / total_masks
                ),

            "strict_one_to_one":
                strict_masks,

            "strict_one_to_one_mask_rate":
                float(
                    strict_masks
                    / total_masks
                ),

            "total_gt":
                int(
                    total_gt
                ),

            "covered_gt":
                int(
                    total_gt
                    - missed_gt
                ),

            "missed_gt":
                missed_gt,

            "single_covered_gt":
                single_covered_gt,

            "duplicate_covered_gt":
                duplicate_covered_gt,

            "gt_coverage_rate":
                float(
                    (
                        total_gt
                        - missed_gt
                    )
                    / total_gt
                ),

            "duplicate_gt_rate":
                float(
                    duplicate_covered_gt
                    / total_gt
                ),

            "strict_one_to_one_gt":
                strict_gt,

            "strict_one_to_one_gt_rate":
                float(
                    strict_gt
                    / total_gt
                ),

            "boundary_gt_points":
                int(
                    gt_df[
                        "is_boundary_gt"
                    ].sum()
                ),

            "boundary_affected_masks":
                int(
                    mask_df[
                        "contains_boundary_gt"
                    ].sum()
                ),
        }
    )


mask_diagnostics_by_system_df = (
    pd.DataFrame(
        system_summary_records
    )
)


# ------------------------------------------------------------
# 8. Save correspondence artefacts
# ------------------------------------------------------------

mask_gt_correspondence_df.to_csv(
    MASK_GT_CORRESPONDENCE_FILE,
    index=False,
)

gt_mask_coverage_df.to_csv(
    GT_MASK_COVERAGE_FILE,
    index=False,
)

mask_diagnostics_per_image_df.to_csv(
    MASK_DIAGNOSTICS_PER_IMAGE_FILE,
    index=False,
)

mask_diagnostics_by_system_df.to_csv(
    MASK_DIAGNOSTICS_BY_SYSTEM_FILE,
    index=False,
)


# ------------------------------------------------------------
# 9. Confirmation
# ------------------------------------------------------------

print("=" * 115)
print("STEP 6.1 - MASK <-> GT-POINT CORRESPONDENCE")
print("=" * 115)

print(
    "Mask records                 :",
    len(
        mask_gt_correspondence_df
    ),
)

print(
    "Expected                     : 15379"
)

print(
    "GT coverage records          :",
    len(
        gt_mask_coverage_df
    ),
)

print(
    "Expected                     : 30135"
)

print(
    "System-image summaries       :",
    len(
        mask_diagnostics_per_image_df
    ),
)

print(
    "Expected                     : 300"
)

print(
    "Canonical GT modified        : NO"
)

print(
    "Boundary GT retained         : YES"
)

print(
    "Discrete lookup              : rint + clip"
)

print(
    "GT used during inference     : NO"
)

print(
    "Test set accessed            : NO"
)


print(
    "\nTrue pooled correspondence diagnostics:"
)

display(
    mask_diagnostics_by_system_df[
        [
            "checkpoint_id",
            "total_masks",
            "zero_gt_masks",
            "one_gt_masks",
            "merged_gt_masks",
            "supported_mask_rate",
            "strict_one_to_one",
            "total_gt",
            "covered_gt",
            "missed_gt",
            "duplicate_covered_gt",
            "gt_coverage_rate",
            "duplicate_gt_rate",
            "strict_one_to_one_gt_rate",
            "boundary_affected_masks",
        ]
    ]
    .round(4)
)


print(
    "\nSaved mask correspondence:",
    MASK_GT_CORRESPONDENCE_FILE,
)

print(
    "Saved GT coverage:",
    GT_MASK_COVERAGE_FILE,
)

print(
    "Saved per-image diagnostics:",
    MASK_DIAGNOSTICS_PER_IMAGE_FILE,
)

print(
    "Saved system diagnostics:",
    MASK_DIAGNOSTICS_BY_SYSTEM_FILE,
)

print(
    "Saved protocol:",
    MASK_CORRESPONDENCE_PROTOCOL_FILE,
)

print("=" * 115)

Mask-GT correspondence:   0%|          | 0/300 [00:00<?, ?it/s]

STEP 6.1 - MASK <-> GT-POINT CORRESPONDENCE
Mask records                 : 15379
Expected                     : 15379
GT coverage records          : 30135
Expected                     : 30135
System-image summaries       : 300
Expected                     : 300
Canonical GT modified        : NO
Boundary GT retained         : YES
Discrete lookup              : rint + clip
GT used during inference     : NO
Test set accessed            : NO

True pooled correspondence diagnostics:


,checkpoint_id,total_masks,zero_gt_masks,one_gt_masks,merged_gt_masks,supported_mask_rate,strict_one_to_one,total_gt,covered_gt,missed_gt,duplicate_covered_gt,gt_coverage_rate,duplicate_gt_rate,strict_one_to_one_gt_rate,boundary_affected_masks
0,overall_joint,5108,696,3856,556,0.8637,710,10045,6753,3292,5407,0.6723,0.5383,0.0707,1
1,sparse_joint,4527,629,3413,485,0.8611,723,10045,6525,3520,5102,0.6496,0.5079,0.0720,0
2,dense_recovery,5744,1025,3977,742,0.8216,1035,10045,8238,1807,6534,0.8201,0.6505,0.1030,0



Saved mask correspondence: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/guided_mask_gt_correspondence.csv
Saved GT coverage: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/guided_gt_mask_coverage.csv
Saved per-image diagnostics: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/guided_mask_diagnostics_per_image.csv
Saved system diagnostics: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/guided_mask_diagnostics_by_system.csv
Saved protocol: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/metadata/guided_mask_correspondence_protocol.json


In [18]:
# ============================================================
# Step 6.2 - Mask Correspondence Diagnostics by Density Stratum
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Output file
# ------------------------------------------------------------

MASK_DIAGNOSTICS_BY_DENSITY_FILE = (
    E2E_TABLES_DIR
    / "guided_mask_diagnostics_by_density.csv"
)


# ------------------------------------------------------------
# Frozen density order
# ------------------------------------------------------------

DENSITY_ORDER = [
    "S1_0_20",
    "S2_20_40",
    "S3_40_60",
    "S4_60_80",
    "S5_80_95",
    "S6_95_100",
]


# ------------------------------------------------------------
# Build true pooled diagnostics for each
# checkpoint × density stratum
# ------------------------------------------------------------

density_summary_records = []


for checkpoint_id in CHECKPOINT_IDS:

    for density_stratum in DENSITY_ORDER:

        # ----------------------------------------------------
        # Per-image subset
        # ----------------------------------------------------

        image_df = (
            mask_diagnostics_per_image_df[
                (
                    mask_diagnostics_per_image_df[
                        "checkpoint_id"
                    ]
                    == checkpoint_id
                )
                &
                (
                    mask_diagnostics_per_image_df[
                        "density_stratum"
                    ]
                    == density_stratum
                )
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Per-mask subset
        # ----------------------------------------------------

        mask_df = (
            mask_gt_correspondence_df[
                (
                    mask_gt_correspondence_df[
                        "checkpoint_id"
                    ]
                    == checkpoint_id
                )
                &
                (
                    mask_gt_correspondence_df[
                        "density_stratum"
                    ]
                    == density_stratum
                )
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Per-GT subset
        # ----------------------------------------------------

        gt_df = (
            gt_mask_coverage_df[
                (
                    gt_mask_coverage_df[
                        "checkpoint_id"
                    ]
                    == checkpoint_id
                )
                &
                (
                    gt_mask_coverage_df[
                        "density_stratum"
                    ]
                    == density_stratum
                )
            ]
            .copy()
        )


        num_images = int(
            len(
                image_df
            )
        )

        total_masks = int(
            len(
                mask_df
            )
        )

        total_gt = int(
            len(
                gt_df
            )
        )


        # ----------------------------------------------------
        # Mask-side diagnostics
        # ----------------------------------------------------

        zero_gt_masks = int(
            (
                mask_df[
                    "mask_gt_class"
                ]
                == "zero_gt"
            ).sum()
        )

        one_gt_masks = int(
            (
                mask_df[
                    "mask_gt_class"
                ]
                == "one_gt"
            ).sum()
        )

        merged_gt_masks = int(
            (
                mask_df[
                    "mask_gt_class"
                ]
                == "merged_gt"
            ).sum()
        )

        strict_one_to_one_masks = int(
            mask_df[
                "strict_one_to_one"
            ].sum()
        )


        # ----------------------------------------------------
        # GT-side diagnostics
        # ----------------------------------------------------

        missed_gt = int(
            (
                gt_df[
                    "gt_coverage_class"
                ]
                == "missed"
            ).sum()
        )

        single_covered_gt = int(
            (
                gt_df[
                    "gt_coverage_class"
                ]
                == "single_covered"
            ).sum()
        )

        duplicate_covered_gt = int(
            (
                gt_df[
                    "gt_coverage_class"
                ]
                == "duplicate_covered"
            ).sum()
        )

        strict_one_to_one_gt = int(
            gt_df[
                "strict_one_to_one"
            ].sum()
        )


        # ----------------------------------------------------
        # Integrity checks
        # ----------------------------------------------------

        assert (
            zero_gt_masks
            + one_gt_masks
            + merged_gt_masks
            == total_masks
        )

        assert (
            missed_gt
            + single_covered_gt
            + duplicate_covered_gt
            == total_gt
        )

        assert (
            strict_one_to_one_masks
            == strict_one_to_one_gt
        )


        # ----------------------------------------------------
        # Boundary diagnostics
        # ----------------------------------------------------

        boundary_gt_points = int(
            gt_df[
                "is_boundary_gt"
            ].sum()
        )

        boundary_affected_masks = int(
            mask_df[
                "contains_boundary_gt"
            ].sum()
        )


        # ----------------------------------------------------
        # Store pooled summary
        # ----------------------------------------------------

        density_summary_records.append(
            {
                "checkpoint_id":
                    checkpoint_id,

                "density_stratum":
                    density_stratum,

                "num_images":
                    num_images,

                "total_masks":
                    total_masks,

                "total_gt":
                    total_gt,

                # ------------------------------
                # Mask-side
                # ------------------------------

                "zero_gt_masks":
                    zero_gt_masks,

                "one_gt_masks":
                    one_gt_masks,

                "merged_gt_masks":
                    merged_gt_masks,

                "zero_gt_mask_rate":
                    (
                        float(
                            zero_gt_masks
                            / total_masks
                        )
                        if total_masks > 0
                        else np.nan
                    ),

                "one_gt_mask_rate":
                    (
                        float(
                            one_gt_masks
                            / total_masks
                        )
                        if total_masks > 0
                        else np.nan
                    ),

                "merged_mask_rate":
                    (
                        float(
                            merged_gt_masks
                            / total_masks
                        )
                        if total_masks > 0
                        else np.nan
                    ),

                "supported_mask_rate":
                    (
                        float(
                            (
                                one_gt_masks
                                + merged_gt_masks
                            )
                            / total_masks
                        )
                        if total_masks > 0
                        else np.nan
                    ),

                # ------------------------------
                # GT-side
                # ------------------------------

                "covered_gt":
                    int(
                        total_gt
                        - missed_gt
                    ),

                "missed_gt":
                    missed_gt,

                "single_covered_gt":
                    single_covered_gt,

                "duplicate_covered_gt":
                    duplicate_covered_gt,

                "gt_coverage_rate":
                    (
                        float(
                            (
                                total_gt
                                - missed_gt
                            )
                            / total_gt
                        )
                        if total_gt > 0
                        else np.nan
                    ),

                "duplicate_gt_rate":
                    (
                        float(
                            duplicate_covered_gt
                            / total_gt
                        )
                        if total_gt > 0
                        else np.nan
                    ),

                "strict_one_to_one_gt":
                    strict_one_to_one_gt,

                "strict_one_to_one_gt_rate":
                    (
                        float(
                            strict_one_to_one_gt
                            / total_gt
                        )
                        if total_gt > 0
                        else np.nan
                    ),

                # ------------------------------
                # Count-load diagnostic
                # ------------------------------

                "mask_to_gt_ratio":
                    (
                        float(
                            total_masks
                            / total_gt
                        )
                        if total_gt > 0
                        else np.nan
                    ),

                # ------------------------------
                # Boundary
                # ------------------------------

                "boundary_gt_points":
                    boundary_gt_points,

                "boundary_affected_masks":
                    boundary_affected_masks,
            }
        )


# ------------------------------------------------------------
# Build dataframe
# ------------------------------------------------------------

mask_diagnostics_by_density_df = (
    pd.DataFrame(
        density_summary_records
    )
)


# ------------------------------------------------------------
# Ordering
# ------------------------------------------------------------

checkpoint_order = {
    checkpoint_id: i
    for i, checkpoint_id
    in enumerate(
        CHECKPOINT_IDS
    )
}

density_order_map = {
    density: i
    for i, density
    in enumerate(
        DENSITY_ORDER
    )
}


mask_diagnostics_by_density_df[
    "_checkpoint_order"
] = (
    mask_diagnostics_by_density_df[
        "checkpoint_id"
    ]
    .map(
        checkpoint_order
    )
)

mask_diagnostics_by_density_df[
    "_density_order"
] = (
    mask_diagnostics_by_density_df[
        "density_stratum"
    ]
    .map(
        density_order_map
    )
)


mask_diagnostics_by_density_df = (
    mask_diagnostics_by_density_df
    .sort_values(
        [
            "_checkpoint_order",
            "_density_order",
        ]
    )
    .drop(
        columns=[
            "_checkpoint_order",
            "_density_order",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Strong integrity checks
# ------------------------------------------------------------

assert (
    len(
        mask_diagnostics_by_density_df
    )
    == 18
)


expected_num_images = {
    "S1_0_20": 20,
    "S2_20_40": 20,
    "S3_40_60": 20,
    "S4_60_80": 20,
    "S5_80_95": 10,
    "S6_95_100": 10,
}


for checkpoint_id in CHECKPOINT_IDS:

    system_df = (
        mask_diagnostics_by_density_df[
            mask_diagnostics_by_density_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
    )


    assert (
        system_df[
            "density_stratum"
        ].tolist()
        ==
        DENSITY_ORDER
    )


    assert (
        system_df[
            "num_images"
        ].tolist()
        ==
        [
            expected_num_images[
                density
            ]
            for density
            in DENSITY_ORDER
        ]
    )


    # Reaggregate back to frozen Step 6.1 totals
    system_global = (
        mask_diagnostics_by_system_df[
            mask_diagnostics_by_system_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
        .iloc[0]
    )


    assert (
        int(
            system_df[
                "total_masks"
            ].sum()
        )
        ==
        int(
            system_global[
                "total_masks"
            ]
        )
    )

    assert (
        int(
            system_df[
                "total_gt"
            ].sum()
        )
        ==
        int(
            system_global[
                "total_gt"
            ]
        )
    )

    assert (
        int(
            system_df[
                "zero_gt_masks"
            ].sum()
        )
        ==
        int(
            system_global[
                "zero_gt_masks"
            ]
        )
    )

    assert (
        int(
            system_df[
                "merged_gt_masks"
            ].sum()
        )
        ==
        int(
            system_global[
                "merged_gt_masks"
            ]
        )
    )

    assert (
        int(
            system_df[
                "missed_gt"
            ].sum()
        )
        ==
        int(
            system_global[
                "missed_gt"
            ]
        )
    )

    assert (
        int(
            system_df[
                "duplicate_covered_gt"
            ].sum()
        )
        ==
        int(
            system_global[
                "duplicate_covered_gt"
            ]
        )
    )


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

mask_diagnostics_by_density_df.to_csv(
    MASK_DIAGNOSTICS_BY_DENSITY_FILE,
    index=False,
)


# ------------------------------------------------------------
# Useful comparison pivots
# ------------------------------------------------------------

gt_coverage_pivot = (
    mask_diagnostics_by_density_df
    .pivot(
        index="density_stratum",
        columns="checkpoint_id",
        values="gt_coverage_rate",
    )
    .reindex(
        DENSITY_ORDER
    )
    [
        CHECKPOINT_IDS
    ]
)


duplicate_gt_pivot = (
    mask_diagnostics_by_density_df
    .pivot(
        index="density_stratum",
        columns="checkpoint_id",
        values="duplicate_gt_rate",
    )
    .reindex(
        DENSITY_ORDER
    )
    [
        CHECKPOINT_IDS
    ]
)


strict_one_to_one_pivot = (
    mask_diagnostics_by_density_df
    .pivot(
        index="density_stratum",
        columns="checkpoint_id",
        values="strict_one_to_one_gt_rate",
    )
    .reindex(
        DENSITY_ORDER
    )
    [
        CHECKPOINT_IDS
    ]
)


zero_gt_mask_pivot = (
    mask_diagnostics_by_density_df
    .pivot(
        index="density_stratum",
        columns="checkpoint_id",
        values="zero_gt_mask_rate",
    )
    .reindex(
        DENSITY_ORDER
    )
    [
        CHECKPOINT_IDS
    ]
)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("=" * 115)
print("STEP 6.2 - MASK CORRESPONDENCE BY DENSITY STRATUM")
print("=" * 115)

print(
    "Rows                         :",
    len(
        mask_diagnostics_by_density_df
    ),
)

print(
    "Expected                     : 3 systems × 6 strata = 18"
)

print(
    "S1-S6 image counts           : PASS"
)

print(
    "Reaggregation to Step 6.1    : PASS"
)

print(
    "Test set accessed            : NO"
)


print(
    "\nGT coverage rate by density:"
)

display(
    (
        gt_coverage_pivot
        * 100
    )
    .round(2)
    .rename_axis(
        columns=None
    )
)


print(
    "\nDuplicate-covered GT rate by density:"
)

display(
    (
        duplicate_gt_pivot
        * 100
    )
    .round(2)
    .rename_axis(
        columns=None
    )
)


print(
    "\nStrict one-to-one GT rate by density:"
)

display(
    (
        strict_one_to_one_pivot
        * 100
    )
    .round(2)
    .rename_axis(
        columns=None
    )
)


print(
    "\nZero-GT mask rate by density:"
)

display(
    (
        zero_gt_mask_pivot
        * 100
    )
    .round(2)
    .rename_axis(
        columns=None
    )
)


print(
    "\nFull pooled S1-S6 diagnostics:"
)

display(
    mask_diagnostics_by_density_df[
        [
            "checkpoint_id",
            "density_stratum",
            "num_images",
            "total_masks",
            "total_gt",
            "zero_gt_mask_rate",
            "merged_mask_rate",
            "gt_coverage_rate",
            "duplicate_gt_rate",
            "strict_one_to_one_gt_rate",
            "mask_to_gt_ratio",
        ]
    ]
    .round(4)
)


print(
    "\nSaved:",
    MASK_DIAGNOSTICS_BY_DENSITY_FILE,
)

print("=" * 115)

STEP 6.2 - MASK CORRESPONDENCE BY DENSITY STRATUM
Rows                         : 18
Expected                     : 3 systems × 6 strata = 18
S1-S6 image counts           : PASS
Reaggregation to Step 6.1    : PASS
Test set accessed            : NO

GT coverage rate by density:


,overall_joint,sparse_joint,dense_recovery
density_stratum,,,
S1_0_20,86.15,85.64,91.28
S2_20_40,82.81,80.62,90.94
S3_40_60,73.18,67.60,76.72
S4_60_80,73.81,70.16,88.23
S5_80_95,43.52,38.15,77.37
S6_95_100,68.67,67.39,81.56



Duplicate-covered GT rate by density:


,overall_joint,sparse_joint,dense_recovery
density_stratum,,,
S1_0_20,68.72,66.15,68.21
S2_20_40,52.81,47.19,55.94
S3_40_60,44.69,41.53,51.58
S4_60_80,53.47,50.55,52.92
S5_80_95,21.55,17.34,39.72
S6_95_100,60.08,57.36,73.05



Strict one-to-one GT rate by density:


...


Zero-GT mask rate by density:


,overall_joint,sparse_joint,dense_recovery
density_stratum,,,
S1_0_20,9.92,10.23,15.95
S2_20_40,18.00,18.70,20.98
S3_40_60,10.73,10.32,22.77
S4_60_80,15.06,15.46,22.60
S5_80_95,3.67,4.01,7.89
S6_95_100,25.27,25.72,13.04



Full pooled S1-S6 diagnostics:


,checkpoint_id,density_stratum,num_images,total_masks,total_gt,zero_gt_mask_rate,merged_mask_rate,gt_coverage_rate,duplicate_gt_rate,strict_one_to_one_gt_rate,mask_to_gt_ratio
0,overall_joint,S1_0_20,20,827,195,0.0992,0.0472,0.8615,0.6872,0.1538,4.2410
1,overall_joint,S2_20_40,20,972,320,0.1800,0.0154,0.8281,0.5281,0.2562,3.0375
2,overall_joint,S3_40_60,20,913,537,0.1073,0.0767,0.7318,0.4469,0.2160,1.7002
3,overall_joint,S4_60_80,20,1149,1096,0.1506,0.1366,0.7381,0.5347,0.1606,1.0484
4,overall_joint,S5_80_95,10,681,1211,0.0367,0.0720,0.4352,0.2155,0.1643,0.5623
5,overall_joint,S6_95_100,10,566,6686,0.2527,0.3993,0.6867,0.6008,0.0160,0.0847
6,sparse_joint,S1_0_20,20,753,195,0.1023,0.0478,0.8564,0.6615,0.1744,3.8615
7,sparse_joint,S2_20_40,20,877,320,0.1870,0.0148,0.8062,0.4719,0.2625,2.7406
8,sparse_joint,S3_40_60,20,804,537,0.1032,0.0808,0.6760,0.4153,0.2067,1.4972
9,sparse_joint,S4_60_80,20,1009,1096,0.1546,0.1298,0.7016,0.5055,0.1670,0.9206



Saved: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/guided_mask_diagnostics_by_density.csv


In [19]:
# ============================================================
# Step 6.3 - Multiplicity Severity Diagnostics
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

MASK_MULTIPLICITY_BY_SYSTEM_FILE = (
    E2E_TABLES_DIR
    / "guided_mask_multiplicity_by_system.csv"
)

GT_MULTIPLICITY_BY_SYSTEM_FILE = (
    E2E_TABLES_DIR
    / "guided_gt_multiplicity_by_system.csv"
)

MASK_MULTIPLICITY_BY_DENSITY_FILE = (
    E2E_TABLES_DIR
    / "guided_mask_multiplicity_by_density.csv"
)

GT_MULTIPLICITY_BY_DENSITY_FILE = (
    E2E_TABLES_DIR
    / "guided_gt_multiplicity_by_density.csv"
)


# ------------------------------------------------------------
# 1. Fixed multiplicity bins
# ------------------------------------------------------------

MULTIPLICITY_BIN_ORDER = [
    "0",
    "1",
    "2",
    "3",
    "4",
    "5",
    ">5",
]


def multiplicity_bin(value):

    value = int(value)

    if value <= 5:
        return str(value)

    return ">5"


# ------------------------------------------------------------
# 2. Add multiplicity bins to detailed tables
# ------------------------------------------------------------

mask_multiplicity_df = (
    mask_gt_correspondence_df
    .copy()
)

mask_multiplicity_df[
    "gt_per_mask_bin"
] = (
    mask_multiplicity_df[
        "num_gt_points_in_mask"
    ]
    .apply(
        multiplicity_bin
    )
)


gt_multiplicity_df = (
    gt_mask_coverage_df
    .copy()
)

gt_multiplicity_df[
    "masks_per_gt_bin"
] = (
    gt_multiplicity_df[
        "num_covering_masks"
    ]
    .apply(
        multiplicity_bin
    )
)


# ------------------------------------------------------------
# 3. Generic binned-summary helper
# ------------------------------------------------------------

def build_bin_summary(
    df,
    group_cols,
    bin_col,
    value_col,
    total_name,
):

    grouped = (
        df
        .groupby(
            group_cols
            + [
                bin_col
            ],
            observed=True,
        )
        .size()
        .rename(
            "count"
        )
        .reset_index()
    )


    # --------------------------------------------------------
    # Complete all missing bin combinations with zero
    # --------------------------------------------------------

    group_frame = (
        df[
            group_cols
        ]
        .drop_duplicates()
        .copy()
    )


    bin_frame = pd.DataFrame(
        {
            bin_col:
                MULTIPLICITY_BIN_ORDER
        }
    )


    group_frame[
        "_key"
    ] = 1

    bin_frame[
        "_key"
    ] = 1


    complete = (
        group_frame
        .merge(
            bin_frame,
            on="_key",
            how="outer",
        )
        .drop(
            columns="_key"
        )
        .merge(
            grouped,
            on=(
                group_cols
                + [
                    bin_col
                ]
            ),
            how="left",
        )
    )


    complete[
        "count"
    ] = (
        complete[
            "count"
        ]
        .fillna(0)
        .astype(int)
    )


    # --------------------------------------------------------
    # Group totals + rates
    # --------------------------------------------------------

    totals = (
        complete
        .groupby(
            group_cols,
            observed=True,
        )[
            "count"
        ]
        .sum()
        .rename(
            total_name
        )
        .reset_index()
    )


    complete = (
        complete
        .merge(
            totals,
            on=group_cols,
            how="left",
            validate="many_to_one",
        )
    )


    complete[
        "rate"
    ] = (
        complete[
            "count"
        ]
        /
        complete[
            total_name
        ]
    )


    return complete


# ------------------------------------------------------------
# 4. System-level multiplicity distributions
# ------------------------------------------------------------

mask_multiplicity_by_system_df = (
    build_bin_summary(
        df=
            mask_multiplicity_df,

        group_cols=[
            "checkpoint_id",
        ],

        bin_col=
            "gt_per_mask_bin",

        value_col=
            "num_gt_points_in_mask",

        total_name=
            "total_masks",
    )
)


gt_multiplicity_by_system_df = (
    build_bin_summary(
        df=
            gt_multiplicity_df,

        group_cols=[
            "checkpoint_id",
        ],

        bin_col=
            "masks_per_gt_bin",

        value_col=
            "num_covering_masks",

        total_name=
            "total_gt",
    )
)


# ------------------------------------------------------------
# 5. Density-level multiplicity distributions
# ------------------------------------------------------------

mask_multiplicity_by_density_df = (
    build_bin_summary(
        df=
            mask_multiplicity_df,

        group_cols=[
            "checkpoint_id",
            "density_stratum",
        ],

        bin_col=
            "gt_per_mask_bin",

        value_col=
            "num_gt_points_in_mask",

        total_name=
            "total_masks",
    )
)


gt_multiplicity_by_density_df = (
    build_bin_summary(
        df=
            gt_multiplicity_df,

        group_cols=[
            "checkpoint_id",
            "density_stratum",
        ],

        bin_col=
            "masks_per_gt_bin",

        value_col=
            "num_covering_masks",

        total_name=
            "total_gt",
    )
)


# ------------------------------------------------------------
# 6. Add exact severity statistics
# ------------------------------------------------------------

mask_exact_stats_records = []
gt_exact_stats_records = []


for checkpoint_id in CHECKPOINT_IDS:

    # --------------------------------------------------------
    # Mask-side exact statistics
    # --------------------------------------------------------

    mask_values = (
        mask_multiplicity_df[
            mask_multiplicity_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ][
            "num_gt_points_in_mask"
        ]
        .astype(int)
        .to_numpy()
    )


    positive_mask_values = (
        mask_values[
            mask_values > 0
        ]
    )


    merged_mask_values = (
        mask_values[
            mask_values >= 2
        ]
    )


    mask_exact_stats_records.append(
        {
            "checkpoint_id":
                checkpoint_id,

            "total_masks":
                int(
                    len(
                        mask_values
                    )
                ),

            "mean_gt_per_mask":
                float(
                    mask_values.mean()
                ),

            "median_gt_per_mask":
                float(
                    np.median(
                        mask_values
                    )
                ),

            "max_gt_per_mask":
                int(
                    mask_values.max()
                ),

            "mean_gt_per_supported_mask":
                (
                    float(
                        positive_mask_values.mean()
                    )
                    if len(
                        positive_mask_values
                    ) > 0
                    else np.nan
                ),

            "mean_gt_per_merged_mask":
                (
                    float(
                        merged_mask_values.mean()
                    )
                    if len(
                        merged_mask_values
                    ) > 0
                    else np.nan
                ),

            "median_gt_per_merged_mask":
                (
                    float(
                        np.median(
                            merged_mask_values
                        )
                    )
                    if len(
                        merged_mask_values
                    ) > 0
                    else np.nan
                ),

            "max_gt_per_merged_mask":
                (
                    int(
                        merged_mask_values.max()
                    )
                    if len(
                        merged_mask_values
                    ) > 0
                    else 0
                ),

            "num_masks_gt_gt5":
                int(
                    (
                        mask_values > 5
                    ).sum()
                ),

            "rate_masks_gt_gt5":
                float(
                    (
                        mask_values > 5
                    ).mean()
                ),
        }
    )


    # --------------------------------------------------------
    # GT-side exact statistics
    # --------------------------------------------------------

    gt_values = (
        gt_multiplicity_df[
            gt_multiplicity_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ][
            "num_covering_masks"
        ]
        .astype(int)
        .to_numpy()
    )


    covered_gt_values = (
        gt_values[
            gt_values > 0
        ]
    )


    duplicate_gt_values = (
        gt_values[
            gt_values >= 2
        ]
    )


    gt_exact_stats_records.append(
        {
            "checkpoint_id":
                checkpoint_id,

            "total_gt":
                int(
                    len(
                        gt_values
                    )
                ),

            "mean_masks_per_gt":
                float(
                    gt_values.mean()
                ),

            "median_masks_per_gt":
                float(
                    np.median(
                        gt_values
                    )
                ),

            "max_masks_per_gt":
                int(
                    gt_values.max()
                ),

            "mean_masks_per_covered_gt":
                (
                    float(
                        covered_gt_values.mean()
                    )
                    if len(
                        covered_gt_values
                    ) > 0
                    else np.nan
                ),

            "mean_masks_per_duplicate_gt":
                (
                    float(
                        duplicate_gt_values.mean()
                    )
                    if len(
                        duplicate_gt_values
                    ) > 0
                    else np.nan
                ),

            "median_masks_per_duplicate_gt":
                (
                    float(
                        np.median(
                            duplicate_gt_values
                        )
                    )
                    if len(
                        duplicate_gt_values
                    ) > 0
                    else np.nan
                ),

            "max_masks_per_duplicate_gt":
                (
                    int(
                        duplicate_gt_values.max()
                    )
                    if len(
                        duplicate_gt_values
                    ) > 0
                    else 0
                ),

            "num_gt_gt5_masks":
                int(
                    (
                        gt_values > 5
                    ).sum()
                ),

            "rate_gt_gt5_masks":
                float(
                    (
                        gt_values > 5
                    ).mean()
                ),
        }
    )


mask_multiplicity_exact_stats_df = (
    pd.DataFrame(
        mask_exact_stats_records
    )
)

gt_multiplicity_exact_stats_df = (
    pd.DataFrame(
        gt_exact_stats_records
    )
)


# ------------------------------------------------------------
# 7. Strong integrity checks
# ------------------------------------------------------------

assert (
    len(
        mask_multiplicity_by_system_df
    )
    == (
        len(
            CHECKPOINT_IDS
        )
        * len(
            MULTIPLICITY_BIN_ORDER
        )
    )
)

assert (
    len(
        gt_multiplicity_by_system_df
    )
    == (
        len(
            CHECKPOINT_IDS
        )
        * len(
            MULTIPLICITY_BIN_ORDER
        )
    )
)


assert (
    len(
        mask_multiplicity_by_density_df
    )
    == (
        len(
            CHECKPOINT_IDS
        )
        * len(
            DENSITY_ORDER
        )
        * len(
            MULTIPLICITY_BIN_ORDER
        )
    )
)

assert (
    len(
        gt_multiplicity_by_density_df
    )
    == (
        len(
            CHECKPOINT_IDS
        )
        * len(
            DENSITY_ORDER
        )
        * len(
            MULTIPLICITY_BIN_ORDER
        )
    )
)


# System mask totals must reaggregate to Step 6.1.
for checkpoint_id in CHECKPOINT_IDS:

    expected_masks = int(
        mask_diagnostics_by_system_df[
            mask_diagnostics_by_system_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ][
            "total_masks"
        ]
        .iloc[0]
    )


    actual_masks = int(
        mask_multiplicity_by_system_df[
            mask_multiplicity_by_system_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ][
            "count"
        ]
        .sum()
    )


    assert (
        actual_masks
        == expected_masks
    )


    expected_gt = int(
        mask_diagnostics_by_system_df[
            mask_diagnostics_by_system_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ][
            "total_gt"
        ]
        .iloc[0]
    )


    actual_gt = int(
        gt_multiplicity_by_system_df[
            gt_multiplicity_by_system_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ][
            "count"
        ]
        .sum()
    )


    assert (
        actual_gt
        == expected_gt
    )


# Rates must sum to 1 within each group.
mask_system_rate_sums = (
    mask_multiplicity_by_system_df
    .groupby(
        "checkpoint_id",
        observed=True,
    )[
        "rate"
    ]
    .sum()
)

assert np.allclose(
    mask_system_rate_sums
    .to_numpy(),
    1.0,
)


gt_system_rate_sums = (
    gt_multiplicity_by_system_df
    .groupby(
        "checkpoint_id",
        observed=True,
    )[
        "rate"
    ]
    .sum()
)

assert np.allclose(
    gt_system_rate_sums
    .to_numpy(),
    1.0,
)


# ------------------------------------------------------------
# 8. Save tables
# ------------------------------------------------------------

mask_multiplicity_by_system_df.to_csv(
    MASK_MULTIPLICITY_BY_SYSTEM_FILE,
    index=False,
)

gt_multiplicity_by_system_df.to_csv(
    GT_MULTIPLICITY_BY_SYSTEM_FILE,
    index=False,
)

mask_multiplicity_by_density_df.to_csv(
    MASK_MULTIPLICITY_BY_DENSITY_FILE,
    index=False,
)

gt_multiplicity_by_density_df.to_csv(
    GT_MULTIPLICITY_BY_DENSITY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 9. Display pivots
# ------------------------------------------------------------

mask_multiplicity_pivot = (
    mask_multiplicity_by_system_df
    .pivot(
        index="gt_per_mask_bin",
        columns="checkpoint_id",
        values="rate",
    )
    .reindex(
        MULTIPLICITY_BIN_ORDER
    )
    [
        CHECKPOINT_IDS
    ]
)


gt_multiplicity_pivot = (
    gt_multiplicity_by_system_df
    .pivot(
        index="masks_per_gt_bin",
        columns="checkpoint_id",
        values="rate",
    )
    .reindex(
        MULTIPLICITY_BIN_ORDER
    )
    [
        CHECKPOINT_IDS
    ]
)


# Dense-strata focus: S4-S6 only
dense_mask_multiplicity_pivot = (
    mask_multiplicity_by_density_df[
        mask_multiplicity_by_density_df[
            "density_stratum"
        ].isin(
            [
                "S4_60_80",
                "S5_80_95",
                "S6_95_100",
            ]
        )
    ]
    .pivot_table(
        index=[
            "density_stratum",
            "gt_per_mask_bin",
        ],
        columns="checkpoint_id",
        values="rate",
        aggfunc="first",
    )
    .reindex(
        pd.MultiIndex.from_product(
            [
                [
                    "S4_60_80",
                    "S5_80_95",
                    "S6_95_100",
                ],
                MULTIPLICITY_BIN_ORDER,
            ],
            names=[
                "density_stratum",
                "gt_per_mask_bin",
            ],
        )
    )
    [
        CHECKPOINT_IDS
    ]
)


dense_gt_multiplicity_pivot = (
    gt_multiplicity_by_density_df[
        gt_multiplicity_by_density_df[
            "density_stratum"
        ].isin(
            [
                "S4_60_80",
                "S5_80_95",
                "S6_95_100",
            ]
        )
    ]
    .pivot_table(
        index=[
            "density_stratum",
            "masks_per_gt_bin",
        ],
        columns="checkpoint_id",
        values="rate",
        aggfunc="first",
    )
    .reindex(
        pd.MultiIndex.from_product(
            [
                [
                    "S4_60_80",
                    "S5_80_95",
                    "S6_95_100",
                ],
                MULTIPLICITY_BIN_ORDER,
            ],
            names=[
                "density_stratum",
                "masks_per_gt_bin",
            ],
        )
    )
    [
        CHECKPOINT_IDS
    ]
)


# ------------------------------------------------------------
# 10. Confirmation
# ------------------------------------------------------------

print("=" * 115)
print("STEP 6.3 - MULTIPLICITY SEVERITY DIAGNOSTICS")
print("=" * 115)

print(
    "Mask-side system bins        : PASS"
)

print(
    "GT-side system bins          : PASS"
)

print(
    "Mask-side S1-S6 bins         : PASS"
)

print(
    "GT-side S1-S6 bins           : PASS"
)

print(
    "Reaggregation to Step 6.1    : PASS"
)

print(
    "Rate sums                    : PASS"
)

print(
    "Test set accessed            : NO"
)


print(
    "\nGT points per SAM2 mask "
    "(percentage of masks):"
)

display(
    (
        mask_multiplicity_pivot
        * 100
    )
    .round(2)
    .rename_axis(
        columns=None
    )
)


print(
    "\nSAM2 masks per GT point "
    "(percentage of GT points):"
)

display(
    (
        gt_multiplicity_pivot
        * 100
    )
    .round(2)
    .rename_axis(
        columns=None
    )
)


print(
    "\nExact mask-side severity statistics:"
)

display(
    mask_multiplicity_exact_stats_df
    .round(4)
)


print(
    "\nExact GT-side severity statistics:"
)

display(
    gt_multiplicity_exact_stats_df
    .round(4)
)


print(
    "\nDense-strata GT points per mask "
    "(S4-S6, percentage):"
)

display(
    (
        dense_mask_multiplicity_pivot
        * 100
    )
    .round(2)
    .rename_axis(
        columns=None
    )
)


print(
    "\nDense-strata masks per GT "
    "(S4-S6, percentage):"
)

display(
    (
        dense_gt_multiplicity_pivot
        * 100
    )
    .round(2)
    .rename_axis(
        columns=None
    )
)


print(
    "\nSaved mask multiplicity by system:",
    MASK_MULTIPLICITY_BY_SYSTEM_FILE,
)

print(
    "Saved GT multiplicity by system:",
    GT_MULTIPLICITY_BY_SYSTEM_FILE,
)

print(
    "Saved mask multiplicity by density:",
    MASK_MULTIPLICITY_BY_DENSITY_FILE,
)

print(
    "Saved GT multiplicity by density:",
    GT_MULTIPLICITY_BY_DENSITY_FILE,
)

print("=" * 115)

STEP 6.3 - MULTIPLICITY SEVERITY DIAGNOSTICS
Mask-side system bins        : PASS
GT-side system bins          : PASS
Mask-side S1-S6 bins         : PASS
GT-side S1-S6 bins           : PASS
Reaggregation to Step 6.1    : PASS
Rate sums                    : PASS
Test set accessed            : NO

GT points per SAM2 mask (percentage of masks):


,overall_joint,sparse_joint,dense_recovery
gt_per_mask_bin,,,
0,13.63,13.89,17.84
1,75.49,75.39,69.24
2,3.37,3.47,4.27
3,0.59,0.60,0.84
4,0.16,0.15,0.19
5,0.10,0.11,0.23
>5,6.68,6.38,7.40



SAM2 masks per GT point (percentage of GT points):


,overall_joint,sparse_joint,dense_recovery
masks_per_gt_bin,,,
0,32.77,35.04,17.99
1,13.40,14.17,16.96
2,5.61,4.50,7.47
3,3.00,2.47,2.95
4,1.64,1.33,2.62
5,1.30,0.81,1.04
>5,42.27,41.68,50.98



Exact mask-side severity statistics:


...


Dense-strata GT points per mask (S4-S6, percentage):


overall_joint  sparse_joint  dense_recovery
density_stratum gt_per_mask_bin                                             
S4_60_80        0                        15.06         15.46           22.60
                1                        71.28         71.56           59.73
                2                         2.79          2.97            3.91
                3                         0.61          0.69            0.59
                4                         0.26          0.30            0.17
                5                         0.17          0.20            0.17
                >5                        9.83          8.82           12.83
S5_80_95        0                         3.67          4.01            7.89
                1                        89.13         88.63           85.15
                2                         4.85          4.85            5.09
                3                         0.29          0.33            0.10
                4                         0.15          0.17            0.10
                5                         0.00          0.00            0.00
                >5                        1.91          2.01            1.66
S6_95_100       0                        25.27         25.72           13.04
                1                        34.81         33.95           52.41
                2                         7.60          7.20            9.11
                3                         3.18          3.29            3.16
                4                         0.35          0.41            0.76
                5                         0.53          0.62            0.25
                >5                       28.27         28.81           21.27


Dense-strata masks per GT (S4-S6, percentage):


overall_joint  sparse_joint  dense_recovery
density_stratum masks_per_gt_bin                                             
S4_60_80        0                         26.19         29.84           11.77
                1                         20.35         19.62           35.31
                2                         13.78         13.32           12.23
                3                          7.12          8.39            8.12
                4                          3.92          3.74            4.20
                5                          3.74          1.73            1.09
                >5                        24.91         23.36           27.28
S5_80_95        0                         56.48         61.85           22.63
                1                         21.97         20.81           37.65
                2                          8.67          6.36           20.31
                3                          3.72          2.23            4.38
                4                          0.74          0.58            9.50
                5                          0.41          0.33            1.57
                >5                         8.01          7.84            3.96
S6_95_100       0                         31.33         32.61           18.44
                1                          8.59         10.04            8.51
                2                          2.59          1.30            3.40
                3                          1.30          0.36            0.78
                4                          0.24          0.21            0.22
                5                          0.36          0.19            0.18
                >5                        55.59         55.29           68.47


Saved mask multiplicity by system: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/guided_mask_multiplicity_by_system.csv
Saved GT multiplicity by system: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/guided_gt_multiplicity_by_system.csv
Saved mask multiplicity by density: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/guided_mask_multiplicity_by_density.csv
Saved GT multiplicity by density: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/guided_gt_multiplicity_by_density.csv


In [20]:
# ============================================================
# Step 7.1 - End-to-End Counting Comparison vs Frozen B0
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

COUNT_COMPARISON_PER_IMAGE_FILE = (
    E2E_TABLES_DIR
    / "count_comparison_per_image.csv"
)

COUNT_METRICS_OVERALL_FILE = (
    E2E_TABLES_DIR
    / "count_metrics_overall.csv"
)

COUNT_METRICS_BY_DENSITY_FILE = (
    E2E_TABLES_DIR
    / "count_metrics_by_density.csv"
)

COUNT_PAIRED_VS_B0_FILE = (
    E2E_TABLES_DIR
    / "count_paired_vs_b0.csv"
)

COUNT_PROXY_TRANSLATION_FILE = (
    E2E_TABLES_DIR
    / "count_proxy_translation_check.csv"
)


# ------------------------------------------------------------
# 1. System order
# ------------------------------------------------------------

B0_SYSTEM_ID = "B0_unguided"

COUNT_SYSTEM_ORDER = [
    B0_SYSTEM_ID,
    "overall_joint",
    "sparse_joint",
    "dense_recovery",
]


# ------------------------------------------------------------
# 2. Build B0 count table
# ------------------------------------------------------------

b0_count_df = (
    b0_per_image_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "pred_count",
        ]
    ]
    .copy()
)

b0_count_df.insert(
    0,
    "system_id",
    B0_SYSTEM_ID,
)


# ------------------------------------------------------------
# 3. Build guided count table
# ------------------------------------------------------------

guided_count_df = (
    guided_inference_df[
        [
            "checkpoint_id",
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "num_reference_points",
            "num_retained_masks",
            "pred_count",
        ]
    ]
    .copy()
)

guided_count_df = guided_count_df.rename(
    columns={
        "checkpoint_id":
            "system_id",
    }
)


# ------------------------------------------------------------
# 4. Strong guided counting-rule validation
# ------------------------------------------------------------

assert (
    guided_count_df[
        "pred_count"
    ]
    ==
    guided_count_df[
        "num_retained_masks"
    ]
).all()

assert (
    guided_count_df[
        "num_retained_masks"
    ]
    ==
    guided_count_df[
        "num_reference_points"
    ]
).all()


# ------------------------------------------------------------
# 5. Explicitly verify guided final counts against
#    frozen Group-E proxy counts
# ------------------------------------------------------------

group_e_proxy_df = (
    e3_checkpoint_points_df[
        [
            "checkpoint_id",
            "dataset_index",
            "image_id",
            "pred_count",
        ]
    ]
    .rename(
        columns={
            "checkpoint_id":
                "system_id",

            "pred_count":
                "proxy_pred_count",
        }
    )
    .copy()
)


proxy_translation_df = (
    guided_count_df
    .merge(
        group_e_proxy_df,
        on=[
            "system_id",
            "dataset_index",
            "image_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert (
    proxy_translation_df[
        "proxy_pred_count"
    ].notna().all()
)

assert (
    proxy_translation_df[
        "pred_count"
    ].astype(int)
    ==
    proxy_translation_df[
        "proxy_pred_count"
    ].astype(int)
).all()


proxy_translation_df[
    "count_translation_delta"
] = (
    proxy_translation_df[
        "pred_count"
    ].astype(int)
    -
    proxy_translation_df[
        "proxy_pred_count"
    ].astype(int)
)


assert (
    proxy_translation_df[
        "count_translation_delta"
    ]
    == 0
).all()


# ------------------------------------------------------------
# 6. Build unified long-form count table
# ------------------------------------------------------------

guided_long_df = (
    guided_count_df[
        [
            "system_id",
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "pred_count",
        ]
    ]
    .copy()
)


count_comparison_df = pd.concat(
    [
        b0_count_df,
        guided_long_df,
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# 7. Derive per-image count errors
# ------------------------------------------------------------

count_comparison_df[
    "signed_error"
] = (
    count_comparison_df[
        "pred_count"
    ].astype(float)
    -
    count_comparison_df[
        "gt_count"
    ].astype(float)
)


count_comparison_df[
    "abs_error"
] = (
    count_comparison_df[
        "signed_error"
    ].abs()
)


count_comparison_df[
    "squared_error"
] = (
    count_comparison_df[
        "signed_error"
    ] ** 2
)


assert (
    count_comparison_df[
        "gt_count"
    ]
    > 0
).all()


count_comparison_df[
    "relative_error"
] = (
    count_comparison_df[
        "abs_error"
    ]
    /
    count_comparison_df[
        "gt_count"
    ].astype(float)
)


count_comparison_df[
    "error_direction"
] = np.select(
    [
        count_comparison_df[
            "signed_error"
        ] < 0,

        count_comparison_df[
            "signed_error"
        ] > 0,
    ],
    [
        "under",
        "over",
    ],
    default="exact",
)


# ------------------------------------------------------------
# 8. Metric helper
# ------------------------------------------------------------

def compute_count_metrics(
    df,
):

    signed_error = (
        df[
            "signed_error"
        ].to_numpy(
            dtype=float
        )
    )

    abs_error = (
        df[
            "abs_error"
        ].to_numpy(
            dtype=float
        )
    )

    relative_error = (
        df[
            "relative_error"
        ].to_numpy(
            dtype=float
        )
    )


    return {
        "num_images":
            int(
                len(df)
            ),

        "mae":
            float(
                abs_error.mean()
            ),

        "rmse":
            float(
                np.sqrt(
                    np.mean(
                        signed_error ** 2
                    )
                )
            ),

        "mean_signed_error":
            float(
                signed_error.mean()
            ),

        "under_count":
            int(
                (
                    signed_error < 0
                ).sum()
            ),

        "exact_count":
            int(
                (
                    signed_error == 0
                ).sum()
            ),

        "over_count":
            int(
                (
                    signed_error > 0
                ).sum()
            ),

        "mean_relative_error":
            float(
                relative_error.mean()
            ),

        "median_relative_error":
            float(
                np.median(
                    relative_error
                )
            ),

        "within_1pct":
            float(
                (
                    relative_error
                    <= 0.01
                ).mean()
            ),

        "within_5pct":
            float(
                (
                    relative_error
                    <= 0.05
                ).mean()
            ),

        "within_10pct":
            float(
                (
                    relative_error
                    <= 0.10
                ).mean()
            ),

        "within_20pct":
            float(
                (
                    relative_error
                    <= 0.20
                ).mean()
            ),
    }


# ------------------------------------------------------------
# 9. Overall metrics
# ------------------------------------------------------------

overall_metric_records = []


for system_id in COUNT_SYSTEM_ORDER:

    system_df = (
        count_comparison_df[
            count_comparison_df[
                "system_id"
            ]
            == system_id
        ]
    )

    assert (
        len(system_df)
        == 100
    )


    metrics = compute_count_metrics(
        system_df
    )

    metrics[
        "system_id"
    ] = system_id

    overall_metric_records.append(
        metrics
    )


count_metrics_overall_df = (
    pd.DataFrame(
        overall_metric_records
    )
)


count_metrics_overall_df[
    "_order"
] = (
    count_metrics_overall_df[
        "system_id"
    ]
    .map(
        {
            system_id: i
            for i, system_id
            in enumerate(
                COUNT_SYSTEM_ORDER
            )
        }
    )
)


count_metrics_overall_df = (
    count_metrics_overall_df
    .sort_values(
        "_order"
    )
    .drop(
        columns="_order"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 10. Validate reconstructed B0 metrics
# ------------------------------------------------------------

b0_recomputed = (
    count_metrics_overall_df[
        count_metrics_overall_df[
            "system_id"
        ]
        == B0_SYSTEM_ID
    ]
    .iloc[0]
)


assert np.isclose(
    float(
        b0_recomputed[
            "mae"
        ]
    ),
    80.69,
)

assert np.isclose(
    float(
        b0_recomputed[
            "rmse"
        ]
    ),
    270.2339,
    atol=1e-4,
)

assert np.isclose(
    float(
        b0_recomputed[
            "mean_signed_error"
        ]
    ),
    -77.47,
)


# ------------------------------------------------------------
# 11. Verify guided MAE/RMSE reproduce frozen Group-E proxy
# ------------------------------------------------------------

proxy_metric_check_records = []


for checkpoint_id in CHECKPOINT_IDS:

    e2e_row = (
        count_metrics_overall_df[
            count_metrics_overall_df[
                "system_id"
            ]
            == checkpoint_id
        ]
        .iloc[0]
    )

    proxy_row = (
        e3_shortlist_df[
            e3_shortlist_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
        .iloc[0]
    )


    mae_match = np.isclose(
        float(
            e2e_row[
                "mae"
            ]
        ),
        float(
            proxy_row[
                "mae"
            ]
        ),
    )

    rmse_match = np.isclose(
        float(
            e2e_row[
                "rmse"
            ]
        ),
        float(
            proxy_row[
                "rmse"
            ]
        ),
    )


    assert mae_match
    assert rmse_match


    proxy_metric_check_records.append(
        {
            "checkpoint_id":
                checkpoint_id,

            "proxy_mae":
                float(
                    proxy_row[
                        "mae"
                    ]
                ),

            "end_to_end_mae":
                float(
                    e2e_row[
                        "mae"
                    ]
                ),

            "proxy_rmse":
                float(
                    proxy_row[
                        "rmse"
                    ]
                ),

            "end_to_end_rmse":
                float(
                    e2e_row[
                        "rmse"
                    ]
                ),

            "mae_match":
                bool(
                    mae_match
                ),

            "rmse_match":
                bool(
                    rmse_match
                ),
        }
    )


count_proxy_translation_check_df = (
    pd.DataFrame(
        proxy_metric_check_records
    )
)


# ------------------------------------------------------------
# 12. Density-stratified count metrics
# ------------------------------------------------------------

density_metric_records = []


for system_id in COUNT_SYSTEM_ORDER:

    for density_stratum in DENSITY_ORDER:

        subset_df = (
            count_comparison_df[
                (
                    count_comparison_df[
                        "system_id"
                    ]
                    == system_id
                )
                &
                (
                    count_comparison_df[
                        "density_stratum"
                    ]
                    == density_stratum
                )
            ]
        )


        expected_n = (
            20
            if density_stratum
            in [
                "S1_0_20",
                "S2_20_40",
                "S3_40_60",
                "S4_60_80",
            ]
            else 10
        )


        assert (
            len(subset_df)
            == expected_n
        )


        metrics = compute_count_metrics(
            subset_df
        )

        metrics[
            "system_id"
        ] = system_id

        metrics[
            "density_stratum"
        ] = density_stratum

        density_metric_records.append(
            metrics
        )


count_metrics_by_density_df = (
    pd.DataFrame(
        density_metric_records
    )
)


# ------------------------------------------------------------
# 13. Paired image-level comparison against B0
# ------------------------------------------------------------

b0_pair_df = (
    count_comparison_df[
        count_comparison_df[
            "system_id"
        ]
        == B0_SYSTEM_ID
    ][
        [
            "dataset_index",
            "image_id",
            "gt_count",
            "abs_error",
            "signed_error",
        ]
    ]
    .rename(
        columns={
            "abs_error":
                "b0_abs_error",

            "signed_error":
                "b0_signed_error",
        }
    )
)


paired_records = []


for checkpoint_id in CHECKPOINT_IDS:

    guided_pair_df = (
        count_comparison_df[
            count_comparison_df[
                "system_id"
            ]
            == checkpoint_id
        ][
            [
                "dataset_index",
                "image_id",
                "category",
                "density_stratum",
                "gt_count",
                "pred_count",
                "abs_error",
                "signed_error",
            ]
        ]
        .rename(
            columns={
                "pred_count":
                    "guided_pred_count",

                "abs_error":
                    "guided_abs_error",

                "signed_error":
                    "guided_signed_error",
            }
        )
    )


    paired_df = (
        guided_pair_df
        .merge(
            b0_pair_df,
            on=[
                "dataset_index",
                "image_id",
                "gt_count",
            ],
            how="left",
            validate="one_to_one",
        )
    )


    assert (
        len(paired_df)
        == 100
    )


    paired_df[
        "delta_abs_error_vs_b0"
    ] = (
        paired_df[
            "guided_abs_error"
        ]
        -
        paired_df[
            "b0_abs_error"
        ]
    )


    paired_df[
        "paired_outcome"
    ] = np.select(
        [
            paired_df[
                "delta_abs_error_vs_b0"
            ] < 0,

            paired_df[
                "delta_abs_error_vs_b0"
            ] > 0,
        ],
        [
            "improved",
            "worsened",
        ],
        default="same",
    )


    improved = int(
        (
            paired_df[
                "paired_outcome"
            ]
            == "improved"
        ).sum()
    )

    same = int(
        (
            paired_df[
                "paired_outcome"
            ]
            == "same"
        ).sum()
    )

    worsened = int(
        (
            paired_df[
                "paired_outcome"
            ]
            == "worsened"
        ).sum()
    )


    assert (
        improved
        + same
        + worsened
        == 100
    )


    paired_records.append(
        {
            "checkpoint_id":
                checkpoint_id,

            "improved_images":
                improved,

            "same_images":
                same,

            "worsened_images":
                worsened,

            "mean_delta_abs_error_vs_b0":
                float(
                    paired_df[
                        "delta_abs_error_vs_b0"
                    ].mean()
                ),

            "median_delta_abs_error_vs_b0":
                float(
                    paired_df[
                        "delta_abs_error_vs_b0"
                    ].median()
                ),

            "total_abs_error_change_vs_b0":
                float(
                    paired_df[
                        "delta_abs_error_vs_b0"
                    ].sum()
                ),
        }
    )


count_paired_vs_b0_df = (
    pd.DataFrame(
        paired_records
    )
)


# ------------------------------------------------------------
# 14. Save artefacts
# ------------------------------------------------------------

count_comparison_df.to_csv(
    COUNT_COMPARISON_PER_IMAGE_FILE,
    index=False,
)

count_metrics_overall_df.to_csv(
    COUNT_METRICS_OVERALL_FILE,
    index=False,
)

count_metrics_by_density_df.to_csv(
    COUNT_METRICS_BY_DENSITY_FILE,
    index=False,
)

count_paired_vs_b0_df.to_csv(
    COUNT_PAIRED_VS_B0_FILE,
    index=False,
)

count_proxy_translation_check_df.to_csv(
    COUNT_PROXY_TRANSLATION_FILE,
    index=False,
)


# ------------------------------------------------------------
# 15. Useful S1-S6 pivots
# ------------------------------------------------------------

mae_density_pivot = (
    count_metrics_by_density_df
    .pivot(
        index="density_stratum",
        columns="system_id",
        values="mae",
    )
    .reindex(
        DENSITY_ORDER
    )
    [
        COUNT_SYSTEM_ORDER
    ]
)


rmse_density_pivot = (
    count_metrics_by_density_df
    .pivot(
        index="density_stratum",
        columns="system_id",
        values="rmse",
    )
    .reindex(
        DENSITY_ORDER
    )
    [
        COUNT_SYSTEM_ORDER
    ]
)


# ------------------------------------------------------------
# 16. Confirmation
# ------------------------------------------------------------

print("=" * 115)
print("STEP 7.1 - END-TO-END COUNTING COMPARISON VS FROZEN B0")
print("=" * 115)

print(
    "Count systems                :",
    len(
        COUNT_SYSTEM_ORDER
    ),
)

print(
    "Images per system            : 100"
)

print(
    "B0 reconstruction            : PASS"
)

print(
    "Final count = retained masks : PASS"
)

print(
    "Retained masks = ref. points : PASS"
)

print(
    "Final count = Group-E proxy  : PASS"
)

print(
    "Proxy MAE/RMSE reproduction  : PASS"
)

print(
    "Test set accessed            : NO"
)


print(
    "\nOverall count metrics:"
)

display(
    count_metrics_overall_df[
        [
            "system_id",
            "num_images",
            "mae",
            "rmse",
            "mean_signed_error",
            "under_count",
            "exact_count",
            "over_count",
            "mean_relative_error",
            "median_relative_error",
            "within_10pct",
            "within_20pct",
        ]
    ]
    .round(4)
)


print(
    "\nMAE by density stratum:"
)

display(
    mae_density_pivot
    .round(2)
    .rename_axis(
        columns=None
    )
)


print(
    "\nRMSE by density stratum:"
)

display(
    rmse_density_pivot
    .round(2)
    .rename_axis(
        columns=None
    )
)


print(
    "\nPaired image-level change vs B0:"
)

display(
    count_paired_vs_b0_df
    .round(4)
)


print(
    "\nProxy -> final SAM2 count verification:"
)

display(
    count_proxy_translation_check_df
    .round(4)
)


print(
    "\nSaved per-image count table:",
    COUNT_COMPARISON_PER_IMAGE_FILE,
)

print(
    "Saved overall metrics:",
    COUNT_METRICS_OVERALL_FILE,
)

print(
    "Saved S1-S6 metrics:",
    COUNT_METRICS_BY_DENSITY_FILE,
)

print(
    "Saved paired comparison:",
    COUNT_PAIRED_VS_B0_FILE,
)

print("=" * 115)

STEP 7.1 - END-TO-END COUNTING COMPARISON VS FROZEN B0
Count systems                : 4
Images per system            : 100
B0 reconstruction            : PASS
Final count = retained masks : PASS
Retained masks = ref. points : PASS
Final count = Group-E proxy  : PASS
Proxy MAE/RMSE reproduction  : PASS
Test set accessed            : NO

Overall count metrics:


,system_id,num_images,mae,rmse,mean_signed_error,under_count,exact_count,over_count,mean_relative_error,median_relative_error,within_10pct,within_20pct
0,B0_unguided,100,80.69,270.2339,-77.47,76,4,20,0.5593,0.6579,0.13,0.22
1,overall_joint,100,88.17,262.7308,-49.37,36,0,64,1.4383,0.8776,0.04,0.10
2,sparse_joint,100,86.42,264.0067,-55.18,38,0,62,1.2641,0.8171,0.04,0.13
3,dense_recovery,100,86.43,259.0033,-43.01,35,2,63,1.3413,0.6692,0.12,0.19



MAE by density stratum:


,B0_unguided,overall_joint,sparse_joint,dense_recovery
density_stratum,,,,
S1_0_20,5.15,31.80,28.10,21.70
S2_20_40,6.55,32.80,28.05,34.55
S3_40_60,14.95,23.00,17.75,37.55
S4_60_80,34.30,20.75,17.55,24.85
S5_80_95,62.10,53.00,61.30,37.40
S6_95_100,622.90,612.00,620.00,589.60



RMSE by density stratum:


...


Paired image-level change vs B0:


,checkpoint_id,improved_images,same_images,worsened_images,mean_delta_abs_error_vs_b0,median_delta_abs_error_vs_b0,total_abs_error_change_vs_b0
0,overall_joint,43,0,57,7.48,9.0,748.0
1,sparse_joint,42,1,57,5.73,7.0,573.0
2,dense_recovery,47,0,53,5.74,2.0,574.0



Proxy -> final SAM2 count verification:


,checkpoint_id,proxy_mae,end_to_end_mae,proxy_rmse,end_to_end_rmse,mae_match,rmse_match
0,overall_joint,88.17,88.17,262.7308,262.7308,True,True
1,sparse_joint,86.42,86.42,264.0067,264.0067,True,True
2,dense_recovery,86.43,86.43,259.0033,259.0033,True,True



Saved per-image count table: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/count_comparison_per_image.csv
Saved overall metrics: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/count_metrics_overall.csv
Saved S1-S6 metrics: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/count_metrics_by_density.csv
Saved paired comparison: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/count_paired_vs_b0.csv


In [21]:
# ============================================================
# Step 7.2 - Error Concentration + Image-Level Drivers vs B0
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

ERROR_CHANGE_BY_DENSITY_FILE = (
    E2E_TABLES_DIR
    / "count_error_change_by_density_vs_b0.csv"
)

ERROR_CHANGE_CONCENTRATION_FILE = (
    E2E_TABLES_DIR
    / "count_error_change_concentration_vs_b0.csv"
)

ERROR_CHANGE_PER_IMAGE_FILE = (
    E2E_TABLES_DIR
    / "count_error_change_per_image_vs_b0.csv"
)

TOP_ERROR_CHANGE_IMAGES_FILE = (
    E2E_TABLES_DIR
    / "count_top_error_change_images_vs_b0.csv"
)


# ------------------------------------------------------------
# 1. Prepare frozen B0 comparison columns
# ------------------------------------------------------------

b0_error_df = (
    count_comparison_df[
        count_comparison_df[
            "system_id"
        ]
        == B0_SYSTEM_ID
    ][
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "pred_count",
            "signed_error",
            "abs_error",
            "squared_error",
        ]
    ]
    .rename(
        columns={
            "pred_count":
                "b0_pred_count",

            "signed_error":
                "b0_signed_error",

            "abs_error":
                "b0_abs_error",

            "squared_error":
                "b0_squared_error",
        }
    )
    .copy()
)


assert (
    len(
        b0_error_df
    )
    == 100
)


# ------------------------------------------------------------
# 2. Build paired per-image change table
# ------------------------------------------------------------

paired_change_frames = []


for checkpoint_id in CHECKPOINT_IDS:

    guided_df = (
        count_comparison_df[
            count_comparison_df[
                "system_id"
            ]
            == checkpoint_id
        ][
            [
                "dataset_index",
                "image_id",
                "category",
                "density_stratum",
                "gt_count",
                "pred_count",
                "signed_error",
                "abs_error",
                "squared_error",
            ]
        ]
        .rename(
            columns={
                "pred_count":
                    "guided_pred_count",

                "signed_error":
                    "guided_signed_error",

                "abs_error":
                    "guided_abs_error",

                "squared_error":
                    "guided_squared_error",
            }
        )
        .copy()
    )


    paired_df = (
        guided_df
        .merge(
            b0_error_df,
            on=[
                "dataset_index",
                "image_id",
                "category",
                "density_stratum",
                "gt_count",
            ],
            how="left",
            validate="one_to_one",
        )
    )


    assert (
        len(
            paired_df
        )
        == 100
    )


    paired_df.insert(
        0,
        "checkpoint_id",
        checkpoint_id,
    )


    # --------------------------------------------------------
    # Signed changes relative to B0
    #
    # Negative = guided improves the error.
    # Positive = guided worsens the error.
    # --------------------------------------------------------

    paired_df[
        "delta_abs_error_vs_b0"
    ] = (
        paired_df[
            "guided_abs_error"
        ]
        -
        paired_df[
            "b0_abs_error"
        ]
    )


    paired_df[
        "delta_squared_error_vs_b0"
    ] = (
        paired_df[
            "guided_squared_error"
        ]
        -
        paired_df[
            "b0_squared_error"
        ]
    )


    # --------------------------------------------------------
    # Improvement / worsening magnitudes
    # --------------------------------------------------------

    paired_df[
        "abs_error_improvement"
    ] = np.maximum(
        -paired_df[
            "delta_abs_error_vs_b0"
        ],
        0,
    )


    paired_df[
        "abs_error_worsening"
    ] = np.maximum(
        paired_df[
            "delta_abs_error_vs_b0"
        ],
        0,
    )


    paired_df[
        "squared_error_improvement"
    ] = np.maximum(
        -paired_df[
            "delta_squared_error_vs_b0"
        ],
        0,
    )


    paired_df[
        "squared_error_worsening"
    ] = np.maximum(
        paired_df[
            "delta_squared_error_vs_b0"
        ],
        0,
    )


    # --------------------------------------------------------
    # Paired outcome
    # --------------------------------------------------------

    paired_df[
        "paired_outcome"
    ] = np.select(
        [
            paired_df[
                "delta_abs_error_vs_b0"
            ] < 0,

            paired_df[
                "delta_abs_error_vs_b0"
            ] > 0,
        ],
        [
            "improved",
            "worsened",
        ],
        default="same",
    )


    paired_change_frames.append(
        paired_df
    )


count_error_change_per_image_df = (
    pd.concat(
        paired_change_frames,
        ignore_index=True,
    )
)


assert (
    len(
        count_error_change_per_image_df
    )
    == 300
)


# ------------------------------------------------------------
# 3. Density-stratified contribution to change
# ------------------------------------------------------------

density_change_records = []


for checkpoint_id in CHECKPOINT_IDS:

    system_df = (
        count_error_change_per_image_df[
            count_error_change_per_image_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
    )


    total_abs_improvement = float(
        system_df[
            "abs_error_improvement"
        ].sum()
    )

    total_abs_worsening = float(
        system_df[
            "abs_error_worsening"
        ].sum()
    )


    total_sq_improvement = float(
        system_df[
            "squared_error_improvement"
        ].sum()
    )

    total_sq_worsening = float(
        system_df[
            "squared_error_worsening"
        ].sum()
    )


    for density_stratum in DENSITY_ORDER:

        density_df = (
            system_df[
                system_df[
                    "density_stratum"
                ]
                == density_stratum
            ]
        )


        abs_improvement = float(
            density_df[
                "abs_error_improvement"
            ].sum()
        )

        abs_worsening = float(
            density_df[
                "abs_error_worsening"
            ].sum()
        )


        sq_improvement = float(
            density_df[
                "squared_error_improvement"
            ].sum()
        )

        sq_worsening = float(
            density_df[
                "squared_error_worsening"
            ].sum()
        )


        density_change_records.append(
            {
                "checkpoint_id":
                    checkpoint_id,

                "density_stratum":
                    density_stratum,

                "num_images":
                    int(
                        len(
                            density_df
                        )
                    ),

                "improved_images":
                    int(
                        (
                            density_df[
                                "paired_outcome"
                            ]
                            == "improved"
                        ).sum()
                    ),

                "same_images":
                    int(
                        (
                            density_df[
                                "paired_outcome"
                            ]
                            == "same"
                        ).sum()
                    ),

                "worsened_images":
                    int(
                        (
                            density_df[
                                "paired_outcome"
                            ]
                            == "worsened"
                        ).sum()
                    ),

                # ------------------------------
                # Absolute-error changes
                # ------------------------------

                "abs_error_improvement":
                    abs_improvement,

                "abs_error_worsening":
                    abs_worsening,

                "net_abs_error_change":
                    float(
                        density_df[
                            "delta_abs_error_vs_b0"
                        ].sum()
                    ),

                "share_total_abs_improvement":
                    (
                        float(
                            abs_improvement
                            /
                            total_abs_improvement
                        )
                        if total_abs_improvement > 0
                        else np.nan
                    ),

                "share_total_abs_worsening":
                    (
                        float(
                            abs_worsening
                            /
                            total_abs_worsening
                        )
                        if total_abs_worsening > 0
                        else np.nan
                    ),

                # ------------------------------
                # Squared-error changes
                # ------------------------------

                "squared_error_improvement":
                    sq_improvement,

                "squared_error_worsening":
                    sq_worsening,

                "net_squared_error_change":
                    float(
                        density_df[
                            "delta_squared_error_vs_b0"
                        ].sum()
                    ),

                "share_total_squared_improvement":
                    (
                        float(
                            sq_improvement
                            /
                            total_sq_improvement
                        )
                        if total_sq_improvement > 0
                        else np.nan
                    ),

                "share_total_squared_worsening":
                    (
                        float(
                            sq_worsening
                            /
                            total_sq_worsening
                        )
                        if total_sq_worsening > 0
                        else np.nan
                    ),
            }
        )


count_error_change_by_density_df = (
    pd.DataFrame(
        density_change_records
    )
)


assert (
    len(
        count_error_change_by_density_df
    )
    == 18
)


# ------------------------------------------------------------
# 4. Top-k concentration of improvements / worsenings
# ------------------------------------------------------------

TOP_K_VALUES = [
    1,
    5,
    10,
    20,
]


concentration_records = []


for checkpoint_id in CHECKPOINT_IDS:

    system_df = (
        count_error_change_per_image_df[
            count_error_change_per_image_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
        .copy()
    )


    improvement_values = (
        system_df[
            "abs_error_improvement"
        ]
        .sort_values(
            ascending=False
        )
        .to_numpy(
            dtype=float
        )
    )


    worsening_values = (
        system_df[
            "abs_error_worsening"
        ]
        .sort_values(
            ascending=False
        )
        .to_numpy(
            dtype=float
        )
    )


    squared_improvement_values = (
        system_df[
            "squared_error_improvement"
        ]
        .sort_values(
            ascending=False
        )
        .to_numpy(
            dtype=float
        )
    )


    total_improvement = float(
        improvement_values.sum()
    )

    total_worsening = float(
        worsening_values.sum()
    )

    total_squared_improvement = float(
        squared_improvement_values.sum()
    )


    for k in TOP_K_VALUES:

        topk_improvement = float(
            improvement_values[
                :k
            ].sum()
        )

        topk_worsening = float(
            worsening_values[
                :k
            ].sum()
        )

        topk_squared_improvement = float(
            squared_improvement_values[
                :k
            ].sum()
        )


        concentration_records.append(
            {
                "checkpoint_id":
                    checkpoint_id,

                "top_k":
                    int(
                        k
                    ),

                "topk_abs_improvement":
                    topk_improvement,

                "share_total_abs_improvement":
                    (
                        topk_improvement
                        /
                        total_improvement
                        if total_improvement > 0
                        else np.nan
                    ),

                "topk_abs_worsening":
                    topk_worsening,

                "share_total_abs_worsening":
                    (
                        topk_worsening
                        /
                        total_worsening
                        if total_worsening > 0
                        else np.nan
                    ),

                "topk_squared_improvement":
                    topk_squared_improvement,

                "share_total_squared_improvement":
                    (
                        topk_squared_improvement
                        /
                        total_squared_improvement
                        if total_squared_improvement > 0
                        else np.nan
                    ),
            }
        )


count_error_change_concentration_df = (
    pd.DataFrame(
        concentration_records
    )
)


# ------------------------------------------------------------
# 5. Identify strongest image-level drivers
# ------------------------------------------------------------

top_image_records = []


for checkpoint_id in CHECKPOINT_IDS:

    system_df = (
        count_error_change_per_image_df[
            count_error_change_per_image_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Top 10 absolute-error improvements
    # --------------------------------------------------------

    best_df = (
        system_df
        .sort_values(
            "delta_abs_error_vs_b0",
            ascending=True,
        )
        .head(
            10
        )
    )


    for rank, row in enumerate(
        best_df.itertuples(
            index=False
        ),
        start=1,
    ):

        top_image_records.append(
            {
                "checkpoint_id":
                    checkpoint_id,

                "change_type":
                    "largest_improvement",

                "rank":
                    rank,

                "dataset_index":
                    int(
                        row.dataset_index
                    ),

                "image_id":
                    str(
                        row.image_id
                    ),

                "category":
                    str(
                        row.category
                    ),

                "density_stratum":
                    str(
                        row.density_stratum
                    ),

                "gt_count":
                    int(
                        row.gt_count
                    ),

                "b0_pred_count":
                    int(
                        row.b0_pred_count
                    ),

                "guided_pred_count":
                    int(
                        row.guided_pred_count
                    ),

                "b0_abs_error":
                    float(
                        row.b0_abs_error
                    ),

                "guided_abs_error":
                    float(
                        row.guided_abs_error
                    ),

                "delta_abs_error_vs_b0":
                    float(
                        row.delta_abs_error_vs_b0
                    ),

                "delta_squared_error_vs_b0":
                    float(
                        row.delta_squared_error_vs_b0
                    ),
            }
        )


    # --------------------------------------------------------
    # Top 10 absolute-error worsenings
    # --------------------------------------------------------

    worst_df = (
        system_df
        .sort_values(
            "delta_abs_error_vs_b0",
            ascending=False,
        )
        .head(
            10
        )
    )


    for rank, row in enumerate(
        worst_df.itertuples(
            index=False
        ),
        start=1,
    ):

        top_image_records.append(
            {
                "checkpoint_id":
                    checkpoint_id,

                "change_type":
                    "largest_worsening",

                "rank":
                    rank,

                "dataset_index":
                    int(
                        row.dataset_index
                    ),

                "image_id":
                    str(
                        row.image_id
                    ),

                "category":
                    str(
                        row.category
                    ),

                "density_stratum":
                    str(
                        row.density_stratum
                    ),

                "gt_count":
                    int(
                        row.gt_count
                    ),

                "b0_pred_count":
                    int(
                        row.b0_pred_count
                    ),

                "guided_pred_count":
                    int(
                        row.guided_pred_count
                    ),

                "b0_abs_error":
                    float(
                        row.b0_abs_error
                    ),

                "guided_abs_error":
                    float(
                        row.guided_abs_error
                    ),

                "delta_abs_error_vs_b0":
                    float(
                        row.delta_abs_error_vs_b0
                    ),

                "delta_squared_error_vs_b0":
                    float(
                        row.delta_squared_error_vs_b0
                    ),
            }
        )


count_top_error_change_images_df = (
    pd.DataFrame(
        top_image_records
    )
)


# ------------------------------------------------------------
# 6. Strong integrity checks
# ------------------------------------------------------------

for checkpoint_id in CHECKPOINT_IDS:

    system_df = (
        count_error_change_per_image_df[
            count_error_change_per_image_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
    )


    assert (
        len(
            system_df
        )
        == 100
    )


    # Net absolute-error change must equal
    # 100 × change in overall MAE.
    guided_mae = float(
        count_metrics_overall_df[
            count_metrics_overall_df[
                "system_id"
            ]
            == checkpoint_id
        ][
            "mae"
        ]
        .iloc[0]
    )


    b0_mae = float(
        count_metrics_overall_df[
            count_metrics_overall_df[
                "system_id"
            ]
            == B0_SYSTEM_ID
        ][
            "mae"
        ]
        .iloc[0]
    )


    expected_abs_change = (
        100
        * (
            guided_mae
            - b0_mae
        )
    )


    actual_abs_change = float(
        system_df[
            "delta_abs_error_vs_b0"
        ].sum()
    )


    assert np.isclose(
        actual_abs_change,
        expected_abs_change,
    )


    # Density components must reaggregate.
    density_df = (
        count_error_change_by_density_df[
            count_error_change_by_density_df[
                "checkpoint_id"
            ]
            == checkpoint_id
        ]
    )


    assert np.isclose(
        density_df[
            "net_abs_error_change"
        ].sum(),
        actual_abs_change,
    )


    assert np.isclose(
        density_df[
            "net_squared_error_change"
        ].sum(),
        system_df[
            "delta_squared_error_vs_b0"
        ].sum(),
    )


# ------------------------------------------------------------
# 7. Save
# ------------------------------------------------------------

count_error_change_per_image_df.to_csv(
    ERROR_CHANGE_PER_IMAGE_FILE,
    index=False,
)

count_error_change_by_density_df.to_csv(
    ERROR_CHANGE_BY_DENSITY_FILE,
    index=False,
)

count_error_change_concentration_df.to_csv(
    ERROR_CHANGE_CONCENTRATION_FILE,
    index=False,
)

count_top_error_change_images_df.to_csv(
    TOP_ERROR_CHANGE_IMAGES_FILE,
    index=False,
)


# ------------------------------------------------------------
# 8. Useful density pivots
# ------------------------------------------------------------

net_abs_change_pivot = (
    count_error_change_by_density_df
    .pivot(
        index="density_stratum",
        columns="checkpoint_id",
        values="net_abs_error_change",
    )
    .reindex(
        DENSITY_ORDER
    )
    [
        CHECKPOINT_IDS
    ]
)


net_squared_change_pivot = (
    count_error_change_by_density_df
    .pivot(
        index="density_stratum",
        columns="checkpoint_id",
        values="net_squared_error_change",
    )
    .reindex(
        DENSITY_ORDER
    )
    [
        CHECKPOINT_IDS
    ]
)


paired_image_pivot = (
    count_error_change_by_density_df
    .pivot(
        index="density_stratum",
        columns="checkpoint_id",
        values="improved_images",
    )
    .reindex(
        DENSITY_ORDER
    )
    [
        CHECKPOINT_IDS
    ]
)


# ------------------------------------------------------------
# 9. Confirmation
# ------------------------------------------------------------

print("=" * 115)
print("STEP 7.2 - ERROR CONCENTRATION + IMAGE-LEVEL DRIVERS")
print("=" * 115)

print(
    "Paired comparison rows       :",
    len(
        count_error_change_per_image_df
    ),
)

print(
    "Expected                     : 300"
)

print(
    "Density rows                 :",
    len(
        count_error_change_by_density_df
    ),
)

print(
    "Expected                     : 18"
)

print(
    "MAE-change reconstruction    : PASS"
)

print(
    "Density reaggregation        : PASS"
)

print(
    "Squared-error reaggregation  : PASS"
)

print(
    "Test set accessed            : NO"
)


print(
    "\nNet absolute-error change vs B0 by density"
    "\n(negative = improvement):"
)

display(
    net_abs_change_pivot
    .round(1)
    .rename_axis(
        columns=None
    )
)


print(
    "\nNet squared-error change vs B0 by density"
    "\n(negative = improvement):"
)

display(
    net_squared_change_pivot
    .round(1)
    .rename_axis(
        columns=None
    )
)


print(
    "\nNumber of improved images vs B0 by density:"
)

display(
    paired_image_pivot
    .astype(int)
    .rename_axis(
        columns=None
    )
)


print(
    "\nTop-k concentration of improvement / worsening:"
)

display(
    count_error_change_concentration_df
    .round(4)
)


print(
    "\nTop 10 largest improvements / worsenings by system:"
)

display(
    count_top_error_change_images_df
)


print(
    "\nSaved per-image changes:",
    ERROR_CHANGE_PER_IMAGE_FILE,
)

print(
    "Saved density decomposition:",
    ERROR_CHANGE_BY_DENSITY_FILE,
)

print(
    "Saved concentration:",
    ERROR_CHANGE_CONCENTRATION_FILE,
)

print(
    "Saved top driver images:",
    TOP_ERROR_CHANGE_IMAGES_FILE,
)

print("=" * 115)

STEP 7.2 - ERROR CONCENTRATION + IMAGE-LEVEL DRIVERS
Paired comparison rows       : 300
Expected                     : 300
Density rows                 : 18
Expected                     : 18
MAE-change reconstruction    : PASS
Density reaggregation        : PASS
Squared-error reaggregation  : PASS
Test set accessed            : NO

Net absolute-error change vs B0 by density
(negative = improvement):


,overall_joint,sparse_joint,dense_recovery
density_stratum,,,
S1_0_20,533.0,459.0,331.0
S2_20_40,525.0,430.0,560.0
S3_40_60,161.0,56.0,452.0
S4_60_80,-271.0,-335.0,-189.0
S5_80_95,-91.0,-8.0,-247.0
S6_95_100,-109.0,-29.0,-333.0



Net squared-error change vs B0 by density
(negative = improvement):


,overall_joint,sparse_joint,dense_recovery
density_stratum,,,
S1_0_20,26757.0,20043.0,18481.0
S2_20_40,28419.0,20008.0,39526.0
S3_40_60,7899.0,2030.0,58068.0
S4_60_80,-18685.0,-20451.0,-10551.0
S5_80_95,-18099.0,-10612.0,-35607.0
S6_95_100,-426181.0,-343697.0,-664281.0



Number of improved images vs B0 by density:


...


Top-k concentration of improvement / worsening:


,checkpoint_id,top_k,topk_abs_improvement,share_total_abs_improvement,topk_abs_worsening,share_total_abs_worsening,topk_squared_improvement,share_total_squared_improvement
0,overall_joint,1,86.0,0.0865,165.0,0.0947,161265.0,0.2909
1,overall_joint,5,301.0,0.3028,408.0,0.2342,484369.0,0.8736
2,overall_joint,10,517.0,0.5201,664.0,0.3812,522263.0,0.9420
3,overall_joint,20,798.0,0.8028,1067.0,0.6125,545818.0,0.9845
4,sparse_joint,1,79.0,0.0832,185.0,0.1216,132544.0,0.2772
5,sparse_joint,5,276.0,0.2908,403.0,0.2648,417271.0,0.8726
6,sparse_joint,10,468.0,0.4932,621.0,0.4080,446086.0,0.9329
7,sparse_joint,20,741.0,0.7808,956.0,0.6281,468351.0,0.9795
8,dense_recovery,1,107.0,0.0858,145.0,0.0796,295344.0,0.3882
9,dense_recovery,5,418.0,0.3352,534.0,0.2932,669183.0,0.8795



Top 10 largest improvements / worsenings by system:


,checkpoint_id,change_type,rank,dataset_index,image_id,category,density_stratum,gt_count,b0_pred_count,guided_pred_count,b0_abs_error,guided_abs_error,delta_abs_error_vs_b0,delta_squared_error_vs_b0
0,overall_joint,largest_improvement,1,253,975.jpg,birds,S6_95_100,718,0,86,718.0,632.0,-86.0,-116100.0
1,overall_joint,largest_improvement,2,1192,6969.jpg,birds,S6_95_100,949,0,58,949.0,891.0,-58.0,-106720.0
2,overall_joint,largest_improvement,3,1187,6931.jpg,birds,S6_95_100,355,3,61,352.0,294.0,-58.0,-37468.0
3,overall_joint,largest_improvement,4,184,840.jpg,books,S6_95_100,637,7,59,630.0,578.0,-52.0,-62816.0
4,overall_joint,largest_improvement,5,125,745.jpg,ants,S4_60_80,60,12,59,48.0,1.0,-47.0,-2303.0
5,overall_joint,largest_improvement,6,183,838.jpg,books,S5_80_95,113,43,89,70.0,24.0,-46.0,-4324.0
6,overall_joint,largest_improvement,7,143,764.jpg,birds,S5_80_95,85,1,46,84.0,39.0,-45.0,-5535.0
7,overall_joint,largest_improvement,8,1269,7060.jpg,pills,S4_60_80,65,11,76,54.0,11.0,-43.0,-2795.0
8,overall_joint,largest_improvement,9,938,5866.jpg,grapes,S6_95_100,267,47,88,220.0,179.0,-41.0,-16359.0
9,overall_joint,largest_improvement,10,227,933.jpg,birds,S4_60_80,67,14,55,53.0,12.0,-41.0,-2665.0



Saved per-image changes: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/count_error_change_per_image_vs_b0.csv
Saved density decomposition: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/count_error_change_by_density_vs_b0.csv
Saved concentration: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/count_error_change_concentration_vs_b0.csv
Saved top driver images: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/count_top_error_change_images_vs_b0.csv


In [22]:
# ============================================================
# Step 8 - Checkpoint Synthesis + Final Freeze Validation
# ============================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Final output artefacts
# ------------------------------------------------------------

CHECKPOINT_SYNTHESIS_FILE = (
    E2E_TABLES_DIR
    / "checkpoint_synthesis.csv"
)

CHECKPOINT_FREEZE_VALIDATION_FILE = (
    E2E_TABLES_DIR
    / "checkpoint_freeze_validation.csv"
)

CHECKPOINT_ARTIFACT_HASHES_FILE = (
    E2E_TABLES_DIR
    / "checkpoint_artifact_hashes.csv"
)

CHECKPOINT_FINAL_MANIFEST_FILE = (
    E2E_METADATA_DIR
    / "checkpoint_final_manifest.json"
)

CHECKPOINT_FROZEN_MARKER_FILE = (
    E2E_METADATA_DIR
    / "CHECKPOINT_FROZEN.json"
)


# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def sha256_file(path):

    path = Path(path)

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def series_all_true(series):

    return (
        series
        .astype(str)
        .str.lower()
        .eq("true")
        .all()
    )


def to_python(value):

    if isinstance(
        value,
        (
            np.integer,
            np.int32,
            np.int64,
        ),
    ):
        return int(value)

    if isinstance(
        value,
        (
            np.floating,
            np.float32,
            np.float64,
        ),
    ):
        return float(value)

    if isinstance(
        value,
        np.bool_,
    ):
        return bool(value)

    return value


# ------------------------------------------------------------
# 3. Required existing artefacts
# ------------------------------------------------------------

required_artifacts = {

    # -------------------------
    # Step 5
    # -------------------------

    "guided_inference_per_image":
        GUIDED_INFERENCE_TABLE_FILE,

    "guided_inference_protocol":
        GUIDED_PROTOCOL_FILE,

    "guided_run_metadata":
        GUIDED_RUN_METADATA_FILE,

    "guided_cache_validation_summary":
        CACHE_VALIDATION_SUMMARY_FILE,

    "guided_cache_validation_metadata":
        CACHE_VALIDATION_METADATA_FILE,


    # -------------------------
    # Step 6
    # -------------------------

    "mask_gt_correspondence":
        MASK_GT_CORRESPONDENCE_FILE,

    "gt_mask_coverage":
        GT_MASK_COVERAGE_FILE,

    "mask_diagnostics_per_image":
        MASK_DIAGNOSTICS_PER_IMAGE_FILE,

    "mask_diagnostics_by_system":
        MASK_DIAGNOSTICS_BY_SYSTEM_FILE,

    "mask_diagnostics_by_density":
        MASK_DIAGNOSTICS_BY_DENSITY_FILE,

    "mask_multiplicity_by_system":
        MASK_MULTIPLICITY_BY_SYSTEM_FILE,

    "gt_multiplicity_by_system":
        GT_MULTIPLICITY_BY_SYSTEM_FILE,

    "mask_multiplicity_by_density":
        MASK_MULTIPLICITY_BY_DENSITY_FILE,

    "gt_multiplicity_by_density":
        GT_MULTIPLICITY_BY_DENSITY_FILE,


    # -------------------------
    # Step 7
    # -------------------------

    "count_comparison_per_image":
        COUNT_COMPARISON_PER_IMAGE_FILE,

    "count_metrics_overall":
        COUNT_METRICS_OVERALL_FILE,

    "count_metrics_by_density":
        COUNT_METRICS_BY_DENSITY_FILE,

    "count_paired_vs_b0":
        COUNT_PAIRED_VS_B0_FILE,

    "count_proxy_translation_check":
        COUNT_PROXY_TRANSLATION_FILE,

    "count_error_change_per_image":
        ERROR_CHANGE_PER_IMAGE_FILE,

    "count_error_change_by_density":
        ERROR_CHANGE_BY_DENSITY_FILE,

    "count_error_change_concentration":
        ERROR_CHANGE_CONCENTRATION_FILE,

    "count_top_error_change_images":
        TOP_ERROR_CHANGE_IMAGES_FILE,
}


# ------------------------------------------------------------
# 4. Check every required artefact exists
# ------------------------------------------------------------

missing_artifacts = [
    name
    for name, path
    in required_artifacts.items()
    if not Path(
        path
    ).exists()
]


assert (
    len(
        missing_artifacts
    )
    == 0
), (
    "Missing required artefacts: "
    f"{missing_artifacts}"
)


# ------------------------------------------------------------
# 5. Reload authoritative saved tables FROM DISK
#
# This avoids relying only on notebook memory.
# ------------------------------------------------------------

final_count_per_image_df = (
    pd.read_csv(
        COUNT_COMPARISON_PER_IMAGE_FILE
    )
)

final_overall_df = (
    pd.read_csv(
        COUNT_METRICS_OVERALL_FILE
    )
)

final_density_df = (
    pd.read_csv(
        COUNT_METRICS_BY_DENSITY_FILE
    )
)

final_paired_df = (
    pd.read_csv(
        COUNT_PAIRED_VS_B0_FILE
    )
)

final_proxy_df = (
    pd.read_csv(
        COUNT_PROXY_TRANSLATION_FILE
    )
)

final_mask_system_df = (
    pd.read_csv(
        MASK_DIAGNOSTICS_BY_SYSTEM_FILE
    )
)

final_mask_density_df = (
    pd.read_csv(
        MASK_DIAGNOSTICS_BY_DENSITY_FILE
    )
)

final_mask_correspondence_df = (
    pd.read_csv(
        MASK_GT_CORRESPONDENCE_FILE
    )
)

final_gt_coverage_df = (
    pd.read_csv(
        GT_MASK_COVERAGE_FILE
    )
)

final_error_density_df = (
    pd.read_csv(
        ERROR_CHANGE_BY_DENSITY_FILE
    )
)

final_error_concentration_df = (
    pd.read_csv(
        ERROR_CHANGE_CONCENTRATION_FILE
    )
)


# ------------------------------------------------------------
# 6. Reload protocols from disk
# ------------------------------------------------------------

with open(
    GUIDED_PROTOCOL_FILE,
    "r",
) as f:

    final_guided_protocol = (
        json.load(
            f
        )
    )


with open(
    MASK_CORRESPONDENCE_PROTOCOL_FILE,
    "r",
) as f:

    final_mask_protocol = (
        json.load(
            f
        )
    )


# ------------------------------------------------------------
# 7. Freeze-validation checks
# ------------------------------------------------------------

validation_records = []


def record_check(
    name,
    passed,
    detail,
):

    validation_records.append(
        {
            "check":
                name,

            "passed":
                bool(
                    passed
                ),

            "detail":
                str(
                    detail
                ),
        }
    )

    assert passed, (
        f"Freeze validation failed: "
        f"{name}"
    )


# ------------------------------------------------------------
# 7A. Dataset/system shape
# ------------------------------------------------------------

record_check(
    "overall_system_rows",
    len(
        final_overall_df
    ) == 4,
    (
        f"observed="
        f"{len(final_overall_df)}, "
        f"expected=4"
    ),
)


record_check(
    "count_per_image_rows",
    len(
        final_count_per_image_df
    ) == 400,
    (
        f"observed="
        f"{len(final_count_per_image_df)}, "
        f"expected=400"
    ),
)


record_check(
    "count_density_rows",
    len(
        final_density_df
    ) == 24,
    (
        f"observed="
        f"{len(final_density_df)}, "
        f"expected=24"
    ),
)


record_check(
    "mask_system_rows",
    len(
        final_mask_system_df
    ) == 3,
    (
        f"observed="
        f"{len(final_mask_system_df)}, "
        f"expected=3"
    ),
)


record_check(
    "mask_density_rows",
    len(
        final_mask_density_df
    ) == 18,
    (
        f"observed="
        f"{len(final_mask_density_df)}, "
        f"expected=18"
    ),
)


record_check(
    "mask_records",
    len(
        final_mask_correspondence_df
    ) == 15379,
    (
        f"observed="
        f"{len(final_mask_correspondence_df)}, "
        f"expected=15379"
    ),
)


record_check(
    "gt_coverage_records",
    len(
        final_gt_coverage_df
    ) == 30135,
    (
        f"observed="
        f"{len(final_gt_coverage_df)}, "
        f"expected=30135"
    ),
)


# ------------------------------------------------------------
# 7B. System identities
# ------------------------------------------------------------

observed_systems = (
    final_overall_df[
        "system_id"
    ].tolist()
)


record_check(
    "system_identity",
    observed_systems
    == COUNT_SYSTEM_ORDER,
    (
        f"observed="
        f"{observed_systems}"
    ),
)


# ------------------------------------------------------------
# 7C. Frozen B0 reconstruction
# ------------------------------------------------------------

final_b0 = (
    final_overall_df[
        final_overall_df[
            "system_id"
        ]
        == B0_SYSTEM_ID
    ]
    .iloc[0]
)


record_check(
    "B0_MAE_reconstruction",
    np.isclose(
        float(
            final_b0[
                "mae"
            ]
        ),
        80.69,
    ),
    (
        f"MAE="
        f"{final_b0['mae']}"
    ),
)


record_check(
    "B0_RMSE_reconstruction",
    np.isclose(
        float(
            final_b0[
                "rmse"
            ]
        ),
        270.2339,
        atol=1e-4,
    ),
    (
        f"RMSE="
        f"{final_b0['rmse']}"
    ),
)


# ------------------------------------------------------------
# 7D. Proxy -> SAM2 final-count identity
# ------------------------------------------------------------

record_check(
    "proxy_MAE_identity",
    series_all_true(
        final_proxy_df[
            "mae_match"
        ]
    ),
    "All three guided systems match frozen Group-E proxy MAE.",
)


record_check(
    "proxy_RMSE_identity",
    series_all_true(
        final_proxy_df[
            "rmse_match"
        ]
    ),
    "All three guided systems match frozen Group-E proxy RMSE.",
)


# Stronger per-image identity against frozen Group-E points.

guided_final_counts = (
    final_count_per_image_df[
        final_count_per_image_df[
            "system_id"
        ].isin(
            CHECKPOINT_IDS
        )
    ][
        [
            "system_id",
            "dataset_index",
            "image_id",
            "pred_count",
        ]
    ]
    .copy()
)


frozen_proxy_counts = (
    e3_checkpoint_points_df[
        [
            "checkpoint_id",
            "dataset_index",
            "image_id",
            "pred_count",
        ]
    ]
    .rename(
        columns={
            "checkpoint_id":
                "system_id",

            "pred_count":
                "proxy_pred_count",
        }
    )
)


final_translation_check = (
    guided_final_counts
    .merge(
        frozen_proxy_counts,
        on=[
            "system_id",
            "dataset_index",
            "image_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


record_check(
    "per_image_proxy_count_identity",
    (
        len(
            final_translation_check
        )
        == 300
        and
        (
            final_translation_check[
                "pred_count"
            ].astype(int)
            ==
            final_translation_check[
                "proxy_pred_count"
            ].astype(int)
        ).all()
    ),
    "300/300 guided final counts equal frozen Group-E proxy counts.",
)


# ------------------------------------------------------------
# 7E. Test-set isolation
# ------------------------------------------------------------

record_check(
    "guided_protocol_test_untouched",
    (
        final_guided_protocol.get(
            "test_set_accessed"
        )
        is False
    ),
    (
        "guided_inference_protocol."
        "test_set_accessed=False"
    ),
)


record_check(
    "mask_protocol_test_untouched",
    (
        final_mask_protocol.get(
            "test_set_accessed"
        )
        is False
    ),
    (
        "mask_correspondence_protocol."
        "test_set_accessed=False"
    ),
)


# ------------------------------------------------------------
# 7F. No inference-time density routing
# ------------------------------------------------------------

record_check(
    "no_density_routing",
    (
        final_guided_protocol.get(
            "density_routing"
        )
        in [
            False,
            None,
            "none",
            "None",
        ]
    ),
    (
        "Frozen representatives evaluated "
        "independently; S1-S6 used only "
        "for post-hoc diagnostics."
    ),
)


# ------------------------------------------------------------
# 7G. Cache integrity / completeness
# ------------------------------------------------------------

cache_files = sorted(
    E2E_MASK_CACHE_DIR.rglob(
        "*.npz"
    )
)


record_check(
    "guided_cache_count",
    len(
        cache_files
    ) == 300,
    (
        f"observed="
        f"{len(cache_files)}, "
        f"expected=300"
    ),
)


# ------------------------------------------------------------
# 8. Derive final checkpoint evidence
# ------------------------------------------------------------

b0_mae = float(
    final_b0[
        "mae"
    ]
)

b0_rmse = float(
    final_b0[
        "rmse"
    ]
)


guided_overall_df = (
    final_overall_df[
        final_overall_df[
            "system_id"
        ].isin(
            CHECKPOINT_IDS
        )
    ]
    .copy()
)


best_guided_mae_row = (
    guided_overall_df
    .sort_values(
        "mae"
    )
    .iloc[0]
)


best_guided_rmse_row = (
    guided_overall_df
    .sort_values(
        "rmse"
    )
    .iloc[0]
)


# ------------------------------------------------------------
# Density-regime pattern
# ------------------------------------------------------------

density_mae_pivot_final = (
    final_density_df
    .pivot(
        index="density_stratum",
        columns="system_id",
        values="mae",
    )
    .reindex(
        DENSITY_ORDER
    )
)


b0_better_all_s1_s3 = all(
    density_mae_pivot_final.loc[
        density,
        B0_SYSTEM_ID
    ]
    <
    density_mae_pivot_final.loc[
        density,
        CHECKPOINT_IDS
    ].min()
    for density
    in [
        "S1_0_20",
        "S2_20_40",
        "S3_40_60",
    ]
)


guided_better_all_s4_s6 = all(
    density_mae_pivot_final.loc[
        density,
        CHECKPOINT_IDS
    ].max()
    <
    density_mae_pivot_final.loc[
        density,
        B0_SYSTEM_ID
    ]
    for density
    in [
        "S4_60_80",
        "S5_80_95",
        "S6_95_100",
    ]
)


record_check(
    "density_regime_complementarity",
    (
        b0_better_all_s1_s3
        and
        guided_better_all_s4_s6
    ),
    (
        "B0 lower MAE than every guided "
        "system in S1-S3; every guided "
        "system lower MAE than B0 in S4-S6."
    ),
)


# ------------------------------------------------------------
# Dense-recovery breadth
# ------------------------------------------------------------

dense_error_density = (
    final_error_density_df[
        final_error_density_df[
            "checkpoint_id"
        ]
        == "dense_recovery"
    ]
)


dense_improved_s4_s6 = int(
    dense_error_density[
        dense_error_density[
            "density_stratum"
        ].isin(
            [
                "S4_60_80",
                "S5_80_95",
                "S6_95_100",
            ]
        )
    ][
        "improved_images"
    ]
    .sum()
)


dense_top5_squared_row = (
    final_error_concentration_df[
        (
            final_error_concentration_df[
                "checkpoint_id"
            ]
            == "dense_recovery"
        )
        &
        (
            final_error_concentration_df[
                "top_k"
            ]
            == 5
        )
    ]
    .iloc[0]
)


dense_top5_squared_share = float(
    dense_top5_squared_row[
        "share_total_squared_improvement"
    ]
)


# ------------------------------------------------------------
# 9. Five-question checkpoint synthesis
# ------------------------------------------------------------

checkpoint_synthesis_df = pd.DataFrame(
    [
        {
            "question_id":
                "Q1",

            "checkpoint_question":
                (
                    "Does A/B/C/D/E semantic-reference-point "
                    "work improve end-to-end SAM2 counting "
                    "over B0?"
                ),

            "conclusion":
                (
                    "Not on overall MAE under the frozen "
                    "one-point-to-one-retained-mask interface. "
                    "All three guided systems have higher "
                    "overall MAE than B0, although all have "
                    "lower RMSE."
                ),

            "key_evidence":
                (
                    f"B0 MAE={b0_mae:.2f}; "
                    f"best guided MAE="
                    f"{best_guided_mae_row['mae']:.2f} "
                    f"({best_guided_mae_row['system_id']}). "
                    f"B0 RMSE={b0_rmse:.4f}; "
                    f"best guided RMSE="
                    f"{best_guided_rmse_row['rmse']:.4f} "
                    f"({best_guided_rmse_row['system_id']})."
                ),

            "design_implication":
                (
                    "Do not replace B0 globally with the "
                    "current guided pipeline."
                ),
        },

        {
            "question_id":
                "Q2",

            "checkpoint_question":
                (
                    "Where across S1-S6 do improvements "
                    "and degradations occur?"
                ),

            "conclusion":
                (
                    "A clear density-regime split is present: "
                    "B0 is stronger in S1-S3, while the guided "
                    "systems improve MAE in S4-S6."
                ),

            "key_evidence":
                (
                    "B0 beats every guided representative "
                    "in S1-S3; every guided representative "
                    "beats B0 in S4-S6."
                ),

            "design_implication":
                (
                    "The useful contribution of semantic "
                    "guidance is concentrated in denser scenes."
                ),
        },

        {
            "question_id":
                "Q3",

            "checkpoint_question":
                (
                    "Do sparse/dense strengths suggest a "
                    "future combined or structured strategy?"
                ),

            "conclusion":
                (
                    "Yes. The results show complementary "
                    "behaviour across density regimes rather "
                    "than one globally superior system."
                ),

            "key_evidence":
                (
                    "B0 remains strong for S1-S3; guided "
                    "systems, especially dense_recovery, "
                    "become useful in S4-S6."
                ),

            "design_implication":
                (
                    "Investigate inference-available "
                    "structured integration, but do not use "
                    "GT-derived S1-S6 labels as routing inputs."
                ),
        },

        {
            "question_id":
                "Q4",

            "checkpoint_question":
                (
                    "Are improvements broad or dominated "
                    "by a few very dense images?"
                ),

            "conclusion":
                (
                    "Both effects are present. Dense-scene "
                    "MAE improvements occur across many images, "
                    "but the RMSE gain is highly concentrated "
                    "in a small number of extreme cases."
                ),

            "key_evidence":
                (
                    f"dense_recovery improves "
                    f"{dense_improved_s4_s6}/40 S4-S6 images; "
                    f"its top-5 images account for "
                    f"{100*dense_top5_squared_share:.2f}% "
                    f"of total squared-error improvement."
                ),

            "design_implication":
                (
                    "Do not interpret lower RMSE alone as "
                    "broad global superiority."
                ),
        },

        {
            "question_id":
                "Q5",

            "checkpoint_question":
                (
                    "Do reference-point proxy improvements "
                    "translate to final SAM2 counting?"
                ),

            "conclusion":
                (
                    "They translate exactly. Under the frozen "
                    "single-positive, multimask=False, "
                    "retain_all interface, final count equals "
                    "the number of semantic reference points."
                ),

            "key_evidence":
                (
                    "300/300 guided image-system cases have "
                    "final count equal to frozen Group-E "
                    "proxy count; proxy and final MAE/RMSE "
                    "are identical."
                ),

            "design_implication":
                (
                    "SAM2 currently contributes spatial mask "
                    "information but does not itself modify "
                    "the counting decision."
                ),
        },
    ]
)


# ------------------------------------------------------------
# 10. Save synthesis before hashing
# ------------------------------------------------------------

checkpoint_synthesis_df.to_csv(
    CHECKPOINT_SYNTHESIS_FILE,
    index=False,
)


# ------------------------------------------------------------
# 11. Build freeze-validation dataframe
# ------------------------------------------------------------

checkpoint_freeze_validation_df = (
    pd.DataFrame(
        validation_records
    )
)


record_check_count = int(
    len(
        checkpoint_freeze_validation_df
    )
)


assert (
    checkpoint_freeze_validation_df[
        "passed"
    ].all()
)


checkpoint_freeze_validation_df.to_csv(
    CHECKPOINT_FREEZE_VALIDATION_FILE,
    index=False,
)


# ------------------------------------------------------------
# 12. Hash all frozen source/output artefacts
# ------------------------------------------------------------

hash_records = []


hash_targets = dict(
    required_artifacts
)

hash_targets[
    "checkpoint_synthesis"
] = CHECKPOINT_SYNTHESIS_FILE

hash_targets[
    "checkpoint_freeze_validation"
] = CHECKPOINT_FREEZE_VALIDATION_FILE


for name, path in hash_targets.items():

    path = Path(
        path
    )

    hash_records.append(
        {
            "artifact_name":
                name,

            "path":
                str(
                    path
                ),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


checkpoint_artifact_hashes_df = (
    pd.DataFrame(
        hash_records
    )
)


checkpoint_artifact_hashes_df.to_csv(
    CHECKPOINT_ARTIFACT_HASHES_FILE,
    index=False,
)


# ------------------------------------------------------------
# 13. Hash guided mask-cache bundle
#
# Deterministic aggregate hash based on:
#   relative path + individual SHA256.
# ------------------------------------------------------------

cache_bundle_hasher = (
    hashlib.sha256()
)

cache_total_bytes = 0


for cache_path in cache_files:

    relative_path = (
        cache_path.relative_to(
            E2E_MASK_CACHE_DIR
        )
    )

    file_hash = sha256_file(
        cache_path
    )

    cache_bundle_hasher.update(
        str(
            relative_path
        ).encode(
            "utf-8"
        )
    )

    cache_bundle_hasher.update(
        file_hash.encode(
            "utf-8"
        )
    )

    cache_total_bytes += int(
        cache_path.stat().st_size
    )


cache_bundle_sha256 = (
    cache_bundle_hasher.hexdigest()
)


# ------------------------------------------------------------
# 14. Frozen representative specification
# ------------------------------------------------------------

representative_spec_columns = [
    col
    for col
    in [
        "checkpoint_id",
        "fusion_family",
        "c_variant",
        "d_variant",
        "tau",
    ]
    if col
    in e3_checkpoint_points_df.columns
]


frozen_representatives = (
    e3_checkpoint_points_df[
        representative_spec_columns
    ]
    .drop_duplicates()
    .sort_values(
        "checkpoint_id"
    )
    .to_dict(
        orient="records"
    )
)


frozen_representatives = [
    {
        k: to_python(v)
        for k, v
        in record.items()
    }
    for record
    in frozen_representatives
]


# ------------------------------------------------------------
# 15. Build final manifest
# ------------------------------------------------------------

freeze_timestamp_utc = (
    datetime.now(
        timezone.utc
    )
    .isoformat()
)


sam2_provenance = (
    b0_metadata.get(
        "code_provenance",
        {}
    )
)


checkpoint_final_manifest = {

    "checkpoint_name":
        "B1_End_to_End_Semantic_Guided_SAM2_Checkpoint",

    "status":
        "FROZEN",

    "freeze_timestamp_utc":
        freeze_timestamp_utc,

    "scope":
        (
            "Development-only end-to-end checkpoint comparing "
            "frozen semantic reference-point representatives "
            "against frozen B0 on FSC-147 dev100."
        ),

    "dataset":
        {
            "name":
                "FSC-147",

            "split":
                "validation",

            "subset":
                "frozen dev100",

            "num_images":
                100,

            "test_set_accessed":
                False,
        },

    "systems":
        {
            "baseline":
                B0_SYSTEM_ID,

            "guided_representatives":
                frozen_representatives,
        },

    "guided_interface":
        {
            "prompt_mode":
                SAM2_PROMPT_MODE,

            "multimask_output":
                bool(
                    SAM2_MULTIMASK_OUTPUT
                ),

            "instance_selection":
                SAM2_INSTANCE_SELECTION,

            "one_reference_point_per_prompt":
                True,

            "mask_filtering":
                False,

            "deduplication":
                False,

            "score_thresholding":
                False,

            "density_routing":
                False,

            "final_count_equals_retained_masks":
                True,

            "retained_masks_equal_reference_points":
                True,
        },

    "sam2_provenance":
        sam2_provenance,

    "frozen_counts":
        {
            "guided_system_image_cases":
                300,

            "semantic_reference_points":
                15379,

            "retained_sam2_masks":
                15379,

            "mask_gt_correspondence_records":
                15379,

            "gt_coverage_records":
                30135,

            "mask_cache_files":
                len(
                    cache_files
                ),
        },

    "cache_bundle":
        {
            "directory":
                str(
                    E2E_MASK_CACHE_DIR
                ),

            "num_files":
                len(
                    cache_files
                ),

            "total_bytes":
                int(
                    cache_total_bytes
                ),

            "sha256":
                cache_bundle_sha256,
        },

    "key_results":
        {
            "B0":
                {
                    "mae":
                        b0_mae,

                    "rmse":
                        b0_rmse,
                },

            "guided":
                {
                    row.system_id:
                        {
                            "mae":
                                float(
                                    row.mae
                                ),

                            "rmse":
                                float(
                                    row.rmse
                                ),

                            "mean_signed_error":
                                float(
                                    row.mean_signed_error
                                ),
                        }

                    for row
                    in guided_overall_df.itertuples(
                        index=False
                    )
                },

            "density_regime_pattern":
                {
                    "B0_better_than_all_guided_S1_S3":
                        bool(
                            b0_better_all_s1_s3
                        ),

                    "all_guided_better_than_B0_S4_S6":
                        bool(
                            guided_better_all_s4_s6
                        ),
                },

            "dense_recovery":
                {
                    "improved_images_S4_S6":
                        dense_improved_s4_s6,

                    "num_images_S4_S6":
                        40,

                    "top5_share_total_squared_error_improvement":
                        dense_top5_squared_share,
                },

            "proxy_to_final_count_identity":
                True,
        },

    "interpretation_constraints":
        {
            "S1_S6_are_posthoc_diagnostic_strata":
                True,

            "S1_S6_not_used_for_inference_routing":
                True,

            "FSC147_GT_points_are_not_instance_masks":
                True,

            "mask_GT_correspondence_is_point_support_diagnostic":
                True,

            "boundary_GT_policy":
                (
                    "Canonical coordinates preserved; "
                    "np.rint + clip used only for discrete "
                    "mask membership lookup."
                ),
        },

    "checkpoint_questions":
        checkpoint_synthesis_df.to_dict(
            orient="records"
        ),

    "freeze_validation":
        {
            "num_checks":
                record_check_count,

            "all_passed":
                True,

            "validation_file":
                str(
                    CHECKPOINT_FREEZE_VALIDATION_FILE
                ),
        },

    "artifact_hashes_file":
        str(
            CHECKPOINT_ARTIFACT_HASHES_FILE
        ),

    "future_work_not_part_of_this_freeze":
        [
            (
                "Structured semantic integration "
                "(Group E / 13D.3)."
            ),

            (
                "Any inference-time density or scene-structure "
                "estimation strategy."
            ),

            (
                "Mask consolidation / duplicate suppression / "
                "merged-region handling."
            ),

            (
                "Final FSC-147 test-set evaluation."
            ),
        ],
}


with open(
    CHECKPOINT_FINAL_MANIFEST_FILE,
    "w",
) as f:

    json.dump(
        checkpoint_final_manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 16. Manifest hash + frozen marker
# ------------------------------------------------------------

final_manifest_sha256 = (
    sha256_file(
        CHECKPOINT_FINAL_MANIFEST_FILE
    )
)


checkpoint_frozen_marker = {

    "status":
        "FROZEN",

    "checkpoint":
        "B1_End_to_End_Semantic_Guided_SAM2_Checkpoint",

    "freeze_timestamp_utc":
        freeze_timestamp_utc,

    "manifest":
        str(
            CHECKPOINT_FINAL_MANIFEST_FILE
        ),

    "manifest_sha256":
        final_manifest_sha256,

    "test_set_accessed":
        False,

    "note":
        (
            "This marker freezes the development checkpoint "
            "results only. Future structured integration work "
            "must be treated as a new experimental stage."
        ),
}


with open(
    CHECKPOINT_FROZEN_MARKER_FILE,
    "w",
) as f:

    json.dump(
        checkpoint_frozen_marker,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 17. Final confirmation
# ------------------------------------------------------------

print("=" * 118)
print("STEP 8 - CHECKPOINT SYNTHESIS + FINAL FREEZE VALIDATION")
print("=" * 118)

print(
    "Required artefacts            : PASS"
)

print(
    "Disk reload                   : PASS"
)

print(
    "Frozen B0 reconstruction      : PASS"
)

print(
    "Guided proxy-count identity   : PASS"
)

print(
    "Mask / GT record counts       : PASS"
)

print(
    "Density-regime validation     : PASS"
)

print(
    "Guided mask caches            :",
    len(
        cache_files
    ),
    "/ 300"
)

print(
    "Freeze validation checks      :",
    record_check_count,
    "PASS"
)

print(
    "Test set accessed             : NO"
)

print(
    "Density routing used          : NO"
)

print(
    "Checkpoint status             : FROZEN"
)


print(
    "\nFive-question checkpoint synthesis:"
)

display(
    checkpoint_synthesis_df
)


print(
    "\nFreeze-validation table:"
)

display(
    checkpoint_freeze_validation_df
)


print(
    "\nOverall frozen count metrics:"
)

display(
    final_overall_df[
        [
            "system_id",
            "mae",
            "rmse",
            "mean_signed_error",
            "under_count",
            "exact_count",
            "over_count",
        ]
    ]
    .round(4)
)


print(
    "\nMask-cache bundle:"
)

print(
    "  files :",
    len(
        cache_files
    )
)

print(
    "  size  :",
    f"{cache_total_bytes / (1024**2):.2f} MiB"
)

print(
    "  SHA256:",
    cache_bundle_sha256
)


print(
    "\nFinal manifest SHA256:"
)

print(
    final_manifest_sha256
)


print(
    "\nSaved synthesis:",
    CHECKPOINT_SYNTHESIS_FILE,
)

print(
    "Saved validation:",
    CHECKPOINT_FREEZE_VALIDATION_FILE,
)

print(
    "Saved artefact hashes:",
    CHECKPOINT_ARTIFACT_HASHES_FILE,
)

print(
    "Saved final manifest:",
    CHECKPOINT_FINAL_MANIFEST_FILE,
)

print(
    "Saved frozen marker:",
    CHECKPOINT_FROZEN_MARKER_FILE,
)

print("=" * 118)
print(
    "B1 END-TO-END DEVELOPMENT CHECKPOINT IS NOW FROZEN."
)
print("=" * 118)

STEP 8 - CHECKPOINT SYNTHESIS + FINAL FREEZE VALIDATION
Required artefacts            : PASS
Disk reload                   : PASS
Frozen B0 reconstruction      : PASS
Guided proxy-count identity   : PASS
Mask / GT record counts       : PASS
Density-regime validation     : PASS
Guided mask caches            : 300 / 300
Freeze validation checks      : 18 PASS
Test set accessed             : NO
Density routing used          : NO
Checkpoint status             : FROZEN

Five-question checkpoint synthesis:


,question_id,checkpoint_question,conclusion,key_evidence,design_implication
0,Q1,Does A/B/C/D/E semantic-reference-point work i...,Not on overall MAE under the frozen one-point-...,B0 MAE=80.69; best guided MAE=86.42 (sparse_jo...,Do not replace B0 globally with the current gu...
1,Q2,Where across S1-S6 do improvements and degrada...,A clear density-regime split is present: B0 is...,B0 beats every guided representative in S1-S3;...,The useful contribution of semantic guidance i...
2,Q3,Do sparse/dense strengths suggest a future com...,Yes. The results show complementary behaviour ...,"B0 remains strong for S1-S3; guided systems, e...",Investigate inference-available structured int...
3,Q4,Are improvements broad or dominated by a few v...,Both effects are present. Dense-scene MAE impr...,dense_recovery improves 29/40 S4-S6 images; it...,Do not interpret lower RMSE alone as broad glo...
4,Q5,Do reference-point proxy improvements translat...,They translate exactly. Under the frozen singl...,300/300 guided image-system cases have final c...,SAM2 currently contributes spatial mask inform...



Freeze-validation table:


,check,passed,detail
0,overall_system_rows,True,"observed=4, expected=4"
1,count_per_image_rows,True,"observed=400, expected=400"
2,count_density_rows,True,"observed=24, expected=24"
3,mask_system_rows,True,"observed=3, expected=3"
4,mask_density_rows,True,"observed=18, expected=18"
5,mask_records,True,"observed=15379, expected=15379"
6,gt_coverage_records,True,"observed=30135, expected=30135"
7,system_identity,True,"observed=['B0_unguided', 'overall_joint', 'spa..."
8,B0_MAE_reconstruction,True,MAE=80.69
9,B0_RMSE_reconstruction,True,RMSE=270.2338801852943



Overall frozen count metrics:


,system_id,mae,rmse,mean_signed_error,under_count,exact_count,over_count
0,B0_unguided,80.69,270.2339,-77.47,76,4,20
1,overall_joint,88.17,262.7308,-49.37,36,0,64
2,sparse_joint,86.42,264.0067,-55.18,38,0,62
3,dense_recovery,86.43,259.0033,-43.01,35,2,63



Mask-cache bundle:
  files : 300
  size  : 6.32 MiB
  SHA256: a28837871b9e9612736bf4312ed5a72563d1d14b1f711e3e2c3cb3d1c2412ed6

Final manifest SHA256:
2c22fd8e244db159c7cc33b3a0ff24b05177f92faf333a860255160fa7ec3e50

Saved synthesis: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/checkpoint_synthesis.csv
Saved validation: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/checkpoint_freeze_validation.csv
Saved artefact hashes: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/tables/checkpoint_artifact_hashes.csv
Saved final manifest: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/metadata/checkpoint_final_manifest.json
Saved frozen marker: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/End_to_End_Checkpoint/metadata/CHECKPOINT_FROZEN.json
B1 END-TO